In [ ]:
import numpy as np
import pandas as pd
import mdtraj as md
from scipy.optimize import curve_fit
from scipy.integrate import simps

import os 
import sys

import scipy.stats as stats
import matplotlib.pyplot as plt

In [ ]:
def _bound_check(func, params):
    """
    Hack for now.
    """
    if len(params) == 1:
        return False
    elif len(params)%2 == 0 :
        s = sum(params[0::2])
        return (s>1)
    else:
        s = params[0]+sum(params[1::2])
        return (s>1)

In [ ]:
def calc_chi(y1, y2, dy=[]):
    if dy != []:
        return np.sum( (y1-y2)**2.0/dy )/len(y1)
    else:
        return np.sum( (y1-y2)**2.0 )/len(y1)

In [ ]:
def _return_parameter_names(num_pars):
    if num_pars==1:
        return ['C_a', 'tau_a']
    elif num_pars==2:
         return ['C_a', 'tau_a']
    elif num_pars==3:
         return ['C_a', 'tau_a', 'tau_b']
    elif num_pars==4:
         return ['C_a', 'tau_a', 'C_b', 'tau_b']
    elif num_pars==5:
         return ['C_a', 'tau_a', 'C_b', 'tau_b', 'tau_g']
    elif num_pars==6:
         return ['C_a', 'tau_a', 'C_b', 'tau_b', 'C_g', 'tau_g']
    elif num_pars==7:
         return ['C_a', 'tau_a', 'C_b', 'tau_b', 'C_g', 'tau_g', 'tau_d']
    elif num_pars==8:
         return ['C_a', 'tau_a', 'C_b', 'tau_b', 'C_g', 'tau_g', 'C_d', 'tau_d']
    elif num_pars==9:
         return ['C_a', 'tau_a', 'C_b', 'tau_b', 'C_g', 'tau_g', 'C_d', 'tau_d', 'tau_e']
    elif num_pars==10:
         return [ 'C_a', 'tau_a', 'C_b', 'tau_b', 'C_g', 'tau_g', 'C_d', 'tau_d', 'C_e', 'tau_e']

    return []

In [ ]:
## Functions 1,3,5,7,9 are the functions that the sum of coefficients are equal to 1. They have one less parameter.
## Functions 2,4,6,8,10 are the functions where the sum of coefficients are not restricted.

def func_exp_decay1(t, tau_a):
    return np.exp(-t/tau_a)
def func_exp_decay2(t, A, tau_a):
    return A*np.exp(-t/tau_a)
def func_exp_decay3(t, A, tau_a, tau_b):
    return A*np.exp(-t/tau_a) + (1-A)*np.exp(-t/tau_b)
def func_exp_decay4(t, A, tau_a, B, tau_b ):
    return A*np.exp(-t/tau_a) + B*np.exp(-t/tau_b)
def func_exp_decay5(t, A, tau_a, B, tau_b, tau_g ):
    return A*np.exp(-t/tau_a) + B*np.exp(-t/tau_b) + (1-A-B)*np.exp(-t/tau_g)
def func_exp_decay6(t, A, tau_a, B, tau_b, G, tau_g ):
    return A*np.exp(-t/tau_a) + B*np.exp(-t/tau_b) + G*np.exp(-t/tau_g)
def func_exp_decay7(t, A, tau_a, B, tau_b, G, tau_g, tau_d):
    return A*np.exp(-t/tau_a) + B*np.exp(-t/tau_b) + G*np.exp(-t/tau_g) + (1-A-B-G)*np.exp(-t/tau_d)
def func_exp_decay8(t, A, tau_a, B, tau_b, G, tau_g, D, tau_d):
    return A*np.exp(-t/tau_a) + B*np.exp(-t/tau_b) + G*np.exp(-t/tau_g) + D*np.exp(-t/tau_d)
def func_exp_decay9(t, A, tau_a, B, tau_b, G, tau_g, D, tau_d, tau_e):
    return A*np.exp(-t/tau_a) + B*np.exp(-t/tau_b) + G*np.exp(-t/tau_g) + D*np.exp(-t/tau_d) + (1-A-B-G-D)*np.exp(-t/tau_e)
def func_exp_decay10(t, A, tau_a, B, tau_b, G, tau_g, D, tau_d, E, tau_e):
    return A*np.exp(-t/tau_a) + B*np.exp(-t/tau_b) + G*np.exp(-t/tau_g) + D*np.exp(-t/tau_d) + E*np.exp(-t/tau_e)


In [ ]:
def do_Expstyle_fit2(num_pars, x, y, dy=[], tinput=50.0, tmax=np.inf,  tmin=0.0005):
    
    """
    Function that calculates the fit to the exponential function
    Chooses the function based off the number of parameters
    tmin is the minimum value the correlation times can take;
    usually set to the timestep used to calculate the correlation function; in units of the timestep;
    tinput: the time value to scale the initial guesses.
    tmax: maximum bounding value of the correlation times
    tmin: minimum bounding value of the correlation times: usually x[t=1]
    """
    ## These are really bad guesses 
    b1_guess = [1 - y[0]*0.99,  y[0]/3+0.15, y[0]/2 +0.3, y[0]] 
    #b1_guess = 1.0/num_pars/2
    t1_guess = [tinput/1280, tinput/640, tinput/16, tinput/4]
    
    tmax=tmax*1.5
    tmax_values = [0.0099, 0.150, 3.5, tmax]
    
    if num_pars==1:
        func=func_exp_decay1
        guess=(t1_guess[2])
        bound=(0.,tmax)
    elif num_pars==2:
        func=func_exp_decay2
        guess=(b1_guess[3], t1_guess[2])
        bound=([0.0, tmin], [1., tmax])
    elif num_pars==3:
        func=func_exp_decay3
        guess=(b1_guess[3], t1_guess[3], t1_guess[0])
        bound=([0.0, tmin, tmin],[1., tmax, tmax])
    elif num_pars==4:
        func=func_exp_decay4
        guess = (b1_guess[3], t1_guess[3], b1_guess[0], t1_guess[1])
        #guess=(b1_guess, t1_guess[3], b1_guess, t1_guess[1])
        bound=([0.0, tmin, 0.0, tmin], [1.0, tmax, 1.0, tmax_values[1]])
    elif num_pars==5:
        func=func_exp_decay5
        guess=(b1_guess[3], t1_guess[3], b1_guess[1], t1_guess[2], t1_guess[1])
        bound=([0.0, tmin, 0.0, tmin, tmin],[1., tmax, 1., tmax, tmax])
    elif num_pars==6:
        func=func_exp_decay6
        guess=(b1_guess[3], t1_guess[3], b1_guess[2], t1_guess[2], b1_guess[0], t1_guess[0])
        bound=([0.0, x[1], 0.0, x[1], 0.0, tmin],[1., tmax, 1., tmax, 1., tmax])
    elif num_pars==7:
        func=func_exp_decay7
        guess=(b1_guess, t1_guess[3], b1_guess, t1_guess[2], b1_guess, t1_guess[1],
               t1_guess[0])
        bound=([0.0, x[0], 0.0, x[0], 0.0, x[0], tmin],[1., tmax, 1., tmax, 1., tmax, tmax])
    elif num_pars==8:
        func=func_exp_decay8
        guess=(b1_guess, t1_guess[3], b1_guess, t1_guess[2], b1_guess, t1_guess[1],
               b1_guess, t1_guess[0])
        bound=([0.0, x[0], 0.0, x[0], 0.0, x[0], 0.0, tmin],[1., tmax, 1., tmax, 1., tmax, 1., tmax])

    if len(dy) > 1:
        popt, popv = curve_fit(func, x, y, p0=guess, sigma=dy, bounds=bound, method='trf', loss='soft_l1', 
                               max_nfev=num_pars*1000)
    else:
        popt, popv = curve_fit(func, x, y, p0=guess, bounds=bound) ##, loss='soft_l1')

    ymodel=[ func(x[i], *popt) for i in range(len(x)) ]
    #print ymodel

    bExceed=_bound_check(func, popt)
    if bExceed:
        print( "= = = WARNING, curve fitting in do_LSstyle_fit returns a sum>1.//")
        return 9999.99, popt, popv, ymodel
    else:
        return calc_chi(y, ymodel, dy), popt, popv, ymodel

In [ ]:
def _restrict_mono2bi(x, y, pmin_list, chi_min, dy=[], max_iter=400):
    
    iterations = 0;
    while iterations < max_iter:
        
        tb_random_guess = np.random.uniform(low=1e-2, high=0.149, size=(1,))
        print('Mono-exponential found at minimum: trying to restrict to bi-exponential')
        restrict_bounds = [(pmin_list[0]-0.025, pmin_list[1]-0.50, 0.0, x[0]),
                           (pmin_list[0]+0.025, pmin_list[1]+0.50, 1.0, 0.150) ]
        restrict_guess  = (pmin_list[0], pmin_list[1], y[0] - pmin_list[0], tb_random_guess[0])
        print(restrict_guess)
        npars = 4
        func = func_exp_decay4
        try:
            if len(dy) > 1:
                params, popv = curve_fit(func, x, y, p0=restrict_guess,
                                         sigma=dy, bounds=restrict_bounds, method='trf', loss='soft_l1', 
                                         max_nfev=npars*1000)
            else:
                params, popv = curve_fit(func, x, y, p0=guess, bounds=bound) ##, loss='soft_l1')
            bBadFit = False;    
            print(params)
            errors = np.sqrt(np.diag(popv))
            ymodel = func(x, *params)
            chi = calc_chi(y, ymodel, dy)
            threshold = 1.0 ###(1-npar_min/npars)*1.0
            
            step_check=0
            while step_check < npars:
            
                ## 1st Check to see if any of the taus are equal to the maximum tau_mem
                tau_chk = np.log(restrict_bounds[1][step_check] - params[step_check])
                print(tau_chk)
                if np.isnan(tau_chk) | (tau_chk < -5.0):
                    print('Tau_mem check failed')
                    bBadFit=True
                chkerr = errors[step_check]/params[step_check]
                if (chkerr>0.25):
                    print('Error Check Failed')
                    bBadFit=True
                    break
                    
                step_check += 1;
                
            chi_check = chi/chi_min    
            print("--- The chi_check for {} parameters is {:.3}".format(npars, chi_check))
            print("--- The threshold for this check is {:.3}>{:.3}".format(threshold, chi/chi_min))    
            if (not bBadFit) and (chi/chi_min < threshold):
                chi_min=chi ; par_min=params ; err_min=errors ;
                npar_min=npars ; ymod_min=ymodel; covar_min = popv;
                break;
            else:
                print('Check did not pass, because of the following conditions:')
                print('Bad Fit: {}'.format(bBadFit))
                print('Threshold Check: {}'.format(chi/chi_min < threshold))
                
        except:
            print('... Restricted Bi-exponential failed.. Reverting back to Mono-exponential')
            e = sys.exc_info()[0]
            print( "<p>Error: %s</p>" % e )
            break;
        iterations += 1
        if iterations == max_iter-1:
            print('max iterations reached without returning an error....')
            return None;
    
    return par_min, err_min, npar_min, chi_min, ymod_min, covar_min
        
    

In [ ]:
def findbest_Expstyle_fits2(x, y, taum=150.0, dy=[], bPrint=True, par_list=[2,3,5,7,9], threshold_scl=0.5):
    chi_min=np.inf
    # Search forwards
    print('Starting New Fit')
    for npars in par_list:
        print(npars)
        names = _return_parameter_names(npars)
        ###(num_pars, x, y, dy=[], tinput=50.0, tmax=np.inf,  tmin=0.0005)
        try:
            chi, params, covarMat, ymodel = do_Expstyle_fit2(npars, x, y, dy, tinput=taum, tmax=taum, tmin=x[0])
        except: 
            print(" ...fit returns an error! Continuing.")
            continue
        bBadFit=False
        reverted=False
        
        errors = np.sqrt(np.diag(covarMat))
        step_check = 0
        
        while step_check < npars:
            
            ## 1st Check to see if any of the taus are equal to the maximum tau_mem
            tau_chk = np.log(taum - params[step_check])
            print(tau_chk)
            if np.isnan(tau_chk) | (tau_chk < -5.0):
                print( " --- fit shows one of the taus is equal to Tau_Mem with %d parameters." % npars )
                print(  "  --- Occurred with parameter %s: %g == %g " % (names[step_check], params[step_check], taum ))
                bBadFit=True
                
            ## 2nd Check the error to make sure there is no overfitting
            chkerr = errors[step_check]/params[step_check]
            #print(chkerr)
            if (chkerr>0.10):
                print( " --- fit shows overfitting with %d parameters." % npars)
                print(  "  --- Occurred with parameter %s: %g +- %g " % (names[step_check], params[step_check],
                                                                         errors[step_check]))
                bBadFit=True
                break
            
            step_check += 1;
            ## 2nd Check to see if the coefficient is less than 1 % of the fit
            #if (i%2)==0:
            #    if params[i] < 0.01:
            #        print( " --- fit returns a coeffieceint with less than 1 percent with %d parameters." % npars)
            #        print(  "  --- Occurred with parameter %s: %g +- %g " % (names[i], params[i], errors[i]))
            #        bBadFit=True
            #        break
            
            
                #covarMat[i-1,:] = 0.0; covarMat[:,i-1] = 0.0;
                #covarMat[i,:] = 0.0; covarMat[:,i] = 0.0;
                
                
            #    bBadFit=True
            #    print(bBadFit)
                #break
                
        chi_check = chi/chi_min
        if npars == par_list[0]:
            threshold = 1.0
            npar_min = par_list[0]
        else:
            threshold = (1-npar_min/npars)*threshold_scl
                
        print("--- The chi_check for {} parameters is {:.3}".format(npars, chi_check))
        print("--- The threshold for this check is {:.3}>{:.3}".format(threshold,chi/chi_min))
        if (not bBadFit) and (chi/chi_min < threshold):
            chi_min=chi ; par_min=params ; err_min=errors ; npar_min=npars ; ymod_min=ymodel; covar_min = covarMat;
        else:
            ## Test continue here vs break
            continue; 
    
    if npar_min == 2:
        new_pars = _restrict_mono2bi(x, y, par_min, chi_min, dy=dy, max_iter=400)
        if new_pars != None:
            par_min = new_pars[0]; err_min=new_pars[1]; npar_min=new_pars[2];
            chi_min= new_pars[3]; ymod_min= new_pars[4]; covar_min= new_pars[5];
        
    tau_min = par_min[1::2]
    sort_tau = np.argsort(tau_min)[::-1]
    nsort_params = np.array([[2*tau_ind, 2*tau_ind+1] for tau_ind in sort_tau]).flatten()
    
    err_min = err_min[nsort_params] 
    par_min = par_min[nsort_params]
    sort_covarMat = covarMat[:,nsort_params][nsort_params]
    names = _return_parameter_names(npar_min)    
    
    if bPrint:       
        print( "= = Found %d parameters to be the minimum necessary to describe curve: chi(%d) = %g vs. chi(%d) = %g)" % (npar_min, npar_min, chi_min,  npars, chi))
        print( "Parameter %d %s: %g +- %g " % (npar_min, len(names), len(par_min), len(err_min)))
        for i in range(npar_min):
            print( "Parameter %d %s: %g +- %g " % (i, names[i], par_min[i], err_min[i]))
        print('\n')   
    return chi_min, names, par_min, err_min, ymod_min, sort_covarMat

In [ ]:
def fitstoDF(resnames, chi_list, pars_list, errs_list, names_list):
    ## Set Up columns indices and names for the data frame
    
    mparnames = _return_parameter_names(8)
    mtau_names = np.array(mparnames)[1::2]
    mc_names = np.array(mparnames)[::2]
    colnames = np.array(['Resname','NumExp'])
    tau_errnames = np.array([[c,"{}_err".format(c)] for c in mtau_names]).flatten()
    mc_errnames = np.array([[c, "{}_err".format(c)] for c in mc_names]).flatten()
    colnames = np.hstack([colnames,mc_errnames])
    colnames = np.hstack([colnames,tau_errnames])
    colnames = np.hstack([colnames,np.array(['Chi_Fit'])])
    FitDF = pd.DataFrame(index=np.arange(len(pars_list)), columns=colnames).fillna(0.0)
    FitDF['Resname'] = resnames
    FitDF['Chi_Fit'] = chi_list
    
    for i in range(len(pars_list)):
        npar = len(pars_list[i])
        if (npar%2)==1:
            ccut = npar-2
            tau_f, terr = pars_list[i][1:ccut+1:2], errs_list[i][1:ccut+1:2]
            tau_f = np.hstack([tau_f, pars_list[i][-1]])
            terr = np.hstack([terr, errs_list[i][-1]])
            sort_tau = np.argsort(tau_f)[::-1]
            coeff, cerr= pars_list[i][0:ccut:2], errs_list[i][0:ccut:2]
            Clast = 1; Clasterr = 0.0;
            for n,m in zip(coeff, cerr):
                Clast -= n
                Clasterr += m
            
            coeff =np.hstack([coeff, np.array(Clast)])
            cerr =np.hstack([cerr, np.array(Clasterr)])
    
            tne = np.array([[c,"{}_err".format(c)] for c in mparnames[1:npar+1:2]]).flatten()
            cne = np.array([[c, "{}_err".format(c)] for c in mparnames[0:npar:2]]).flatten()
                
        else:
            tau_f, terr = pars_list[i][1::2], errs_list[i][1::2] 
            coeff, cerr= pars_list[i][0::2], errs_list[i][0::2]
            sort_tau = np.argsort(tau_f)[::-1]
            tne = np.array([[c,"{}_err".format(c)] for c in names_list[i][1::2]]).flatten()
            cne = np.array([[c, "{}_err".format(c)] for c in names_list[i][0::2]]).flatten()
    
        NumExp=np.array(len(tau_f))
        tau_err = np.array([[t,e] for t,e in zip(tau_f[sort_tau],terr[sort_tau])]).flatten()
        c_err = np.array([[c,e] for c,e in zip(coeff[sort_tau], cerr[sort_tau])]).flatten()
        namesarr = np.hstack([np.array('NumExp'),cne,tne])
        valarr = np.hstack([NumExp,c_err,tau_err])
    
        FitDF.loc[i, namesarr] = valarr
    
    FitDF['AUC_a'] = FitDF.C_a*FitDF.tau_a; FitDF['AUC_b'] = FitDF.C_b*FitDF.tau_b;
    FitDF['AUC_g'] = FitDF.C_g*FitDF.tau_g; FitDF['AUC_d'] = FitDF.C_d*FitDF.tau_d;
    FitDF['AUC_Total'] = FitDF[['AUC_a','AUC_b','AUC_g','AUC_d']].sum(axis=1)
    
    return FitDF

In [ ]:
def fitCorrF(CorrDF, dCorrDF, tau_mem, pars_l, tcorr, tscl,  tstart=1, logt=[], fixfit=False, first_neg=False):
    
    NH_Res = CorrDF.columns
    chi_list=[] ; names_list=[] ; pars_list=[] ; errs_list=[] ; ymodel_list=[]; covarMat_list = [];
    for i in CorrDF.columns:
        
        if (not first_neg):
            tstop = np.where(CorrDF.index.values==tau_mem)[0][0] 
            
        else:
            print('Using first negative value as the final value in the fit:')
            tstop_arr = np.where(CorrDF[i].values < 0.005)[0]
            if tstop_arr.size > 0:
                tstop = tstop_arr[0]
            else:
                print('No Negative values found in the correlation')
                tstop = np.where(CorrDF.index.values==tau_mem*2)[0][0]
            print('Final Value in the Fit is at {} : {} ns'.format(tstop-1, CorrDF.index.values[tstop-1]))
        
        if len(logt) > 0:
            full_tseries = CorrDF.index.values
            logt_mask = np.array([gts for gts in logt if gts in full_tseries[tstart:tstop-1]])
            x = CorrDF.loc[logt_mask, i].index
            y = CorrDF.loc[logt_mask, i].values
        else:
            x = CorrDF.index.values[tstart:tstop-1:20]
            y = CorrDF[i].values[tstart:tstop-1:20]
            
        if len(dCorrDF) > 1:
            if len(logt) > 0:
                #logt_mask = logt[np.where(logt<tstop)]
                dy = dCorrDF.loc[logt_mask,i].values
            else:
                dy = dCorrDF[i].values[tstart:tstop-1:20]
        else:
            dy = dCorrDF
        ## if not fixfit then find find the best expstyle fit. Otherwise force the fit to nparams 
        if (not fixfit)&(len(pars_l)>1):
            print("Finding the best fit for residue {}".format(i))
            
            chi, names, pars, errs, ymodel, covarMat = findbest_Expstyle_fits2(x, y, tcorr*tscl, dy,
                                                                               par_list=pars_l, threshold_scl=thresh)
        
        elif (fixfit)&(len(pars_l)==1):
            print("Performing a fixed fit for {} exponentials for residue {}".format(int(pars_l[0]/2),i))
            ###(num_pars, x, y, dy=[], tinput=50.0, tmax=np.inf,  tmin=0.0005)
            chi, pars, covarMat, ymodel = do_Expstyle_fit2(pars_l[0], x, y, dy, tinput=tcorr*tscl,
                                                           tmax= tcorr*tscl, tmin=x[0])
            names = _return_parameter_names(len(pars))
            errs = np.sqrt(np.diag(covarMat))
            
        else:
            print("The list of parameters is empty. Breaking out.")
            break;
            
        chi_list.append(chi)
        names_list.append(names)
        pars_list.append(pars)
        errs_list.append(errs)
        ymodel_list.append(ymodel)
        covarMat_list.append(covarMat)
        
    FitDF = fitstoDF(NH_Res, chi_list, pars_list, errs_list, names_list)
    
    return FitDF, covarMat_list

In [ ]:
def fitCorrFNVT_wNVE(CorrDF, dCorrDF, tau_mem, NVEFitDF, tcorr, tscl, tstart=0, first_neg=False):
    
    NH_Res = CorrDF.columns
    chi_list=[] ; names_list=[] ; pars_list=[] ; errs_list=[] ; ymodel_list=[]; covarMat_list = [];
    for resindx, rowseries in NVEFitDF[['Resname', 'NumExp']].iterrows():
        
        i = str(int(rowseries.Resname))
        pars_l = [int(rowseries.NumExp*2.0)]
        
        if (not first_neg):
            tstop = np.where(CorrDF.index.values==tau_mem)[0][0]
        else:
            print('Using first negative value as the final value in the fit:')
            tstop_arr = np.where(CorrDF[i].values < 0.025)[0]
            if tstop_arr.size > 0:
                tstop = tstop_arr[0]
            else:
                print('No Negative values found in the correlation')
                tstop = np.where(CorrDF.index.values==tau_mem*2)[0][0]
            print('Final Value in the Fit is at {} : {} ns'.format(tstop-1, CorrDF.index.values[tstop-1]))
        
        x = CorrDF.index.values[tstart:tstop-1:20]
        y = CorrDF[i].values[tstart:tstop-1:20]
        dy = dCorrDF[i].values[tstart:tstop-1:20]

        print("Performing a fixed fit for {} exponentials for residue {}".format(int(pars_l[0]/2),i))
        ###(num_pars, x, y, dy=[], tinput=50.0, tmax=np.inf,  tmin=0.0005)
        chi, pars, covarMat, ymodel = do_Expstyle_fit2(pars_l[0], x, y, dy, tinput=tcorr*tscl,
                                                       tmax=tcorr*tscl, tmin=x[0])
        names = _return_parameter_names(len(pars))
        errs = np.sqrt(np.diag(covarMat))
            
        chi_list.append(chi)
        names_list.append(names)
        pars_list.append(pars)
        errs_list.append(errs)
        ymodel_list.append(ymodel)
        covarMat_list.append(covarMat)
        
    FitDF = fitstoDF(NH_Res, chi_list, pars_list, errs_list, names_list)
    
    return FitDF

In [ ]:
def fitCorrF_noSTD(CorrDF, tau_mem, pars_l,  tstart=0, fixfit=False, first_neg=False):
    
    NH_Res = CorrDF.columns
    chi_list=[] ; names_list=[] ; pars_list=[] ; errs_list=[] ; ymodel_list=[]; covarMat_list = [];
    for i in CorrDF.columns:
        
        if (not first_neg):
            tstop = np.where(CorrDF.index.values==tau_mem)[0][0]
        else:
            print('Using first negative value as the final value in the fit:')
            tstop_arr = np.where(CorrDF[i].values < 0)[0]
            if tstop_arr.size > 0:
                tstop = tstop_arr[0]
            else:
                print('No Negative values found in the correlation')
                tstop = np.where(CorrDF.index.values==tau_mem*2)[0][0]
            print('Final Value in the Fit is at {} : {} ns'.format(tstop-1, CorrDF.index.values[tstop-1]))
        
        x = CorrDF.index.values[tstart:tstop-1]
        y = CorrDF[i].values[tstart:tstop-1]
        #dy = dCorrDF[i].values[tstart:tstop-1]
        
        ## if not fixfit then find find the best expstyle fit. Otherwise force the fit to nparams 
        if (not fixfit)&(len(pars_l)>1):
            print("Finding the best fit for residue {}".format(i))
            dy=[]
            chi, names, pars, errs, ymodel, covarMat = findbest_Expstyle_fits2(x, y, dy,tau_mem, 
                                                                               par_list=pars_l, threshold_scl=thresh)
        
        elif (fixfit)&(len(pars_l)==1):
            print("Performing a fixed fit for {} exponentials for residue {}".format(int(pars_l[0]/2),i))
            dy=[]
            chi, pars, covarMat, ymodel = do_Expstyle_fit2(pars_l[0], x, y, dy, tau_mem)
            names = _return_parameter_names(len(pars))
            errs = np.sqrt(np.diag(covarMat))
            
        else:
            print("The list of parameters is empty. Breaking out.")
            break;
            
        chi_list.append(chi)
        names_list.append(names)
        pars_list.append(pars)
        errs_list.append(errs)
        ymodel_list.append(ymodel)
        covarMat_list.append(covarMat)
        
    FitDF = fitstoDF(NH_Res, chi_list, pars_list, errs_list, names_list)
    
    return FitDF, covarMat_list

In [ ]:
def _read_CorrFunctions(atom_df, prot, ens, dt, dtau, corr_type='NH', gamma = 'CF2ps'):
    
    """
    Function to read the correlation functions calculated from cpptraj 
    
    """
    corr_time_index = np.arange(0,dtau+1, 1)*dt/1e3
    rescorrdf = pd.DataFrame(index= corr_time_index, columns=atom_df.iloc[1:]['RESID'])
    rescorrdf_STD = rescorrdf.copy()
    for rcol in rescorrdf.columns:
        rescfdf = pd.DataFrame(index=corr_time_index, columns=RUNS)
        for R in RUNS:
            
            if ens == 'NVE':
                corrdf = pd.read_csv('{}/{}CorrFunc/NH{}CorrelationFunctions.dat'.format(LocDF_NVE.loc[R,(prot,ens)],
                                                                                        corr_type, rcol),
                                    delim_whitespace=True)
            elif ens =='NVT':
                corrdf = pd.read_csv('{}/{}CorrFunc/NH{}CorrelationFunctions.dat'.format(LocDF_NVT.loc[R,(prot,gamma)],
                                                                                        corr_type, rcol),
                                    delim_whitespace=True)
                
            rescfdf.loc[:,R] = corrdf['<P2>'].values
            
        rescorrdf[rcol] = rescfdf.mean(axis=1)
        rescorrdf_STD[rcol] = rescfdf.std(axis=1)
        
        zero_std = rescorrdf_STD[rcol][rescorrdf_STD[rcol]==0.0].index
        rescorrdf_STD.loc[zero_std,rcol] = 1e-6
    
    return rescorrdf, rescorrdf_STD
    

In [ ]:
def read_OrderParameterFiles(atom_df, prot, ens, gamma = 'CF2ps'):
    
    """
    Function to read the order parameters functions calculated from cpptraj and python script 
    
    """
    
    resS2df = pd.DataFrame(index=atom_df.iloc[1:]['RESID'], columns=RUNS)
    resS2df_STD = resS2df.copy()
    for R in RUNS:
            
        if ens == 'NVE':
            s2df = pd.read_csv('{}/OrderParameter_Direct.csv'.format(LocDF_NVE.loc[R,(prot, ens)]),
                                                                         names=['RESN',r'$S^{2}$'])
        elif ens =='NVT':
            s2df = pd.read_csv('{}/OrderParameter_Direct.csv'.format(LocDF_NVT.loc[R,(prot, gamma)]),
                                   names=['RESN',r'$S^{2}$'])

        resS2df.loc[:,R] = s2df[r'$S^{2}$'].values
            
    return resS2df

In [ ]:
def func_CorrInt2(t, A, tau_a, s2):
    
    return A*np.exp(-t/tau_a) + s2

def func_CorrInt2Uf(t, A, tau_a, s2, U, tau_uf):
    
    return A*np.exp(-t/tau_a) + U*np.exp(-t/tau_uf) + s2

def func_CorrInt3(t, sf2, tf, ts, s2):
    
    return (1-sf2)*np.exp(-t/tf) + (sf2-s2)*np.exp(-t/ts) + s2

def func_CorrIntMany(t, A, tau_a, B, tau_b, G, tau_g, D, tau_d, E, tau_e, s2):

    return A*np.exp(-t/tau_a) + B*np.exp(-t/tau_b) + G*np.exp(-t/tau_g) + D*np.exp(-t/tau_g) + E*np.exp(-t/tau_e) + s2


In [ ]:
def fitCI(npars, x, y, order, dy=[], t_mem=3.0, tmin = 0.0005, freeS2 = False):
    
    
    if (not freeS2):
        print('freeS2 is False')
        print('Restricting the fit by predefining the order parameter as the precalculated results')
        
        if npars == 2:
            guess = (0.25, 0.050)
            bound = [(0.0, tmin), (1.0, t_mem)] 
            func_CI = lambda t, a, ta : func_CorrInt2(t, a, ta, order)
        
        elif npars == 3:
            guess = (0.25, 0.050, 1.5)
            bound = [(0.0, tmin, tmin), (1.0, 0.500, np.inf)] 
            
            func_CI = lambda t, a, ta, tb: func_CorrInt3(t, a, ta, tb, order)
        
        elif npars == 10:
            guess = (0.05, 0.001, 0.05, 0.020, 0.05, 0.100, 0.05, 0.500, 0.05, 1.0)
            bound = [(0., tmin, 0., tmin, 0., tmin, 0., tmin,  0. , tmin),
                     (1., 0.010, 1., 0.050, 1., 0.200, 1., 1.0 , 1., 5.00)]
            func_CI = lambda t, a, ta, b, tb, g, tg, d, td, e, te : func_CorrIntMany(t, a, ta, b, tb, g, tg, d, td, e, te, order)
    
    else:
        print('freeS2 is True')
        print('Not restricting the order parameter and setting the calculated order parameter as the initial guess')
        if npars == 2:
            
            guess = (0.25, 0.250, order)
            bound = [(0.0, tmin, 0.0), (1.0, np.inf, 1.0)] 
            func_CI = func_CorrInt2
        
        elif npars == 3:
            
            guess = (0.25, 0.050, 0.500, order)
            bound = [(0.0, tmin, tmin, 0.0), (1.0, 0.250, t_mem, 1.0)] 
            func_CI = func_CorrInt3
        
        elif npars == 10:
            guess = (0.05, 0.001, 0.05, 0.020, 0.05, 0.100, 0.05, 0.500, 0.05, 1.0, order)
            bound = [(0., tmin, 0., tmin, 0., tmin, 0., tmin,  0. , tmin, 0.0),
                     (1., 0.010, 1., 0.050, 1., 0.200, 1., 1.0 , 1., 5.00, 1.0)]
            func_CI = func_CorrIntMany
    
    if dy.size > 1:
        popt, popv = curve_fit(func_CI, x, y, p0=guess, sigma=dy, bounds=bound, method='trf', loss='soft_l1')
    else:
        popt, popv = curve_fit(func_CI, x, y, p0=guess, bounds=bound, loss='soft_l1')

    ymodel=[func_CI(x[i], *popt) for i in range(len(x)) ]
    #print ymodel

    bExceed=_bound_check(func_CI, popt)
    if bExceed:
        print( "= = = WARNING, curve fitting in do_LSstyle_fit returns a sum>1.//")
        return 9999.99, popt, popv, ymodel
    else:
        return calc_chi(y, ymodel, dy), popt, popv, ymodel
    

In [ ]:
def FitMFCorr(CorrDF, dCorrDF, orderS2, t_mem, fS2, parms=[2], tstart=0):
    
    NH_Res = CorrDF.columns
    chi_list=[] ; names_list=[] ; pars_list=[] ; errs_list=[] ; ymodel_list=[]; covarMat_list = [];
    
    MS2 = orderS2.mean(axis=1); DS2 = orderS2.std(axis=1);
    
    for i in CorrDF.columns:
        
        print('Running MF approach ')
        tstop = np.where(CorrDF.index.values==t_mem)[0][0]
        tstop = np.where(ResCorrDF_GB3_NVE['2'].values < 0)[0][0]
        
        x = CorrDF.index.values[tstart:tstop]
        y = CorrDF[i].values[tstart:tstop]
        dy = dCorrDF[i].values[tstart:tstop]
        os2 = MS2.loc[int(i)]
        
        
        chi, pars, covarMat, ymodel = fitCI(parms[0], x, y, os2, dy, tmem=t_mem, tmin=0.0005, freeS2=fS2)
        names = _return_parameter_names(len(pars))
        errs = np.sqrt(np.diag(covarMat))
        
        chi_list.append(chi)
        names_list.append(names)
        pars_list.append(pars)
        errs_list.append(errs)
        ymodel_list.append(ymodel)
        covarMat_list.append(covarMat)
    
    #FitDF = pd.DataFrame(index=np.arange(0,len(NH_Res),1), columns=['RESID','S2','1-S2','tau_e', 'S2_Err', 'te_err', 'chi_sq']).fillna(0.0)
    FitDF = fitstoDF(NH_Res, chi_list, pars_list, errs_list, names_list)
    
    return FitDF, covarMat_list
    

In [ ]:
def J_direct_transform(om, consts, taus):
    
    ## Calculation for the direct spectral density 
    ndecay=len(consts) ; noms=1;###lnden(om)
    Jmat = np.zeros( (ndecay, noms ) )
    
    for i in range(ndecay):
        
        Jmat[i] = consts[i]*(taus[i]*1e-9)/(
            1 + np.power((taus[i]*1e-9)*(om),2.))
        
    return Jmat.sum(axis=0)

In [ ]:
def calc_NMR_Relax(J, fdd, fcsa, gammaH, gammaN):
    
    R1 = fdd * (J['Diff'] + 3*J['15N'] + 6*J['Sum']) + fcsa * J['15N']
    
    R2 = (0.5 * fdd * (4*J['0'] + J['Diff'] + 3*J['15N'] + 6*J['1H'] + 6*J['Sum']) 
          + (1./6.) * fcsa*(4*J['0'] + 3*J['15N']) )
    
    NOE = 1 + ((fdd*gammaH)/(gammaN*R1))*(6*J['Sum'] - J['Diff'])
    
    return R1, R2, NOE

In [ ]:
def calc_NMRFit(fit_df, magfield, exp_nmrdf=[]):
    
    """
    fit_df : dataframe containing the fitting results
    exp_nmrdf : dataframe containing the experimental NMR results
    magfield : magnetic field the experiments use (MHz)
    """
    ## Parameters and Physical Constants for calculation of Relaxation Rates
    H_gyro = 2*np.pi*42.57748*1e6     ## Gyromagnetic Ratio: Hydrogen ([rad]/[s][T]) 
    N_gyro = -2*np.pi*4.317267*1e6     ## Gyromagnetic Ratio: Nitrogen ([rad]/[s][T])
    B0 = 2*np.pi*magfield*1e6/H_gyro   ## Field Strength = 18.8 Teslas

    ## Need 5 Frequencies: ## J[0], J[wH], J[wN], J[wH-wN], J[wH+wN]
    Larmor1H = H_gyro*B0              ## Larmor Frequency: Hydrogen ([rad]/[s])
    Larmor15N = N_gyro*B0             ## Larmor Frequency: Hydrogen ([rad]/[s])
    omDiff = Larmor1H - Larmor15N    ## Diff in Larmor Frequencies of Spin IS
    omSum  = Larmor1H + Larmor15N    ## Sum of Larmor Frequencies of Spin I
    
    mu_0 = 4*np.pi*1e-7    ; ## H/m
    hbar = 1.0545718e-34  ; # [J] * [s] = [kg] * [m^2] * [s^-1] 
    R_NH = 1.02e-10                     ## distance between N-H atoms in Angstroms
    dSigmaN = -170e-6 
    
    FDD = (1./10.)*np.power((mu_0*hbar*H_gyro*N_gyro)/(4*np.pi*np.power(R_NH,3)),2)
    FCSA = (2.0/15.0)*(Larmor15N**2)*(dSigmaN**2)       
    
    ## Calculate 
    Jarr = []

    for i,fit in fit_df.iterrows():
        c = fit[['C_a','C_b','C_g','C_d']].values
        t = fit[['tau_a','tau_b','tau_g','tau_d']].values
        Jdict = {'0':0, '1H':0,'15N':0,'Sum':0,'Diff':0} 
        J0 = J_direct_transform(0, c, t)
        JH = J_direct_transform(Larmor1H, c, t)
        JN = J_direct_transform(Larmor15N, c, t)
        JSum = J_direct_transform(omSum,  c, t)
        JDiff = J_direct_transform(omDiff,  c, t)
        Jdict['1H'] = JH ; Jdict['15N'] = JN; Jdict['0'] = J0; 
        Jdict['Sum'] = JSum; Jdict['Diff'] = JDiff;
        Jarr.append(Jdict)
    
    NMRRelaxDF = pd.DataFrame(np.zeros((len(Jarr),3)), index=range(1,len(Jarr)+1), columns=['T1','T2','NOE'])
    for index in range(1,len(Jarr)+1):
        r1, r2, noe = calc_NMR_Relax(Jarr[index-1], FDD, FCSA, H_gyro, N_gyro)
        NMRRelaxDF.loc[index,'R1'] = r1;
        NMRRelaxDF.loc[index,'T1'] = 1/r1;
        NMRRelaxDF.loc[index,'R2'] = r2;
        NMRRelaxDF.loc[index,'T2'] = 1/r2; 
        NMRRelaxDF.loc[index,'NOE'] = noe;
    NMRRelaxDF['Resname'] = fit_df['Resname'].values
    NMRRelaxDF['RESNUM'] = NMRRelaxDF['Resname'].astype(str).str.extract('([0-9]+)',expand=False).astype('int')
    FitRelaxDF = fit_df.merge(NMRRelaxDF, how='left', left_on='Resname', right_on='Resname').set_index(NMRRelaxDF.index)
    
    ## Calculation of RMSE between NMR values and EXP:
    if len(exp_nmrdf)>0:
        RMSE_DF = pd.DataFrame(index=NMRRelaxDF['Resname'],
                               columns = ['{}_SE'.format(nmr) for nmr in ['R1','T1','R2','T2','NOE']])
        for nmr in ['R1','T1','R2','T2','NOE']:
            RMSE_DF['{}_SE'.format(nmr)] = (NMRRelaxDF[['Resname', nmr]].set_index('Resname') - 
                                            exp_nmrdf.rename(columns={'RESID':'Resname'})[['Resname',nmr]].set_index('Resname'))**2
        FitRelaxDF = FitRelaxDF.merge(RMSE_DF.reset_index(), how='left', left_on='Resname', right_on='Resname')
    
    return FitRelaxDF

In [ ]:
def MFCorrInt_IntegralMethod(mfcorr, orderS2):
    
    TFIntegral = pd.DataFrame(index=mfcorr.columns, columns=(['integral','normalized']))
    x = mfcorr.index.values
    for i in mfcorr.columns:
        
        shifted_corr = mfcorr[i].values-orderS2.loc[int(i)]
        
        if np.any(shifted_corr < 0.0):
            neg_index = np.where(shifted_corr < 0.0)[0][0]
            print(x[neg_index], shifted_corr[neg_index])
            
        integral = simps(shifted_corr[:neg_index], x[:neg_index] )
        TFIntegral.loc[i, 'integral'] = integral
        TFIntegral.loc[i, 'normalized'] = (1/(1-orderS2.loc[int(i)]))*integral
        
    return TFIntegral

In [ ]:
def _read_CorrFunctions_NVEopt(atom_df, prot, optlocations, dt, dtau, corr_type='NH', gamma = 'CF2ps'):
    
    """
    Function to read the correlation functions calculated from cpptraj 
    
    """
    corr_time_index = np.arange(0,dtau+1, 1)*dt/1e3
    rescorrdf = pd.DataFrame(index= corr_time_index, columns=atom_df.iloc[1:]['RESID'])
    rescorrdf_STD = rescorrdf.copy()
    for rcol in rescorrdf.columns:
        rescfdf = pd.DataFrame(index=corr_time_index, columns=optlocations)
        for rloc in optlocations:
            
            corrdf = pd.read_csv('{}{}/{}{}CorrFunc/NH{}CorrelationFunctions.dat'.format(CorrFuncLoc2, prot, rloc,
                                                                                        corr_type, rcol),
                                    delim_whitespace=True)
                
            rescfdf.loc[:,rloc] = corrdf['<P2>'].values
            
        rescorrdf[rcol] = rescfdf.mean(axis=1)
        rescorrdf_STD[rcol] = rescfdf.std(axis=1)
        
        zero_std = rescorrdf_STD[rcol][rescorrdf_STD[rcol]==0.0].index
        rescorrdf_STD.loc[zero_std, rcol] = 1e-6
    
    return rescorrdf, rescorrdf_STD
    

In [ ]:
CorrFuncLOC = './Documents/Research/DiffusionTip4pDSoluteSize/'
CorrFuncLoc2 = './Research/LangevinDynamics_RotationalDiffusion/'
ENSEMBLES = ['NVE', 'NVT']
CouplingFrequency = ['CF0-2ps', 'CF2ps', 'CF20ps']
RUNS = ['Run{}'.format(n) for n in range(1,5,1)]
Proteins = ['GB3', 'Ubiquitin']
MINDX = pd.MultiIndex.from_product([Proteins, ENSEMBLES])
MINDX_NVT=pd.MultiIndex.from_product([Proteins, CouplingFrequency])
LocDF_NVE = pd.DataFrame(index=RUNS, columns=MINDX)
LocDF_NVT = pd.DataFrame(index=RUNS, columns=MINDX_NVT)
FileLocDict = {}

In [ ]:
optimalGB3Loc = ['PROD_NVE/Run3/', 'PROD_NVE/Run4/', 'PROD_NVE_Run2/Run2/', 'PROD_NVE_Run2/Run3/']
optimalUBQLoc = ['PROD_NVE/Run1/', 'PROD_NVE/Run2/', 'PROD_NVE_Run2/Run1/', 'PROD_NVE_Run3/Run3/']

In [ ]:
for items, values in LocDF_NVT.iteritems():
    print(items)
    for indx, loc in values.iteritems():
        LocDF_NVT.loc[indx, items] = '{}{}/PROD_NVT/{}/{}'.format(CorrFuncLoc2, items[0], items[1], indx)
        
for items, values in LocDF_NVE.iteritems():
    print(items)
    for indx, loc in values.iteritems():
        LocDF_NVE.loc[indx, items] = '{}{}/PROD_{}/{}'.format(CorrFuncLoc2, items[0], items[1], indx)

In [ ]:
LocDF_NVE.iloc[1,0]

In [ ]:
LocDF_NVT.iloc[1,4]

In [ ]:
GB3Top = md.load_prmtop('{}GB3/GB3_2Na_FF14SB.prmtop'.format(CorrFuncLOC))
GB3HInd = GB3Top.select('name H and protein')
GB3Hatom_name = pd.Series(['{}'.format(GB3Top.atom(atmx)) for atmx in GB3HInd])
GB3atom_df = GB3Hatom_name.str.split('-',expand=True).rename(columns={0:'RESNAME',1:'ATMNAME'})
GB3atom_df['RESNAME'] = GB3atom_df['RESNAME'].apply(lambda rr: '{}'.format(rr[:3] + str(int(rr[3:])+1)))
GB3atom_df['RESID'] = GB3atom_df['RESNAME'].apply(lambda rr: int(rr[3:]))

In [ ]:
UBQTop = md.load_prmtop('{}Analysis/Ubiquitin/PROD_noH20.UBQ_10mMNaCl_FF14SB.prmtop'.format(CorrFuncLOC))
UBQHInd = UBQTop.select('name H and protein')
UBQHatom_name = pd.Series(['{}'.format(UBQTop.atom(atmx)) for atmx in UBQHInd])
UBQatom_df = UBQHatom_name.str.split('-',expand=True).rename(columns={0:'RESNAME',1:'ATMNAME'})
UBQatom_df['RESNAME'] = UBQatom_df['RESNAME'].apply(lambda rr: '{}'.format(rr[:3] + str(int(rr[3:])+1)))
UBQatom_df['RESID'] = UBQatom_df['RESNAME'].apply(lambda rr: int(rr[3:]))

## Loading and averaging the Correlation Functions for GB3
### Calculated using the cpptraj timecorr function using a saving frequency of 0.5 ps (tstep).

In [ ]:
## Loading Optimum Correlation Functions

In [ ]:
OptResCorrDF_GB3_NVE_Avg, OptResCorrDF_GB3_NVE_Std = _read_CorrFunctions_NVEopt(GB3atom_df, 'GB3', optimalGB3Loc, 0.5, 500000, corr_type='NH')

In [ ]:
OptResCorrDF_GB3_NVE_Avg.to_csv('{}GB3/Analysis/Optimum_AverageFullCorrelationFunctions_NVE.csv'.format(CorrFuncLoc2))
OptResCorrDF_GB3_NVE_Std.to_csv('{}GB3/Analysis/Optimum_STDFullCorrelationFunctions_NVE.csv'.format(CorrFuncLoc2))

In [ ]:

ResCorrDF_GB3_NVE, RESCorrDF_GB3_NVE_STD = _read_CorrFunctions(GB3atom_df, 'GB3', 'NVE', 0.5, 500000, corr_type='NH')
ResCorrDF_GB3_NVE.to_csv('{}GB3/Analysis/AverageFullCorrelationFunctions_NVE.csv'.format(CorrFuncLoc2))
RESCorrDF_GB3_NVE_STD.to_csv('{}GB3/Analysis/STDFullCorrelationFunctions_NVE.csv'.format(CorrFuncLoc2))

In [ ]:
ResCorrDF_GB3_NVT_CF2ps, RESCorrDF_GB3_NVT_CF2ps_STD = _read_CorrFunctions(GB3atom_df, 'GB3','NVT',
                                                                           0.5, 500000, corr_type='NH',
                                                                           gamma=CouplingFrequency[1])

In [ ]:
ResCorrDF_GB3_NVT_CF2ps.to_csv('{}GB3/Analysis/AverageFullCorrelationFunctions_NVT_CF2ps.csv'.format(CorrFuncLoc2))
RESCorrDF_GB3_NVT_CF2ps_STD.to_csv('{}GB3/Analysis/STDFullCorrelationFunctions_NVT_CF2ps.csv'.format(CorrFuncLoc2))

In [ ]:
ResCorrDF_GB3_NVT_CF02ps, RESCorrDF_GB3_NVT_CF02ps_STD = _read_CorrFunctions(GB3atom_df, 'GB3','NVT',
                                                                           0.5, 500000, corr_type='NH',
                                                                             gamma=CouplingFrequency[0])

In [ ]:
ResCorrDF_GB3_NVT_CF02ps.to_csv('{}GB3/Analysis/AverageFullCorrelationFunctions_NVT_CF02ps.csv'.format(CorrFuncLoc2))
RESCorrDF_GB3_NVT_CF02ps_STD.to_csv('{}GB3/Analysis/STDFullCorrelationFunctions_NVT_CF02ps.csv'.format(CorrFuncLoc2))

In [ ]:
ResCorrDF_GB3_NVT_CF20ps, RESCorrDF_GB3_NVT_CF20ps_STD = _read_CorrFunctions(GB3atom_df, 'GB3','NVT',
                                                                           0.5, 500000, corr_type='NH',
                                                                             gamma=CouplingFrequency[2])

In [ ]:
ResCorrDF_GB3_NVT_CF20ps.to_csv('{}GB3/Analysis/AverageFullCorrelationFunctions_NVT_CF20ps.csv'.format(CorrFuncLoc2))
RESCorrDF_GB3_NVT_CF20ps_STD.to_csv('{}GB3/Analysis/STDFullCorrelationFunctions_NVT_CF20ps.csv'.format(CorrFuncLoc2))

### Load  the averaged GB3 Correlation Functions:

In [ ]:
ResCorrDF_GB3_NVE = pd.read_csv('{}GB3/Analysis/AverageFullCorrelationFunctions_NVE.csv'.format(CorrFuncLoc2), index_col=0)
RESCorrDF_GB3_NVE_STD = pd.read_csv('{}GB3/Analysis/STDFullCorrelationFunctions_NVE.csv'.format(CorrFuncLoc2), index_col=0)

OptResCorrDF_GB3_NVE_Avg = pd.read_csv('{}GB3/Analysis/Optimum_AverageFullCorrelationFunctions_NVE.csv'.format(CorrFuncLoc2), index_col=0)
OptResCorrDF_GB3_NVE_Std = pd.read_csv('{}GB3/Analysis/Optimum_STDFullCorrelationFunctions_NVE.csv'.format(CorrFuncLoc2), index_col=0)

ResCorrDF_GB3_NVT_CF2ps = pd.read_csv('{}GB3/Analysis/AverageFullCorrelationFunctions_NVT_CF2ps.csv'.format(CorrFuncLoc2), index_col=0)
RESCorrDF_GB3_NVT_CF2ps_STD = pd.read_csv('{}GB3/Analysis/STDFullCorrelationFunctions_NVT_CF2ps.csv'.format(CorrFuncLoc2), index_col=0)

ResCorrDF_GB3_NVT_CF02ps = pd.read_csv('{}GB3/Analysis/AverageFullCorrelationFunctions_NVT_CF02ps.csv'.format(CorrFuncLoc2), index_col=0)
RESCorrDF_GB3_NVT_CF02ps_STD = pd.read_csv('{}GB3/Analysis/STDFullCorrelationFunctions_NVT_CF02ps.csv'.format(CorrFuncLoc2), index_col=0)

ResCorrDF_GB3_NVT_CF20ps = pd.read_csv('{}GB3/Analysis/AverageFullCorrelationFunctions_NVT_CF20ps.csv'.format(CorrFuncLoc2), index_col=0)
RESCorrDF_GB3_NVT_CF20ps_STD = pd.read_csv('{}GB3/Analysis/STDFullCorrelationFunctions_NVT_CF20ps.csv'.format(CorrFuncLoc2), index_col=0)

In [ ]:
## Log Time spaced arrays
equal_spaced_index = np.array([5e-4, 1e-3, 2e-3, 4e-3, 8e-3, 1.6e-2, 3.2e-2, 6.4e-2, 1.28e-1,
                      2.56e-1, 5.12e-1, 1.024e0, 2.048e0, 4.096e0, 8.192e0, 1.6384e1, 3.2768e1, 6.5536e1])
lnt_steps = np.unique(np.array([5e-4*np.round(np.power(1.02,i)).astype(int) for i in range(1000)]))

In [ ]:
## Testing the log-spacing and selecting only timesteps that appear in the indices
full_tseries = OptResCorrDF_GB3_NVE_Avg.index.values
tstop_test = np.where(full_tseries<10.0)[0]
lnt_steps_inFullT = np.array([gts for gts in lnt_steps if gts in full_tseries[tstop_test]])
#OptResCorrDF_GB3_NVE_Avg.loc[lnt_steps_inFullT].to_csv('{}{}/PROD_NVE/AveCorrFunction_NVE_GB3_LnT.csv'.format(CorrFuncLoc2,'GB3'))
#OptResCorrDF_GB3_NVE_Std.loc[lnt_steps_inFullT].to_csv('{}{}/PROD_NVE/STDCorrFunction_NVE_GB3_LnT.csv'.format(CorrFuncLoc2,'GB3'))

In [ ]:
#full_tseries = ResCorrDF_GB3_NVT_CF2ps.index.values
#tstop_test = np.where(full_tseries<14.0)[0]
#lnt_steps_inFullT = np.array([gts for gts in lnt_steps if gts in full_tseries[tstop_test]])
ResCorrDF_GB3_NVT_CF2ps.loc[lnt_steps_inFullT].to_csv('{}{}/PROD_NVT/CF2ps/AveCorrFunction_CF2ps_GB3_LnT.csv'.format(CorrFuncLoc2,'GB3'))
RESCorrDF_GB3_NVT_CF2ps_STD.loc[lnt_steps_inFullT].to_csv('{}{}/PROD_NVT/CF2ps/STDCorrFunction_CF2ps_GB3_LnT.csv'.format(CorrFuncLoc2,'GB3'))

In [ ]:
## Plot Correlation functions
ResCorrFig = plt.figure(1, figsize=(8,6))
for rcol in ResCorrDF_GB3_NVT_CF20ps.columns:
    axCF = ResCorrFig.add_subplot(111)
    
    ResCorrDF_GB3_NVE[rcol].plot(logx=True, ax = axCF, label=r'$\mathbf{ NVE }$')
    ResCorrDF_GB3_NVT_CF02ps[rcol].plot(logx=True, ax = axCF, label=r'$\mathbf{ NVT : \gamma = 0.2 \ \ ps^{-1}}$')
    ResCorrDF_GB3_NVT_CF2ps[rcol].plot(logx=True, ax = axCF, label=r'$\mathbf{ NVT : \gamma = 2 \ \ ps^{-1}}$')
    ResCorrDF_GB3_NVT_CF20ps[rcol].plot(logx=True, ax = axCF, label=r'$\mathbf{ NVT : \gamma = 20 \ \ ps^{-1}}$')
    axCF.set_title('Residue: {}'.format(rcol), fontsize=15, weight='bold')
    axCF.legend(loc=3)
    axCF.set_ylabel(r'$\mathbf{C(\tau)}$', fontsize=13)
    axCF.set_xlabel(r'$\mathbf{\tau \ \ (ns)}$',fontsize=13)
    axCF.tick_params(labelsize=15)
    ResCorrFig.savefig('{}GB3/Analysis/CompareCorrelationFunctions_WithCoupling_Res{}.png'.format(CorrFuncLoc2, rcol),dpi=600, bbox_inches='tight')
    ResCorrFig.clear()
    

In [ ]:
ResCorrDF_GB3_NVE['11'].plot(logx=True, style='-b', label=r'$\mathbf{THR11-NVE }$')
ResCorrDF_GB3_NVT_CF2ps['11'].plot(logx=True, style='-g',   label=r'$\mathbf{THR11-NVT : \gamma = 2 \ \ ps^{-1}}$')
ResCorrDF_GB3_NVE['36'].plot(logx=True, style='--b', label=r'$\mathbf{ASP36-NVE }$')
ResCorrDF_GB3_NVT_CF2ps['36'].plot(logx=True, style='--g', label=r'$\mathbf{ASP36-NVT : \gamma = 2 \ \ ps^{-1}}$')
plt.gca().legend(loc=3)
plt.gca().set_ylabel(r'$\mathbf{C(\tau)}$', fontsize=13)
plt.gca().set_xlabel(r'$\mathbf{\tau \ \ (ns)}$',fontsize=13)
plt.gca().tick_params(labelsize=15)
plt.savefig('{}GB3/Analysis/CorrelationFunction_Compare_Residue11-36_NVE_Cf2ps.png'.format(CorrFuncLoc2), bbox_inches='tight', dpi=600)


## Perform the fits for the Correlation Functions for GB3

In [ ]:
GB3_FirstNegTime_NVE = _get_FirstNegativeTime(ResCorrDF_GB3_NVE)
GB3_FirstNegTime_NVT_CF02ps = _get_FirstNegativeTime(ResCorrDF_GB3_NVT_CF02ps)
GB3_FirstNegTime_NVT_CF2ps = _get_FirstNegativeTime(ResCorrDF_GB3_NVT_CF2ps)
GB3_FirstNegTime_NVT_CF20ps = _get_FirstNegativeTime(ResCorrDF_GB3_NVT_CF20ps)

In [ ]:
print(True)

In [ ]:
import seaborn as sns


In [ ]:
sns.palplot(sns.color_palette('gist_yarg_r',3*2)[0:-1:2])

In [ ]:
def _plot_CorrelationFunctionCompare(dir_out, protein, fname_ext, fig_obj, fitdf_list, mdcorr_list, fitdf_names=[], mdnames=[]):
    """
    A Function to plot several correlation functions to compare them to the raw MD code
    """

    tseries = mdcorr_list[0].index.values
    for nresindx, residue in enumerate(mdcorr_list[0].columns): 
        
        axbict = fig_obj.add_subplot(111)
        if len(mdcorr_list) > 1:
            ctemplate = sns.color_palette('gist_yarg',len(mdcorr_list)*2)[0:-1:2]
        else:
            ctemplate = ['k']
        for mdct, mdn, mdcol in zip(mdcorr_list, mdnames, ctemplate):
            mdct.iloc[:, nresindx].plot(logx=True, label=mdn, linestyle='-', color=mdcol, ax=axbict)
        
        exp_lbl_list = []
        for fit, fitname in zip(fitdf_list, fitdf_names):
            func_pars = fit.iloc[nresindx].NumExp
            print(func_pars)
            if func_pars == 1:
                func = func_exp_decay2
                plist = ['C_a','tau_a']
                             
            elif func_pars == 2:               
                func = func_exp_decay4
                plist = ['C_a','tau_a','C_b','tau_b']
                
            elif func_pars == 3:
                func = func_exp_decay6
                plist = ['C_a','tau_a','C_b','tau_b', 'C_g', 'tau_g']

                
            ymodel = func(tseries, *fit.iloc[nresindx][plist].values)
            axbict.semilogx(tseries, ymodel, linestyle= '-', label=fitname)
            exp_lbl_list.append('Number of Exponentials = {}'.format(func_pars))
        
        axbict.set_ylim(-.1, 1.1)
        axbict.set_title('Residue : {}'.format(residue), weight='bold', fontsize=15)
        lgnd1 = axbict.legend(loc='upper right', frameon=False)
        lgnd_hndles, lgnd1_labels = axbict.get_legend_handles_labels()
        axbict.add_artist(lgnd1)
        axbict.legend(lgnd_hndles[len(mdcorr_list):], exp_lbl_list, frameon=False, loc='lower left')
        axbict.tick_params(labelsize=15)
        fig_obj.canvas.draw()
        fig_obj.savefig('{}{}/{}/CorrFunctionCompare_Residue{}_{}.png'.format(CorrFuncLoc2, protein,
                                                                              dir_out,  residue, fname_ext), 
                                    dpi=600, bbox_inches='tight')
        fig_obj.clear()
    
    return None

### Begin Fitting Correlation Functions for GB3

In [ ]:
## Fitting to NVE GB3 starting at C(t=1) using the best model fitting procedure for 1 and 2 exponentials
## using a cut-off of 10.0 ns without log time
tau_mem=10.0; thresh=0.50; fthresh=False; ShortSim=False; FixFit=False; Fix_Tf=False;
FitDF_GB3_NVE_Opt_FixTM10, covarMat_lists = fitCorrF(OptResCorrDF_GB3_NVE_Avg,
                                                     OptResCorrDF_GB3_NVE_Std, tau_mem, [2, 4],
                                                     tstart=1, tcorr=3.53, tscl=2.0,
                                                     fixfit=FixFit, first_neg=False)
FitDF_GB3_NVE_Opt_FixTM10['Resname'] = FitDF_GB3_NVE_Opt_FixTM10['Resname'].astype('int64')

In [ ]:
## Fitting to NVE GB3 starting at C(t=1) using the a bi-exponential fit
## using a cut-off of 10.0 ns with log time
tau_mem=10.0; thresh=0.50; fthresh=False; ShortSim=False; FixFit=True; Fix_Tf=False;
FitDF_GB3_NVE_Opt_FixTM10_FixFit_lnt, covarMat_lists = fitCorrF(OptResCorrDF_GB3_NVE_Avg,
                                                     OptResCorrDF_GB3_NVE_Std, tau_mem, [4],
                                                     tstart=1, tcorr=3.53, tscl=2.0,
                                                     logt=lnt_steps,
                                                     fixfit=FixFit, first_neg=False)
FitDF_GB3_NVE_Opt_FixTM10_FixFit_lnt['Resname'] = FitDF_GB3_NVE_Opt_FixTM10_FixFit_lnt['Resname'].astype('int64')

In [ ]:
## Fitting to NVE GB3 starting at C(t=1) using the a bi-exponential fit
## using a cut-off of 10.0 ns with log time
tau_mem=10.0; thresh=0.50; fthresh=False; ShortSim=False; FixFit=False; Fix_Tf=False;
FitDF_GB3_NVE_Opt_FixTM10_lnt_newB, covarMat_lists = fitCorrF(OptResCorrDF_GB3_NVE_Avg,
                                                         OptResCorrDF_GB3_NVE_Std, tau_mem, [2, 4],
                                                         tstart=1, tcorr=3.53, tscl=2.0,
                                                         logt=lnt_steps,
                                                         fixfit=FixFit, first_neg=False)
FitDF_GB3_NVE_Opt_FixTM10_lnt_newB['Resname'] = FitDF_GB3_NVE_Opt_FixTM10_lnt_newB['Resname'].astype('int64')

In [ ]:
## Fitting to NVE GB3 starting at C(t=1) using the a bi-exponential fit
## using a cut-off of 10.0 ns with log time
tau_mem=10.0; thresh=0.50; fthresh=False; ShortSim=False; FixFit=False; Fix_Tf=False;
FitDF_GB3_NVE_Opt_FixTM10_lnt_newB2, covarMat_lists = fitCorrF(OptResCorrDF_GB3_NVE_Avg,
                                                         OptResCorrDF_GB3_NVE_Std, tau_mem, [2, 4],
                                                         tstart=1, tcorr=3.53, tscl=2.0,
                                                         logt=lnt_steps,
                                                         fixfit=FixFit, first_neg=False)
FitDF_GB3_NVE_Opt_FixTM10_lnt_newB2['Resname'] = FitDF_GB3_NVE_Opt_FixTM10_lnt_newB2['Resname'].astype('int64')

In [ ]:
FitDF_GB3_NVE_Opt_FixTM10_lnt_newB2['tau_b']

In [ ]:
FitDF_GB3_NVE_Opt_FixTM10_FixFit_lnt.plot(y='tau_b',logy=True)
FitDF_GB3_NVE_Opt_FixTM10_lnt_newB.plot(y='tau_b',logy=True, ax=plt.gca())
FitDF_GB3_NVE_Opt_FixTM10_lnt_newB2.plot(y='tau_b',logy=True, ax=plt.gca())

In [ ]:
FitDF_GB3_NVT_02ps_FixTM11, covarMat_lists = fitCorrF(ResCorrDF_GB3_NVT_CF02ps,
                                                      RESCorrDF_GB3_NVT_CF02ps_STD, 12.0, [2,4],
                                                      tstart=1, tcorr=3.53, tscl=3.0,
                                                      logt=[],
                                                      fixfit=False, first_neg=False)
FitDF_GB3_NVT_02ps_FixTM11['Resname']= FitDF_GB3_NVT_02ps_FixTM11['Resname'].astype('int64')

In [ ]:
FitDF_GB3_NVT_02ps_FixTM11_lnt, covarMat_lists = fitCorrF(ResCorrDF_GB3_NVT_CF02ps,
                                                      RESCorrDF_GB3_NVT_CF02ps_STD, 11.0, [2, 4],
                                                      tstart=1, tcorr=3.53, tscl=3.0,
                                                      logt=lnt_steps,
                                                      fixfit=False, first_neg=False)
FitDF_GB3_NVT_02ps_FixTM11_lnt['Resname']= FitDF_GB3_NVT_02ps_FixTM11_lnt['Resname'].astype('int64')

In [ ]:
FitDF_GB3_NVT_02ps_FixTM11_lnt_restB, covarMat_lists = fitCorrF(ResCorrDF_GB3_NVT_CF02ps,
                                                      RESCorrDF_GB3_NVT_CF02ps_STD, 11.0, [2, 4],
                                                      tstart=1, tcorr=3.53, tscl=3.0,
                                                      logt=lnt_steps,
                                                      fixfit=False, first_neg=False)
FitDF_GB3_NVT_02ps_FixTM11_lnt_restB['Resname']= FitDF_GB3_NVT_02ps_FixTM11_lnt_restB['Resname'].astype('int64')

In [ ]:
FitDF_GB3_NVT_02ps_FixTM11_lnt_restB2, covarMat_lists = fitCorrF(ResCorrDF_GB3_NVT_CF02ps,
                                                      RESCorrDF_GB3_NVT_CF02ps_STD, 11.0, [2, 4],
                                                      tstart=1, tcorr=3.53, tscl=2.5,
                                                      logt=lnt_steps,
                                                      fixfit=False, first_neg=False)
FitDF_GB3_NVT_02ps_FixTM11_lnt_restB2['Resname']= FitDF_GB3_NVT_02ps_FixTM11_lnt_restB2['Resname'].astype('int64')

In [ ]:
FitDF_GB3_NVT_02ps_FixTM11_lnt_restB_randtb, covarMat_lists = fitCorrF(ResCorrDF_GB3_NVT_CF02ps,
                                                      RESCorrDF_GB3_NVT_CF02ps_STD, 11.0, [2, 4],
                                                      tstart=1, tcorr=3.53, tscl=3.0,
                                                      logt=lnt_steps,
                                                      fixfit=False, first_neg=False)
FitDF_GB3_NVT_02ps_FixTM11_lnt_restB_randtb['Resname']= FitDF_GB3_NVT_02ps_FixTM11_lnt_restB_randtb['Resname'].astype('int64')

In [ ]:
FitDF_GB3_NVT_02ps_FixTM11_lnt_restB_randtb.iloc[np.where(FitDF_GB3_NVT_02ps_FixTM11_lnt_restB_randtb['C_b']==0)]

In [ ]:
full_tseries = ResCorrDF_GB3_NVT_CF02ps.index.values
tstop_test = np.where(full_tseries<11.0)[0]
lnt_steps_inFullT = np.array([gts for gts in lnt_steps if gts in full_tseries[tstop_test]])
for r in np.where(FitDF_GB3_NVT_02ps_FixTM11_lnt_restB_randtb['C_b_err']==0)[0]:
    FitDF_GB3_NVT_02ps_FixTM11_lnt_restB_randtb.loc[r,'C_b'] = (ResCorrDF_GB3_NVT_CF02ps.iloc[1,r] - 
                                                                FitDF_GB3_NVT_02ps_FixTM11_lnt_restB_randtb.iloc[r]['C_a'])
    FitDF_GB3_NVT_02ps_FixTM11_lnt_restB_randtb.loc[r,'tau_b'] = np.random.uniform(1e-2,1e-1)
    rand_model = func_exp_decay4(lnt_steps_inFullT,
                                 *FitDF_GB3_NVT_02ps_FixTM11_lnt_restB_randtb.loc[r,['C_a','tau_a','C_b','tau_b']].values)
    chi_rand = calc_chi(ResCorrDF_GB3_NVT_CF02ps.loc[lnt_steps_inFullT].values[:,r],
                        rand_model, RESCorrDF_GB3_NVT_CF02ps_STD.loc[lnt_steps_inFullT].values[:,r])
    print(FitDF_GB3_NVT_02ps_FixTM11_lnt_restB_randtb.loc[r,'Chi_Fit'], chi_rand)
    #itDF_GB3_NVT_02ps_FixTM11_lnt_restB_randtb.loc[r,'Chi_Fit'] = chi_rand 

In [ ]:
FitDF_GB3_NVT_02ps_FixTM11_lnt_restB_randtb.iloc[r]['C_a']

In [ ]:
ResCorrDF_GB3_NVT_CF02ps.iloc[1,r]

In [ ]:
np.where(FitDF_GB3_NVT_02ps_FixTM11_lnt_restB_randtb['C_b']==0)[0]

In [ ]:
calc_chi(ResCorrDF_GB3_NVT_CF02ps.iloc[1:,r].values, rand_model, RESCorrDF_GB3_NVT_CF02ps_STD.iloc[1:,r].values)

In [ ]:
FitDF_GB3_NVT_02ps_FixTM11_FixFit_lnt, covarMat_lists = fitCorrF(ResCorrDF_GB3_NVT_CF02ps,
                                                      RESCorrDF_GB3_NVT_CF02ps_STD, 11.0, [4],
                                                      tstart=1, tcorr=3.53, tscl=2.5,
                                                      logt=lnt_steps,
                                                      fixfit=True, first_neg=False)
FitDF_GB3_NVT_02ps_FixTM11_FixFit_lnt['Resname']= FitDF_GB3_NVT_02ps_FixTM11_FixFit_lnt['Resname'].astype('int64')

In [ ]:
not (not True)

In [ ]:

#FitDF_GB3_NVT_02ps_FixTM11_lnt_SameT1B.plot(y='tau_b',logy=True,ax=plt.gca())
FitDF_GB3_NVT_02ps_FixTM11_lnt_restB.plot(y='tau_b',logy=True,ax=plt.gca())
FitDF_GB3_NVT_02ps_FixTM11_FixFit_lnt.plot(y='tau_b',logy=True,ax=plt.gca())
FitDF_GB3_NVT_02ps_FixTM11_lnt_restB2.plot(y='tau_b',logy=True,ax=plt.gca())

In [ ]:
thresh=0.50; tau_mem=14.0;
FitDF_GB3_NVT_CF2ps_FixTM16, covarMat_lists = fitCorrF(ResCorrDF_GB3_NVT_CF2ps,
                                                 RESCorrDF_GB3_NVT_CF2ps_STD, 16.0, [2,4],
                                                 tstart=1, tcorr=3.53, tscl=3.0,
                                                 logt=[],
                                                 fixfit=False, first_neg=False)
FitDF_GB3_NVT_CF2ps_FixTM16['Resname']= FitDF_GB3_NVT_CF2ps_FixTM16['Resname'].astype('int64')

In [ ]:
thresh=0.50; tau_mem=14.0;
FitDF_GB3_NVT_CF2ps_FixTM16_lnt, covarMat_lists = fitCorrF(ResCorrDF_GB3_NVT_CF2ps,
                                                 RESCorrDF_GB3_NVT_CF2ps_STD, 16.0, [2,4],
                                                 tstart=1, tcorr=3.53, tscl=4.0,
                                                 logt=lnt_steps,
                                                 fixfit=False, first_neg=False)
FitDF_GB3_NVT_CF2ps_FixTM16_lnt['Resname']= FitDF_GB3_NVT_CF2ps_FixTM16_lnt['Resname'].astype('int64')

In [ ]:
thresh=0.50; tau_mem=14.0;
FitDF_GB3_NVT_CF2ps_FixTM16_FixFit_lnt, covarMat_lists = fitCorrF(ResCorrDF_GB3_NVT_CF2ps,
                                                 RESCorrDF_GB3_NVT_CF2ps_STD, 16.0, [4],
                                                 tstart=1, tcorr=3.53, tscl=4.0,
                                                 logt=lnt_steps,
                                                 fixfit=True, first_neg=False)
FitDF_GB3_NVT_CF2ps_FixTM16_FixFit_lnt['Resname']= FitDF_GB3_NVT_CF2ps_FixTM16_FixFit_lnt['Resname'].astype('int64')

In [ ]:
FitDF_GB3_NVT_20ps_FixFit_FixTM50_LnT, covarMat_lists = fitCorrF(ResCorrDF_GB3_NVT_CF20ps,
                                                                 RESCorrDF_GB3_NVT_CF20ps_STD, 50.0, [4],
                                                                 tstart=1, tcorr=3.53, tscl=10.0,
                                                                 logt=lnt_steps,
                                                                 fixfit=True, first_neg=False)
FitDF_GB3_NVT_20ps_FixFit_FixTM50_LnT['Resname']= FitDF_GB3_NVT_20ps_FixFit_FixTM50_LnT['Resname'].astype('int64')

In [ ]:
fig_compcorr = plt.figure(3, figsize=(8,6))
_plot_CorrelationFunctionCompare('Analysis/CompareFit_BiExp_Lnt', 'NVEvsNVT_lnt',
                                 fig_compcorr, [FitDF_GB3_NVE_Opt_FixTM10, FitDF_GB3_NVE_Opt_FixTM10_lnt,
                                                FitDF_GB3_NVT_FixTM14, FitDF_GB3_NVT_FixTM14_lnt],
                                               [OptResCorrDF_GB3_NVE_Avg,
                                                    ResCorrDF_GB3_NVT_CF2ps],
                                                   fitdf_names=['NVE Fit TM 10.0ns', 'NVE Fit TM 10.0ns-ln(t)',
                                                                'NVT Fit TM 14.0ns', 'NVT Fit TM 14.0ns-ln(t)'],
                                                   mdnames=['MD NVE', 'MD NVT: 2 ps^(-1)'])

In [ ]:
FitDF_GB3_NVE_Opt_FixTM10['Chi_Fit'].plot(logy=True, color='m')
FitDF_GB3_NVE_Opt_FixTM10_lnt['Chi_Fit'].plot(logy=True, color='b')
FitDF_GB3_NVT_FixTM14['Chi_Fit'].plot(ax=plt.gca(), color='g')
FitDF_GB3_NVT_FixTM14_lnt['Chi_Fit'].plot(ax=plt.gca(), color='r')

In [ ]:
fig_compcorr = plt.figure(3, figsize=(8,6))
_plot_CorrelationFunctionCompare('Analysis/CompareFt_VariableExp', 'NVEvsNVT',
                                 fig_compcorr, [FitDF_GB3_NVE_Opt_FixTM10_lnt,
                                                FitDF_GB3_NVT_02ps_FixTM11_lnt,
                                                FitDF_GB3_NVT_CF2ps_FixTM16_lnt],
                                               [OptResCorrDF_GB3_NVE_Avg,
                                                ResCorrDF_GB3_NVT_CF02ps,
                                                ResCorrDF_GB3_NVT_CF2ps],
                                                   fitdf_names=['NVE Fit TM 10.0ns',
                                                                'CF 0.2 ps^-1: Fit TM 11.0ns',
                                                                'CF 2 ps^-1: Fit TM 16.0ns'],
                                                   mdnames=['MD NVE', 'MD NVT: 0.2 ps^(-1)', 'MD NVT: 2 ps^(-1)'])

In [ ]:
figcompopt, axcompopt = plt.subplots(2, 2, sharex=True, figsize=(18, 14))
figcompopt.subplots_adjust(hspace=0.1,wspace=0.1)

FitDF_GB3_NVE_Opt_FixTM10_FixFit_lnt[['Resname','C_a','C_a_err']].plot(x='Resname', y='C_a', yerr='C_a_err', c='blue',
                                                                ax=axcompopt[0,0], label=r'$\mathbf{NVE}$',
                                                         linewidth=2, marker='o')

FitDF_GB3_NVT_02ps_FixTM11_FixFit_lnt[['Resname','C_a','C_a_err']].plot(x='Resname', y='C_a', yerr='C_a_err', c='orange',
                                                               ax=axcompopt[0,0],
                                                               label=r'$\mathbf{ NVT : \gamma = 0.2 \ \ ps^{-1}}$',
                                                               linewidth=2, marker='*')

FitDF_GB3_NVT_CF2ps_FixTM16_FixFit_lnt[['Resname','C_a','C_a_err']].plot(x='Resname', y='C_a', yerr='C_a_err', c='green',
                                                               ax=axcompopt[0,0],
                                                               label=r'$\mathbf{ NVT : \gamma = 2 \ \ ps^{-1}}$',
                                                               linewidth=2, marker='d')

axcompopt[0,0].set_ylabel('Amplitude', fontsize=15, weight='bold')


FitDF_GB3_NVE_Opt_FixTM10_FixFit_lnt[['Resname','C_b','C_b_err']].plot(x='Resname', y='C_b', yerr='C_b_err', c='blue',
                                                                ax=axcompopt[0,1],  label=r'$\mathbf{NVE}$',
                                                        linewidth=2, marker='o')

FitDF_GB3_NVT_02ps_FixTM11_FixFit_lnt[['Resname','C_b','C_b_err']].plot(x='Resname', y='C_b', yerr='C_b_err', c='orange',
                                                               ax=axcompopt[0,1],
                                                               label=r'$\mathbf{ NVT : \gamma = 0.2 \ \ ps^{-1}}$',
                                                               linewidth=2, marker='*')

FitDF_GB3_NVT_CF2ps_FixTM16_FixFit_lnt[['Resname','C_b','C_b_err']].plot(x='Resname', y='C_b', yerr='C_b_err', c='green',
                                                               ax=axcompopt[0,1],
                                                               label=r'$\mathbf{ NVT : \gamma = 2 \ \ ps^{-1}}$',
                                                               linewidth=2, marker='d')

FitDF_GB3_NVE_Opt_FixTM10_FixFit_lnt[['Resname','tau_a','tau_a_err']].plot(x='Resname', y='tau_a', yerr='tau_a_err', c='blue',
                                                                    ax=axcompopt[1,0], label=r'$\mathbf{NVE}$',
                                                             linewidth=2, marker='o')
FitDF_GB3_NVT_02ps_FixTM11_FixFit_lnt[['Resname','tau_a','tau_a_err']].plot(x='Resname', y='tau_a', yerr='tau_a_err', c='orange',
                                                                   ax=axcompopt[1,0],
                                                                   label=r'$\mathbf{ NVT : \gamma = 0.2 \ \ ps^{-1}}$', 
                                                                   linewidth=2, marker='*')
FitDF_GB3_NVT_CF2ps_FixTM16_FixFit_lnt[['Resname','tau_a','tau_a_err']].plot(x='Resname', y='tau_a', yerr='tau_a_err', c='green',
                                                                   ax=axcompopt[1,0],
                                                                   label=r'$\mathbf{ NVT : \gamma = 2 \ \ ps^{-1}}$', 
                                                                   linewidth=2, marker='d')
axcompopt[1,0].set_ylim(2, 8)
axcompopt[1,0].set_ylabel('Correlation Time (ns)', fontsize=15, weight='bold')

FitDF_GB3_NVE_Opt_FixTM10_FixFit_lnt[['Resname','tau_b','tau_b_err']].plot(x='Resname', y='tau_b', yerr='tau_b_err', c='blue',
                                                                    ax=axcompopt[1,1], label=r'$\mathbf{NVE}$',
                                                             logy=True, linewidth=2, marker='o')
FitDF_GB3_NVT_02ps_FixTM11_FixFit_lnt[['Resname','tau_b','tau_b_err']].plot(x='Resname', y='tau_b', yerr='tau_b_err', c='orange',
                                                                   ax=axcompopt[1,1],
                                                                   label=r'$\mathbf{ NVT : \gamma = 0.2 \ \ ps^{-1}}$', 
                                                                   linewidth=2, marker='*')
FitDF_GB3_NVT_CF2ps_FixTM16_FixFit_lnt[['Resname','tau_b','tau_b_err']].plot(x='Resname', y='tau_b', yerr='tau_b_err', c='green',
                                                                   ax=axcompopt[1,1],
                                                                   label=r'$\mathbf{ NVT : \gamma = 2 \ \ ps^{-1}}$', 
                                                                   linewidth=2, marker='d')

axcompopt[1,1].set_ylim(1e-4, 5e0)

for axcpt,ttext in zip(axcompopt.flatten(), [r'$\mathbf{A_{1}}$', r'$\mathbf{A_{2}}$',
                                             r'$\mathbf{\tau_{1}}$', r'$\mathbf{\tau_{2}}$']):
    axcpt.tick_params(labelsize=16)
    axcpt.legend(frameon=False, prop={'size':15})
    axcpt.set_title(ttext, weight='bold', fontsize=18,  y=1.0)
    
for axcpt in axcompopt.flatten()[0:2]:
    axcpt.set_ylim(0,1.0)

for axcpt in axcompopt.flatten()[2:]:
    axcpt.set_xticks(np.arange(0,60,10))
    axcpt.set_xticks(np.arange(0,60,5), minor=True)
    axcpt.set_xlabel('Residue Number', weight='bold', fontsize=16)
    
figcompopt.savefig('{}GB3/Analysis/Compare_NVE_NVTCf2ps_FittingParameters_linear_lnt_FixTM_FixFit_tbm150ps.png'.format(CorrFuncLoc2), dpi=600, bbox_inches='tight')

In [ ]:
figcompopt, axcompopt = plt.subplots(2, 2, sharex=True, figsize=(18, 14))
figcompopt.subplots_adjust(hspace=0.1,wspace=0.1)

FitDF_GB3_NVE_Opt_FirstNeg_t1_lin_unb[['Resname','C_a','C_a_err']].plot(x='Resname', y='C_a', yerr='C_a_err', c='blue',
                                                                ax=axcompopt[0,0], label=r'$\mathbf{NVE}$',
                                                         linewidth=2, marker='o')

FitDF_GB3_NVT_02ps_FirstNeg_t1_lin_unb[['Resname','C_a','C_a_err']].plot(x='Resname', y='C_a', yerr='C_a_err', c='orange',
                                                               ax=axcompopt[0,0],
                                                               label=r'$\mathbf{ NVT : \gamma = 0.2 \ \ ps^{-1}}$',
                                                               linewidth=2, marker='*')

FitDF_GB3_NVT_2ps_FirstNeg_t1_lin_unb[['Resname','C_a','C_a_err']].plot(x='Resname', y='C_a', yerr='C_a_err', c='green',
                                                               ax=axcompopt[0,0],
                                                               label=r'$\mathbf{ NVT : \gamma = 2 \ \ ps^{-1}}$',
                                                               linewidth=2, marker='d')

axcompopt[0,0].set_ylabel('Amplitude', fontsize=15, weight='bold')


FitDF_GB3_NVE_Opt_FirstNeg_t1_lin_unb[['Resname','C_b','C_b_err']].plot(x='Resname', y='C_b', yerr='C_b_err', c='blue',
                                                                ax=axcompopt[0,1],  label=r'$\mathbf{NVE}$',
                                                        linewidth=2, marker='o')

FitDF_GB3_NVT_02ps_FirstNeg_t1_lin_unb[['Resname','C_b','C_b_err']].plot(x='Resname', y='C_b', yerr='C_b_err', c='orange',
                                                               ax=axcompopt[0,1],
                                                               label=r'$\mathbf{ NVT : \gamma = 0.2 \ \ ps^{-1}}$',
                                                               linewidth=2, marker='*')

FitDF_GB3_NVT_2ps_FirstNeg_t1_lin_unb[['Resname','C_b','C_b_err']].plot(x='Resname', y='C_b', yerr='C_b_err', c='green',
                                                               ax=axcompopt[0,1],
                                                               label=r'$\mathbf{ NVT : \gamma = 2 \ \ ps^{-1}}$',
                                                               linewidth=2, marker='d')

FitDF_GB3_NVE_Opt_FirstNeg_t1_lin_unb[['Resname','tau_a','tau_a_err']].plot(x='Resname', y='tau_a', yerr='tau_a_err', c='blue',
                                                                    ax=axcompopt[1,0], label=r'$\mathbf{NVE}$',
                                                             linewidth=2, marker='o')
FitDF_GB3_NVT_02ps_FirstNeg_t1_lin_unb[['Resname','tau_a','tau_a_err']].plot(x='Resname', y='tau_a', yerr='tau_a_err', c='orange',
                                                                   ax=axcompopt[1,0],
                                                                   label=r'$\mathbf{ NVT : \gamma = 0.2 \ \ ps^{-1}}$', 
                                                                   linewidth=2, marker='*')
FitDF_GB3_NVT_2ps_FirstNeg_t1_lin_unb[['Resname','tau_a','tau_a_err']].plot(x='Resname', y='tau_a', yerr='tau_a_err', c='green',
                                                                   ax=axcompopt[1,0],
                                                                   label=r'$\mathbf{ NVT : \gamma = 2 \ \ ps^{-1}}$', 
                                                                   linewidth=2, marker='d')
axcompopt[1,0].set_ylim(2, 8)
axcompopt[1,0].set_ylabel('Correlation Time (ns)', fontsize=15, weight='bold')

FitDF_GB3_NVE_Opt_FirstNeg_t1_lin_unb[['Resname','tau_b','tau_b_err']].plot(x='Resname', y='tau_b', yerr='tau_b_err', c='blue',
                                                                    ax=axcompopt[1,1], label=r'$\mathbf{NVE}$',
                                                             logy=True, linewidth=2, marker='o')
FitDF_GB3_NVT_02ps_FirstNeg_t1_lin_unb[['Resname','tau_b','tau_b_err']].plot(x='Resname', y='tau_b', yerr='tau_b_err', c='orange',
                                                                   ax=axcompopt[1,1],
                                                                   label=r'$\mathbf{ NVT : \gamma = 0.2 \ \ ps^{-1}}$', 
                                                                   linewidth=2, marker='*')
FitDF_GB3_NVT_2ps_FirstNeg_t1_lin_unb[['Resname','tau_b','tau_b_err']].plot(x='Resname', y='tau_b', yerr='tau_b_err', c='green',
                                                                   ax=axcompopt[1,1],
                                                                   label=r'$\mathbf{ NVT : \gamma = 2 \ \ ps^{-1}}$', 
                                                                   linewidth=2, marker='d')

axcompopt[1,1].set_ylim(1e-4, 5e0)

for axcpt,ttext in zip(axcompopt.flatten(), [r'$\mathbf{A_{1}}$', r'$\mathbf{A_{2}}$',
                                             r'$\mathbf{\tau_{1}}$', r'$\mathbf{\tau_{2}}$']):
    axcpt.tick_params(labelsize=16)
    axcpt.legend(frameon=False, prop={'size':15})
    axcpt.set_title(ttext, weight='bold', fontsize=18,  y=1.0)
    
for axcpt in axcompopt.flatten()[0:2]:
    axcpt.set_ylim(0,1.0)

for axcpt in axcompopt.flatten()[2:]:
    axcpt.set_xticks(np.arange(0,60,10))
    axcpt.set_xticks(np.arange(0,60,5), minor=True)
    axcpt.set_xlabel('Residue Number', weight='bold', fontsize=16)
    
figcompopt.savefig('{}GB3/Analysis/Compare_NVE_NVTCf2ps_FittingParameters_FirstNeg_wCf02ps_TS1_linear_unbounded.png'.format(CorrFuncLoc2), dpi=600, bbox_inches='tight')

In [ ]:
## Save the GB3 fits
#FitDF_GB3_NVE.to_csv('{}{}/PROD_NVE/FitDF_Fix2Exp_tm11ns_FixTM_SoftL1_T00.csv'.format(CorrFuncLoc2, 'GB3'))
FitDF_GB3_NVE_Opt_FixTM10_FixFit_lnt.to_csv('{}{}/PROD_NVE/FitDF_OptSim_Fix2Exp_tm10ns_FixTM_FixFit_LnT_tbm150ps.csv'.format(CorrFuncLoc2, 'GB3'))
FitDF_GB3_NVT_02ps_FixTM11_FixFit_lnt.to_csv('{}{}/PROD_NVT/CF0-2ps/FitDF_Fix2Exp_Coupling02ps_tm11ns_FixTM_FixFit_LnT_tbm150ps.csv'.format(CorrFuncLoc2,'GB3'))
FitDF_GB3_NVT_CF2ps_FixTM16_FixFit_lnt.to_csv('{}{}/PROD_NVT/CF2ps/FitDF_Fix2Exp_Coupling2ps_tm16ns_FixTM_FixFit_LnT_tbm150ps.csv'.format(CorrFuncLoc2,'GB3'))
FitDF_GB3_NVT_20ps_FixFit_FixTM50_LnT.to_csv('{}{}/PROD_NVT/CF20ps/FitDF_Fix2Exp_Coupling20ps_tm50ns_FixTM_FixFit_LnT_tbm150ps.csv'.format(CorrFuncLoc2,'GB3'))

## ShK and Trp-Cage Fits

In [ ]:
pd.read_csv('D:/Research/LangevinDynamics_RotationalDiffusion/SHK/ShK_AverageFullCorrelationFunctions_NVE.csv')

In [ ]:
##Testing TRP-Cage Fits
AvgResCorrFunctions_TrpCage_NVE = pd.read_csv('D:/Research/LangevinDynamics_RotationalDiffusion/TRP-Cage/AverageFullCorrelationFunctions_NVE.csv', index_col=0)
STDResCorrFunctions_TrpCage_NVE = pd.read_csv('D:/Research/LangevinDynamics_RotationalDiffusion/TRP-Cage/STDFullCorrelationFunctions_NVE.csv', index_col=0)

AvgResCorrFunctions_TrpCage_NVT_Cf2ps = pd.read_csv('D:/Research/LangevinDynamics_RotationalDiffusion/TRP-Cage/AverageFullCorrelationFunctions_NVT_CF2ps.csv', index_col=0)
STDResCorrFunctions_TrpCage_NVT_Cf2ps = pd.read_csv('D:/Research/LangevinDynamics_RotationalDiffusion/TRP-Cage/STDFullCorrelationFunctions_NVT_CF2ps.csv', index_col=0)


In [ ]:
tau_mem=4.0; thresh=0.25; fthresh=False; ShortSim=False; FixFit=True; Fix_Tf=False;
FitDF_TRP_NVE_Tstop0_BT0, covarMat_lists = fitCorrF(AvgResCorrFunctions_TrpCage_NVE,
                                     STDResCorrFunctions_TrpCage_NVE, tau_mem, [4], 0, fixfit=FixFit, first_neg=False)


FitDF_TRP_NVT_2ps_Tstop0_BT0, covarMat_lists = fitCorrF(AvgResCorrFunctions_TrpCage_NVT_Cf2ps,
                                     STDResCorrFunctions_TrpCage_NVT_Cf2ps, tau_mem, [4], 0, fixfit=FixFit, first_neg=False)

In [ ]:
FitDF_TRP_NVE_Tstop0_BT0.to_csv('{}/TRP-Cage/FitDF_Fix2Exp_NVE_FixTM_4ns.csv'.format(CorrFuncLoc2))
FitDF_TRP_NVT_2ps_Tstop0_BT0.to_csv('{}/TRP-Cage/FitDF_Fix2Exp_NVT_Cf2ps_FixTM_4ns.csv'.format(CorrFuncLoc2))

In [ ]:
figcval, axcvak = plt.subplots(2, 2, sharex=True, figsize=(16,14))
figcval.subplots_adjust(hspace=0.025,wspace=0.075)

#FitDF_SHK_Fix2Exp_tm10ns_Matt[['Resname','C_a','C_a_err']].plot(x='Resname', y='C_a', yerr='C_a_err', c='blue',
#                                                                ax=axcvak[0,0], label='Matt')
FitDF_TRP_NVE_Tstop0_BT0[['Resname','C_a','C_a_err']].plot(x='Resname', y='C_a', yerr='C_a_err', c='goldenrod',
                                                           ax=axcvak[0,0], label='Alan')

#FitDF_SHK_Fix2Exp_tm10ns_Matt[['Resname','C_b','C_b_err']].plot(x='Resname', y='C_b', yerr='C_b_err', c='blue',
#                                                                ax=axcvak[0,1],  label='Matt')
FitDF_TRP_NVE_Tstop0_BT0[['Resname','C_b','C_b_err']].plot(x='Resname', y='C_b', yerr='C_b_err', c='goldenrod',
                                                           ax=axcvak[0,1], label='Alan')

#FitDF_SHK_Fix2Exp_tm10ns_Matt[['Resname','tau_a','tau_a_err']].plot(x='Resname', y='tau_a', yerr='tau_a_err', c='blue',
#                                                                    ax=axcvak[1,0], label='Matt')
FitDF_TRP_NVE_Tstop0_BT0[['Resname','tau_a','tau_a_err']].plot(x='Resname', y='tau_a', yerr='tau_a_err', c='goldenrod',
                                                               ax=axcvak[1,0], label='Alan')

#FitDF_SHK_Fix2Exp_tm10ns_Matt[['Resname','tau_b','tau_b_err']].plot(x='Resname', y='tau_b', yerr='tau_b_err', c='blue',
#                                                                    ax=axcvak[1,1], label='Matt')
FitDF_TRP_NVE_Tstop0_BT0[['Resname','tau_b','tau_b_err']].plot(x='Resname', y='tau_b', yerr='tau_b_err', c='goldenrod',
                                                               ax=axcvak[1,1], label='Alan', logy=True)

figcval.savefig('{}/TRP-Cage/CrossValidate_SHK_Fix2Exp_TM10_Start1.png'.format(CorrFuncLoc2), bbox_inches='tight', dpi=600)

In [ ]:
## Testing SHK Fits 
ResCorrDF_SHK_NVE = pd.read_csv('{}SHK/ShK_AverageFullCorrelationFunctions_NVE.csv'.format(CorrFuncLoc2), index_col=0)
RESCorrDF_SHK_NVE_STD = pd.read_csv('{}SHK/ShK_STDFullCorrelationFunctions_NVE.csv'.format(CorrFuncLoc2), index_col=0)

ResCorrDF_SHK_NVT_CF2ps = pd.read_csv('{}SHK/ShK_AverageFullCorrelationFunctions_NVT_CF2ps.csv'.format(CorrFuncLoc2), index_col=0)
RESCorrDF_SHK_NVT_CF2ps_STD = pd.read_csv('{}SHK/ShK_AverageFullCorrelationFunctions_NVT_CF2ps.csv'.format(CorrFuncLoc2), index_col=0)

In [ ]:
tau_mem=10.0; thresh=0.25; fthresh=False; ShortSim=False; FixFit=True; Fix_Tf=False;
FitDF_SHK_NVE_Tstop0_BT0, covarMat_lists = fitCorrF(ResCorrDF_SHK_NVE,
                                     RESCorrDF_SHK_NVE_STD, tau_mem, [4], 0, fixfit=FixFit, first_neg=False)


FitDF_SHK_NVT_2ps_Tstop0_BT0, covarMat_lists = fitCorrF(ResCorrDF_SHK_NVT_CF2ps,
                                     RESCorrDF_SHK_NVT_CF2ps_STD, tau_mem, [4], 0, fixfit=FixFit, first_neg=False)

In [ ]:
FitDF_SHK_NVE_Tstop0_BT0.to_csv('{}/SHK/FitDF_Fix2Exp_NVE_FixTM_10ns.csv'.format(CorrFuncLoc2))
FitDF_SHK_NVT_2ps_Tstop0_BT0.to_csv('{}/SHK/FitDF_Fix2Exp_NVT_Cf2ps_FixTM_10ns.csv'.format(CorrFuncLoc2))

In [ ]:
FitDF_SHK_NVE_Tstop0_BT0['Resname'] = FitDF_SHK_NVE_Tstop0_BT0['Resname'].astype('int64')

In [ ]:
FitDF_SHK_Fix2Exp_tm10ns_Matt = pd.read_csv('{}/SHK/FitDF_Fix2Exp_tm10ns.csv'.format(CorrFuncLoc2), index_col=0)

In [ ]:
figcval, axcvak = plt.subplots(2, 2, sharex=True, figsize=(16,14))
figcval.subplots_adjust(hspace=0.025,wspace=0.075)

FitDF_SHK_Fix2Exp_tm10ns_Matt[['Resname','C_a','C_a_err']].plot(x='Resname', y='C_a', yerr='C_a_err', c='blue',
                                                                ax=axcvak[0,0], label='Matt')
FitDF_SHK_NVE_Tstop0_BT0[['Resname','C_a','C_a_err']].plot(x='Resname', y='C_a', yerr='C_a_err', c='goldenrod',
                                                           ax=axcvak[0,0], label='Alan')

FitDF_SHK_Fix2Exp_tm10ns_Matt[['Resname','C_b','C_b_err']].plot(x='Resname', y='C_b', yerr='C_b_err', c='blue',
                                                                ax=axcvak[0,1],  label='Matt')
FitDF_SHK_NVE_Tstop0_BT0[['Resname','C_b','C_b_err']].plot(x='Resname', y='C_b', yerr='C_b_err', c='goldenrod',
                                                           ax=axcvak[0,1], label='Alan')

FitDF_SHK_Fix2Exp_tm10ns_Matt[['Resname','tau_a','tau_a_err']].plot(x='Resname', y='tau_a', yerr='tau_a_err', c='blue',
                                                                    ax=axcvak[1,0], label='Matt')
FitDF_SHK_NVE_Tstop0_BT0[['Resname','tau_a','tau_a_err']].plot(x='Resname', y='tau_a', yerr='tau_a_err', c='goldenrod',
                                                               ax=axcvak[1,0], label='Alan')

FitDF_SHK_Fix2Exp_tm10ns_Matt[['Resname','tau_b','tau_b_err']].plot(x='Resname', y='tau_b', yerr='tau_b_err', c='blue',
                                                                    ax=axcvak[1,1], label='Matt')
FitDF_SHK_NVE_Tstop0_BT0[['Resname','tau_b','tau_b_err']].plot(x='Resname', y='tau_b', yerr='tau_b_err', c='goldenrod',
                                                               ax=axcvak[1,1], label='Alan', logy=True)

figcval.savefig('{}/SHK/CrossValidate_SHK_Fix2Exp_TM10.png'.format(CorrFuncLoc2), bbox_inches='tight', dpi=600)

In [ ]:
figcval, axcvak = plt.subplots(1,1, figsize=(8,6))
FitDF_SHK_Fix2Exp_tm10ns_Matt[['Resname','C_b','C_b_err']].plot(x='Resname', y='C_b', yerr='C_b_err', c='blue',ax=axcvak)
FitDF_SHK_NVE_Tstop0_BT0[['Resname','C_b','C_b_err']].plot(x='Resname', y='C_b', yerr='C_b_err', c='goldenrod',ax=axcvak)

In [ ]:
figcval, axcvak = plt.subplots(1,1, figsize=(8,6))
FitDF_SHK_Fix2Exp_tm10ns_Matt[['Resname','tau_a','tau_a_err']].plot(x='Resname', y='tau_a', yerr='tau_a_err', c='blue',ax=axcvak)
FitDF_SHK_NVE_Tstop0_BT0[['Resname','tau_a','tau_a_err']].plot(x='Resname', y='tau_a', yerr='tau_a_err', c='goldenrod',ax=axcvak)

In [ ]:
figcval, axcvak = plt.subplots(1,1, figsize=(8,6))
FitDF_SHK_Fix2Exp_tm10ns_Matt[['Resname','tau_b','tau_b_err']].plot(x='Resname', y='tau_b', yerr='tau_b_err', c='blue', logy=True, ax=axcvak)
FitDF_SHK_NVE_Tstop0_BT0[['Resname','tau_b','tau_b_err']].plot(x='Resname', y='tau_b', yerr='tau_b_err', c='goldenrod', logy=True,ax=axcvak)

## Load and calculate the average correlation functions for GB3 with overall tumbling removed
### Calculated from CPPTRAJ

In [ ]:
MFCorrDF_GB3_NVE, MFCorrDF_GB3_NVE_STD = _read_CorrFunctions(GB3atom_df, 'GB3', 'NVE', 0.5, 500000, corr_type='MF')

In [ ]:
MFCorrDF_GB3_NVE.to_csv('{}GB3/Analysis/AverageCorrelationFunctions_MF_NVE.csv'.format(CorrFuncLoc2))
MFCorrDF_GB3_NVE_STD.to_csv('{}GB3/Analysis/STDCorrelationFunctions_MF_NVE.csv'.format(CorrFuncLoc2))

In [ ]:
MFCorrDF_GB3_NVT_CF2ps, MFCorrDF_GB3_NVT_CF2ps_STD = _read_CorrFunctions(GB3atom_df, 'GB3','NVT',
                                                                           0.5, 500000, corr_type='MF',
                                                                           gamma=CouplingFrequency[1])

In [ ]:
MFCorrDF_GB3_NVT_CF2ps.to_csv('{}GB3/Analysis/AverageCorrelationFunctions_MF_NVT_2ps.csv'.format(CorrFuncLoc2))
MFCorrDF_GB3_NVT_CF2ps_STD.to_csv('{}GB3/Analysis/STDCorrelationFunctions_MF_NVT_2ps.csv'.format(CorrFuncLoc2))

In [ ]:
MFCorrDF_GB3_NVT_CF02ps, MFCorrDF_GB3_NVT_CF02ps_STD = _read_CorrFunctions(GB3atom_df, 'GB3','NVT',
                                                                           0.5, 500000, corr_type='MF',
                                                                           gamma=CouplingFrequency[0])

In [ ]:
MFCorrDF_GB3_NVT_CF02ps.to_csv('{}GB3/Analysis/AverageCorrelationFunctions_MF_NVT_02ps.csv'.format(CorrFuncLoc2))
MFCorrDF_GB3_NVT_CF02ps_STD.to_csv('{}GB3/Analysis/STDCorrelationFunctions_MF_NVT_02ps.csv'.format(CorrFuncLoc2))

In [ ]:
MFCorrDF_GB3_NVT_CF20ps, MFCorrDF_GB3_NVT_CF20ps_STD = _read_CorrFunctions(GB3atom_df, 'GB3','NVT',
                                                                           0.5, 500000, corr_type='MF',
                                                                           gamma=CouplingFrequency[2])

In [ ]:
MFCorrDF_GB3_NVT_CF20ps.to_csv('{}GB3/Analysis/AverageCorrelationFunctions_MF_NVT_20ps.csv'.format(CorrFuncLoc2))
MFCorrDF_GB3_NVT_CF20ps_STD.to_csv('{}GB3/Analysis/STDCorrelationFunctions_MF_NVT_20ps.csv'.format(CorrFuncLoc2))

In [ ]:
## Load GB3 Correlation Functions:
MFCorrDF_GB3_NVE = pd.read_csv('{}GB3/Analysis/AverageCorrelationFunctions_MF_NVE.csv'.format(CorrFuncLoc2), index_col=0)
MFCorrDF_GB3_NVE_STD = pd.read_csv('{}GB3/Analysis/STDCorrelationFunctions_MF_NVE.csv'.format(CorrFuncLoc2), index_col=0)

MFCorrDF_GB3_NVT_CF2ps = pd.read_csv('{}GB3/Analysis/AverageCorrelationFunctions_MF_NVT_2ps.csv'.format(CorrFuncLoc2), index_col=0)
MFCorrDF_GB3_NVT_CF2ps_STD = pd.read_csv('{}GB3/Analysis/STDCorrelationFunctions_MF_NVT_2ps.csv'.format(CorrFuncLoc2), index_col=0)

MFCorrDF_GB3_NVT_CF02ps = pd.read_csv('{}GB3/Analysis/AverageCorrelationFunctions_MF_NVT_02ps.csv'.format(CorrFuncLoc2), index_col=0)
MFCorrDF_GB3_NVT_CF02ps_STD = pd.read_csv('{}GB3/Analysis/STDCorrelationFunctions_MF_NVT_02ps.csv'.format(CorrFuncLoc2), index_col=0)

MFCorrDF_GB3_NVT_CF20ps = pd.read_csv('{}GB3/Analysis/AverageCorrelationFunctions_MF_NVT_20ps.csv'.format(CorrFuncLoc2), index_col=0)
MFCorrDF_GB3_NVT_CF20ps_STD = pd.read_csv('{}GB3/Analysis/STDCorrelationFunctions_MF_NVT_20ps.csv'.format(CorrFuncLoc2), index_col=0)

In [ ]:
MFCorrDF_GB3_NVE.loc[10.0,:].plot( style='-^' , label=r'$\mathbf{ NVE : \gamma = 0}$')
MFCorrDF_GB3_NVT_CF02ps.loc[10.0,:].plot( style='-s', label=r'$\mathbf{ NVT : \gamma = 0.2 \ \ ps^{-1}}$')
MFCorrDF_GB3_NVT_CF2ps.loc[10.0,:].plot( style='-o', label=r'$\mathbf{ NVT : \gamma = 2 \ \ ps^{-1}}$')
MFCorrDF_GB3_NVT_CF20ps.loc[10.0,:].plot( style='-d', label=r'$\mathbf{ NVT : \gamma = 20 \ \ ps^{-1}}$')

In [ ]:
## S^2 plot using the final time point of C(tau)

S2MFFig = plt.figure(327, figsize=(10,6))
axS2MF = S2MFFig.add_subplot(111)
MFCorrDF_GB3_NVE.iloc[-1,:].plot(ax = axS2MF, style='-^' , label=r'$\mathbf{ NVE : \gamma = 0}$')
MFCorrDF_GB3_NVT_CF02ps.iloc[-1,:].plot(ax = axS2MF, style='-s', label=r'$\mathbf{ NVT : \gamma = 0.2 \ \ ps^{-1}}$')
MFCorrDF_GB3_NVT_CF2ps.iloc[-1,:].plot(ax = axS2MF, style='-o', label=r'$\mathbf{ NVT : \gamma = 2 \ \ ps^{-1}}$')
MFCorrDF_GB3_NVT_CF20ps.iloc[-1,:].plot(ax = axS2MF, style='-d', label=r'$\mathbf{ NVT : \gamma = 20 \ \ ps^{-1}}$')
axS2MF.legend(loc=8, frameon=False, prop={'size':16})
axS2MF.set_ylabel(r'$\mathbf{S_{Long \ \ Time} ^{2}}$', fontsize=15)
axS2MF.tick_params(labelsize=15)
axS2MF.set_xlabel(r'Residue Number', fontsize=15, weight='bold')
axS2MF.set_ylim(0,1.0)
axS2MF.grid(linestyle='--',)
axS2MF.set_yticks(np.arange(0,1.1,0.1), minor=True)
S2MFFig.savefig('{}GB3/Analysis/OrderParameterS2_LongTime_CompareCoupling.png'.format(CorrFuncLoc2), dpi=600, bbox_inches='tight')

In [ ]:
## S^2 plot using the final time point of C(tau)

S2MFFig = plt.figure(327, figsize=(10,6))
axS2MF = S2MFFig.add_subplot(111)
MFCorrDF_GB3_NVE.loc[10.0,:].plot(ax = axS2MF, style='-^' , label=r'$\mathbf{ NVE : \gamma = 0}$')
MFCorrDF_GB3_NVT_CF02ps.loc[10.0,:].plot(ax = axS2MF, style='-s', label=r'$\mathbf{ NVT : \gamma = 0.2 \ \ ps^{-1}}$')
MFCorrDF_GB3_NVT_CF2ps.loc[10.0,:].plot(ax = axS2MF, style='-o', label=r'$\mathbf{ NVT : \gamma = 2 \ \ ps^{-1}}$')
MFCorrDF_GB3_NVT_CF20ps.loc[10.0,:].plot(ax = axS2MF, style='-d', label=r'$\mathbf{ NVT : \gamma = 20 \ \ ps^{-1}}$')
axS2MF.legend(loc=8, frameon=False, prop={'size':16})
axS2MF.set_ylabel(r'$\mathbf{S_{Long \ \ Time} ^{2}}$', fontsize=15)
axS2MF.tick_params(labelsize=15)
axS2MF.set_xlabel(r'Residue Number', fontsize=15, weight='bold')
axS2MF.set_ylim(0,1.0)
axS2MF.grid(linestyle='--',)
axS2MF.set_yticks(np.arange(0,1.1,0.1), minor=True)
S2MFFig.savefig('{}GB3/Analysis/OrderParameterS2_10ns_CompareCoupling.png'.format(CorrFuncLoc2), dpi=600, bbox_inches='tight')

In [ ]:
## Plot Correlation functions
MFCorrFig = plt.figure(1, figsize=(8,6))
for rcol in MFCorrDF_GB3_NVT_CF20ps.columns:
    axMF = MFCorrFig.add_subplot(111)
    
    MFCorrDF_GB3_NVE[rcol].plot(logx=True, ax = axMF, label=r'$\mathbf{ NVE }$')
    MFCorrDF_GB3_NVT_CF02ps[rcol].plot(logx=True, ax = axMF, label=r'$\mathbf{ NVT : \gamma = 0.2 \ \ ps^{-1}}$')
    MFCorrDF_GB3_NVT_CF2ps[rcol].plot(logx=True, ax = axMF, label=r'$\mathbf{ NVT : \gamma = 2 \ \ ps^{-1}}$')
    MFCorrDF_GB3_NVT_CF20ps[rcol].plot(logx=True, ax = axMF, label=r'$\mathbf{ NVT : \gamma = 20 \ \ ps^{-1}}$')
    axMF.set_title('Residue: {}'.format(rcol), fontsize=15, weight='bold')
    axMF.legend(loc=3)
    axMF.set_ylabel(r'$\mathbf{C(\tau)}$', fontsize=13)
    axMF.set_xlabel(r'$\mathbf{\tau \ \ (ns)}$',fontsize=13)
    axMF.tick_params(labelsize=15)
    axMF.set_ylim(0.0,1.0)
    MFCorrFig.savefig('{}GB3/Analysis/MFCorrelationFunctions/CompareCorrelationFunctions_WithCoupling_Res{}.png'.format(CorrFuncLoc2, rcol),dpi=600, bbox_inches='tight')
    MFCorrFig.clear()
    

### Load the Order Parameters calculated using CPPTRAJ and Python Script

In [ ]:
GB3_S2_NVE = read_OrderParameterFiles(GB3atom_df, 'GB3', 'NVE', gamma='CF2ps')
GB3_S2_CF2ps = read_OrderParameterFiles(GB3atom_df, 'GB3','NVT', gamma='CF2ps')
GB3_S2_CF02ps = read_OrderParameterFiles(GB3atom_df, 'GB3','NVT', gamma='CF0-2ps')
GB3_S2_CF20ps = read_OrderParameterFiles(GB3atom_df, 'GB3','NVT', gamma='CF20ps')

In [ ]:
MFCorrDF_GB3_NVE['5'].plot(logx=True)


## Perform the fits of the internal correlation functions

In [ ]:
tau_mem = 6.86 ## == 2x tau_c or the rotational correlation time of the protein
MFFits_GB3_NVE, cvmatris_NVE = FitMFCorr(MFCorrDF_GB3_NVE, MFCorrDF_GB3_NVE_STD, GB3_S2_NVE,
                                         tau_mem, True, parms=[3],  tstart=1)

MFFits_GB3_NVT_2ps, cvmatris_NVT = FitMFCorr(MFCorrDF_GB3_NVT_CF2ps, MFCorrDF_GB3_NVT_CF2ps_STD, GB3_S2_CF2ps,
                                             tau_mem, True, parms=[3],  tstart=1)

MFFits_GB3_NVT_02ps, cvmatris_NVT = FitMFCorr(MFCorrDF_GB3_NVT_CF02ps, MFCorrDF_GB3_NVT_CF02ps_STD, GB3_S2_CF02ps,
                                              tau_mem, True, parms=[3],  tstart=1)

MFFits_GB3_NVT_20ps, cvmatris_NVT = FitMFCorr(MFCorrDF_GB3_NVT_CF20ps, MFCorrDF_GB3_NVT_CF20ps_STD, GB3_S2_CF20ps,
                                              tau_mem, True, parms=[3],  tstart=1)


In [ ]:
MFFits_GB3_NVT_02ps[['C_a']]

In [ ]:
## Save the GB3 fits
MFFits_GB3_NVE.to_csv('{}{}/PROD_NVE/MFFitDF_Fix2Exp_tm11ns.csv'.format(CorrFuncLoc2, 'GB3'))
MFFits_GB3_NVT_02ps.to_csv('{}{}/PROD_NVT/CF0-2ps/MFFitDF_Fix2Exp_Coupling02ps_tm11ns.csv'.format(CorrFuncLoc2,'GB3'))
MFFits_GB3_NVT_2ps.to_csv('{}{}/PROD_NVT/CF2ps/MFFitDF_Fix2Exp_Coupling2ps_tm11ns.csv'.format(CorrFuncLoc2,'GB3'))
MFFits_GB3_NVT_20ps.to_csv('{}{}/PROD_NVT/CF20ps/MFFitDF_Fix2Exp_Coupling20ps_tm50ns.csv'.format(CorrFuncLoc2,'GB3'))

In [ ]:
MFFits_GB3_NVE['C_b'].plot()
GB3_S2_NVE.mean(axis=1).reset_index(drop=True).plot()
MFCorrDF_GB3_NVE.loc[11.0,:].reset_index(drop=True).plot()

In [ ]:
MFFits_GB3_NVE['tau_a'].plot(logy=True)
MFFits_GB3_NVT_02ps['tau_a'].plot(logy=True)
MFFits_GB3_NVT_2ps['tau_a'].plot(logy=True)
MFFits_GB3_NVT_20ps['tau_a'].plot(logy=True)

In [ ]:
FitDF_GB3_NVE['tau_b']

In [ ]:
FitDF_GB3_NVT_20ps['tau_a']/FitDF_GB3_NVE['tau_a']

In [ ]:
## Load the GB3 Fits:
FitDF_GB3_NVE = pd.read_csv('{}{}/PROD_NVE/FitDF_Fix2Exp_tm11ns.csv'.format(CorrFuncLoc2, 'GB3'))
FitDF_GB3_NVT_02ps = pd.read_csv('{}{}/PROD_NVT/CF0-2ps/FitDF_Fix2Exp_Coupling02ps_tm11ns.csv'.format(CorrFuncLoc2,'GB3'))
FitDF_GB3_NVT_2ps = pd.read_csv('{}{}/PROD_NVT/CF2ps/FitDF_Fix2Exp_Coupling2ps_tm11ns.csv'.format(CorrFuncLoc2,'GB3'))
FitDF_GB3_NVT_20ps = pd.read_csv('{}{}/PROD_NVT/CF20ps/FitDF_Fix2Exp_Coupling20ps_tm50ns.csv'.format(CorrFuncLoc2,'GB3'))

In [ ]:
FitDF_GB3_NVE['NumExp'].plot()
FitDF_GB3_NVT_02ps['NumExp'].plot()
FitDF_GB3_NVT_2ps['NumExp'].plot()
FitDF_GB3_NVT_20ps['NumExp'].plot()

## Load the Full Correlation Functions for Ubiquitin then save the averages

In [ ]:
ResCorrDF_UBQ_NVE, RESCorrDF_UBQ_NVE_STD = _read_CorrFunctions(UBQatom_df, 'Ubiquitin', 'NVE', 0.5, 500000, corr_type='NH')

In [ ]:
OptResCorrDF_UBQ_NVE_Avg, OptResCorrDF_UBQ_NVE_Std = _read_CorrFunctions_NVEopt(UBQatom_df, 'Ubiquitin', optimalUBQLoc, 0.5, 500000, corr_type='NH')

In [ ]:
OptResCorrDF_UBQ_NVE_Avg.to_csv('{}Ubiquitin/Analysis/Optimum_AverageFullCorrelationFunctions_NVE.csv'.format(CorrFuncLoc2))
OptResCorrDF_UBQ_NVE_Std.to_csv('{}Ubiquitin/Analysis/Optimum_STDFullCorrelationFunctions_NVE.csv'.format(CorrFuncLoc2))

In [ ]:
ResCorrDF_UBQ_NVT_CF2ps, RESCorrDF_UBQ_NVT_CF2ps_STD = _read_CorrFunctions(UBQatom_df, 'Ubiquitin','NVT', 0.5, 500000, corr_type='NH')

In [ ]:
ResCorrDF_UBQ_NVT_CF2ps.to_csv('{}Ubiquitin/Analysis/AverageFullCorrelationFunctions_NVT_2ps.csv'.format(CorrFuncLoc2))
RESCorrDF_UBQ_NVT_CF2ps_STD.to_csv('{}Ubiquitin/Analysis/STDFullCorrelationFunctions_NVT_2ps.csv'.format(CorrFuncLoc2))

In [ ]:
ResCorrDF_UBQ_NVT_CF02ps, RESCorrDF_UBQ_NVT_CF02ps_STD = _read_CorrFunctions(UBQatom_df, 'Ubiquitin','NVT', 0.5, 500000, corr_type='NH', gamma=CouplingFrequency[0])

In [ ]:
ResCorrDF_UBQ_NVT_CF02ps.to_csv('{}Ubiquitin/Analysis/AverageFullCorrelationFunctions_NVT_02ps.csv'.format(CorrFuncLoc2))
RESCorrDF_UBQ_NVT_CF02ps_STD.to_csv('{}Ubiquitin/Analysis/STDFullCorrelationFunctions_NVT_02ps.csv'.format(CorrFuncLoc2))

In [ ]:
ResCorrDF_UBQ_NVT_CF20ps, RESCorrDF_UBQ_NVT_CF20ps_STD = _read_CorrFunctions(UBQatom_df, 'Ubiquitin', 'NVT', 0.5, 500000, corr_type='NH', gamma=CouplingFrequency[2])

In [ ]:
ResCorrDF_UBQ_NVT_CF20ps.to_csv('{}Ubiquitin/Analysis/AverageFullCorrelationFunctions_NVT_20ps.csv'.format(CorrFuncLoc2))
RESCorrDF_UBQ_NVT_CF20ps_STD.to_csv('{}Ubiquitin/Analysis/STDFullCorrelationFunctions_NVT_20ps.csv'.format(CorrFuncLoc2))

In [ ]:
## Load Ubiquitin Correlation Functions:
ResCorrDF_UBQ_NVE = pd.read_csv('{}Ubiquitin/Analysis/AverageFullCorrelationFunctions_NVE.csv'.format(CorrFuncLoc2), index_col=0)
RESCorrDF_UBQ_NVE_STD = pd.read_csv('{}Ubiquitin/Analysis/STDFullCorrelationFunctions_NVE.csv'.format(CorrFuncLoc2), index_col=0)

OptResCorrDF_UBQ_NVE_Avg = pd.read_csv('{}Ubiquitin/Analysis/Optimum_AverageFullCorrelationFunctions_NVE.csv'.format(CorrFuncLoc2), index_col=0)
OptResCorrDF_UBQ_NVE_Std = pd.read_csv('{}Ubiquitin/Analysis/Optimum_STDFullCorrelationFunctions_NVE.csv'.format(CorrFuncLoc2), index_col=0)

ResCorrDF_UBQ_NVT_CF2ps = pd.read_csv('{}Ubiquitin/Analysis/AverageFullCorrelationFunctions_NVT_2ps.csv'.format(CorrFuncLoc2), index_col=0)
RESCorrDF_UBQ_NVT_CF2ps_STD = pd.read_csv('{}Ubiquitin/Analysis/STDFullCorrelationFunctions_NVT_2ps.csv'.format(CorrFuncLoc2), index_col=0)

ResCorrDF_UBQ_NVT_CF02ps = pd.read_csv('{}Ubiquitin/Analysis/AverageFullCorrelationFunctions_NVT_02ps.csv'.format(CorrFuncLoc2), index_col=0)
RESCorrDF_UBQ_NVT_CF02ps_STD = pd.read_csv('{}Ubiquitin/Analysis/STDFullCorrelationFunctions_NVT_02ps.csv'.format(CorrFuncLoc2), index_col=0)

ResCorrDF_UBQ_NVT_CF20ps = pd.read_csv('{}Ubiquitin/Analysis/AverageFullCorrelationFunctions_NVT_20ps.csv'.format(CorrFuncLoc2), index_col=0)
RESCorrDF_UBQ_NVT_CF20ps_STD = pd.read_csv('{}Ubiquitin/Analysis/STDFullCorrelationFunctions_NVT_20ps.csv'.format(CorrFuncLoc2), index_col=0)

In [ ]:
## Plot Correlation functions
ResCorrFig = plt.figure(1, figsize=(8,6))
for rcol in ResCorrDF_UBQ_NVT_CF20ps.columns:
    axCF = ResCorrFig.add_subplot(111)
    
    ResCorrDF_UBQ_NVE[rcol].plot(logx=True, ax = axCF, label=r'$\mathbf{ NVE : \gamma = 0}$')
    ResCorrDF_UBQ_NVT_CF02ps[rcol].plot(logx=True, ax = axCF, label=r'$\mathbf{ NVT : \gamma = 0.2 \ \ ps^{-1}}$')
    ResCorrDF_UBQ_NVT_CF2ps[rcol].plot(logx=True, ax = axCF, label=r'$\mathbf{ NVT : \gamma = 2 \ \ ps^{-1}}$')
    ResCorrDF_UBQ_NVT_CF20ps[rcol].plot(logx=True, ax = axCF, label=r'$\mathbf{ NVT : \gamma = 20 \ \ ps^{-1}}$')
    axCF.set_title('Residue: {}'.format(rcol), fontsize=15, weight='bold')
    axCF.legend(loc=3)
    axCF.set_ylabel(r'$\mathbf{C(\tau)}$', fontsize=13)
    axCF.set_xlabel(r'$\mathbf{\tau \ \ (ns)}$',fontsize=13)
    axCF.tick_params(labelsize=15)
    ResCorrFig.savefig('{}Ubiquitin/Analysis/CompareCorrelationFunctions_WithCoupling_Res{}.png'.format(CorrFuncLoc2, rcol),dpi=600, bbox_inches='tight')
    ResCorrFig.clear()
    

In [ ]:
MFCorrDF_UBQ_NVE, MFCorrDF_UBQ_NVE_STD = _read_CorrFunctions(UBQatom_df, 'Ubiquitin', 'NVE', 0.5, 500000, corr_type='MF')

In [ ]:
MFCorrDF_UBQ_NVE.to_csv('{}Ubiquitin/PROD_NVE/AverageCorrelationFunctions_MF_NVE.csv'.format(CorrFuncLoc2))
MFCorrDF_UBQ_NVE_STD.to_csv('{}Ubiquitin/PROD_NVE/STDCorrelationFunctions_MF_NVE.csv'.format(CorrFuncLoc2))

In [ ]:
MFCorrDF_UBQ_NVT_CF2ps, MFCorrDF_UBQ_NVT_CF2ps_STD = _read_CorrFunctions(UBQatom_df, 'Ubiquitin','NVT', 0.5, 500000, corr_type='MF',
                                                                         gamma=CouplingFrequency[1])

In [ ]:
MFCorrDF_UBQ_NVT_CF2ps.to_csv('{}Ubiquitin/PROD_NVT/CF2ps/AverageCorrelationFunctions_MF_NVT_2ps.csv'.format(CorrFuncLoc2))
MFCorrDF_UBQ_NVT_CF2ps_STD.to_csv('{}Ubiquitin/PROD_NVT/CF2ps/STDCorrelationFunctions_MF_NVT_2ps.csv'.format(CorrFuncLoc2))

In [ ]:
MFCorrDF_UBQ_NVT_CF02ps, MFCorrDF_UBQ_NVT_CF02ps_STD = _read_CorrFunctions(UBQatom_df, 'Ubiquitin','NVT', 0.5, 500000, corr_type='MF', 
                                                                          gamma=CouplingFrequency[0])

In [ ]:
MFCorrDF_UBQ_NVT_CF02ps.to_csv('{}Ubiquitin/PROD_NVT/CF0-2ps/AverageCorrelationFunctions_MF_NVT_02ps.csv'.format(CorrFuncLoc2))
MFCorrDF_UBQ_NVT_CF02ps_STD.to_csv('{}Ubiquitin/PROD_NVT/CF0-2ps/STDCorrelationFunctions_MF_NVT_02ps.csv'.format(CorrFuncLoc2))

In [ ]:
MFCorrDF_UBQ_NVT_CF20ps, MFCorrDF_UBQ_NVT_CF20ps_STD = _read_CorrFunctions(UBQatom_df, 'Ubiquitin','NVT', 0.5, 500000, corr_type='MF',
                                                                          gamma=CouplingFrequency[2])

In [ ]:
MFCorrDF_UBQ_NVT_CF20ps.to_csv('{}Ubiquitin/PROD_NVT/CF20ps/AverageCorrelationFunctions_MF_NVT_20ps.csv'.format(CorrFuncLoc2))
MFCorrDF_UBQ_NVT_CF20ps_STD.to_csv('{}Ubiquitin/PROD_NVT/CF20ps/STDCorrelationFunctions_MF_NVT_20ps.csv'.format(CorrFuncLoc2))

In [ ]:
## Load GB3 Correlation Functions:
MFCorrDF_UBQ_NVE = pd.read_csv('{}Ubiquitin/PROD_NVE/AverageCorrelationFunctions_MF_NVE.csv'.format(CorrFuncLoc2), index_col=0)
MFCorrDF_UBQ_NVE_STD = pd.read_csv('{}Ubiquitin/PROD_NVE/STDCorrelationFunctions_MF_NVE.csv'.format(CorrFuncLoc2), index_col=0)

MFCorrDF_UBQ_NVT_CF2ps = pd.read_csv('{}Ubiquitin/PROD_NVT/CF2ps/AverageCorrelationFunctions_MF_NVT_2ps.csv'.format(CorrFuncLoc2), index_col=0)
MFCorrDF_UBQ_NVT_CF2ps_STD = pd.read_csv('{}Ubiquitin/PROD_NVT/CF2ps/STDCorrelationFunctions_MF_NVT_2ps.csv'.format(CorrFuncLoc2), index_col=0)

MFCorrDF_UBQ_NVT_CF02ps = pd.read_csv('{}Ubiquitin/PROD_NVT/CF0-2ps/AverageCorrelationFunctions_MF_NVT_02ps.csv'.format(CorrFuncLoc2), index_col=0)
MFCorrDF_UBQ_NVT_CF02ps_STD = pd.read_csv('{}Ubiquitin/PROD_NVT/CF0-2ps/STDCorrelationFunctions_MF_NVT_02ps.csv'.format(CorrFuncLoc2), index_col=0)

MFCorrDF_UBQ_NVT_CF20ps = pd.read_csv('{}Ubiquitin/PROD_NVT/CF20ps/AverageCorrelationFunctions_MF_NVT_20ps.csv'.format(CorrFuncLoc2), index_col=0)
MFCorrDF_UBQ_NVT_CF20ps_STD = pd.read_csv('{}Ubiquitin/PROD_NVT/CF20ps/STDCorrelationFunctions_MF_NVT_20ps.csv'.format(CorrFuncLoc2), index_col=0)

In [ ]:
## Plot Correlation functions
MFCorrFig = plt.figure(4, figsize=(8,6))
for rcol in ResCorrDF_UBQ_NVT_CF20ps.columns:
    axMF = MFCorrFig.add_subplot(111)
    
    MFCorrDF_UBQ_NVE[rcol].plot(logx=True, ax = axMF, label=r'$\mathbf{ NVE : \gamma = 0}$')
    MFCorrDF_UBQ_NVT_CF02ps[rcol].plot(logx=True, ax = axMF, label=r'$\mathbf{ NVT : \gamma = 0.2 \ \ ps^{-1}}$')
    MFCorrDF_UBQ_NVT_CF2ps[rcol].plot(logx=True, ax = axMF, label=r'$\mathbf{ NVT : \gamma = 2 \ \ ps^{-1}}$')
    MFCorrDF_UBQ_NVT_CF20ps[rcol].plot(logx=True, ax = axMF, label=r'$\mathbf{ NVT : \gamma = 20 \ \ ps^{-1}}$')
    axMF.set_title('Residue: {}'.format(rcol), fontsize=15, weight='bold')
    axMF.legend(loc=3)
    axMF.set_ylabel(r'$\mathbf{C(\tau)}$', fontsize=13)
    axMF.set_xlabel(r'$\mathbf{\tau \ \ (ns)}$',fontsize=13)
    axMF.tick_params(labelsize=15)
    axMF.set_ylim(0.0,1.0)
    MFCorrFig.savefig('{}Ubiquitin/Analysis/MFCorrelationFunctions/CompareCorrelationFunctions_WithCoupling_Res{}.png'.format(CorrFuncLoc2, rcol),dpi=600, bbox_inches='tight')
    MFCorrFig.clear()
    

In [ ]:
## S^2 plot using the final time point of C(tau)

S2MFFig = plt.figure(328, figsize=(10,6))
axS2MF = S2MFFig.add_subplot(111)
MFCorrDF_UBQ_NVE.iloc[-1,:].plot(ax = axS2MF, style='-^' , label=r'$\mathbf{ NVE : \gamma = 0}$')
MFCorrDF_UBQ_NVT_CF02ps.iloc[-1,:].plot(ax = axS2MF, style='-s', label=r'$\mathbf{ NVT : \gamma = 0.2 \ \ ps^{-1}}$')
MFCorrDF_UBQ_NVT_CF2ps.iloc[-1,:].plot(ax = axS2MF, style='-o', label=r'$\mathbf{ NVT : \gamma = 2 \ \ ps^{-1}}$')
MFCorrDF_UBQ_NVT_CF20ps.iloc[-1,:].plot(ax = axS2MF, style='-d', label=r'$\mathbf{ NVT : \gamma = 20 \ \ ps^{-1}}$')
axS2MF.legend(loc=8, frameon=False, prop={'size':16})
axS2MF.set_ylabel(r'$\mathbf{S_{Long \ \ Time} ^{2}}$', fontsize=15)
axS2MF.tick_params(labelsize=15)
axS2MF.set_xlabel(r'Residue Number', fontsize=15, weight='bold')
axS2MF.set_ylim(0,1.0)
axS2MF.grid(linestyle='--',)
axS2MF.set_yticks(np.arange(0,1.1,0.1), minor=True)
S2MFFig.savefig('{}Ubiquitin/Analysis/OrderParameterS2_LongTime_CompareCoupling.png'.format(CorrFuncLoc2), dpi=600, bbox_inches='tight')

In [ ]:
## S^2 plot using the final time point of C(tau)

S2MFFig = plt.figure(329, figsize=(10,6))
axS2MF = S2MFFig.add_subplot(111)
MFCorrDF_UBQ_NVE.loc[10.0,:].plot(ax = axS2MF, style='-^' , label=r'$\mathbf{ NVE : \gamma = 0}$')
MFCorrDF_UBQ_NVT_CF02ps.loc[10.0,:].plot(ax = axS2MF, style='-s', label=r'$\mathbf{ NVT : \gamma = 0.2 \ \ ps^{-1}}$')
MFCorrDF_UBQ_NVT_CF2ps.loc[10.0,:].plot(ax = axS2MF, style='-o', label=r'$\mathbf{ NVT : \gamma = 2 \ \ ps^{-1}}$')
MFCorrDF_UBQ_NVT_CF20ps.loc[10.0,:].plot(ax = axS2MF, style='-d', label=r'$\mathbf{ NVT : \gamma = 20 \ \ ps^{-1}}$')
axS2MF.legend(loc=8, frameon=False, prop={'size':16})
axS2MF.set_ylabel(r'$\mathbf{S_{Long \ \ Time} ^{2}}$', fontsize=15)
axS2MF.tick_params(labelsize=15)
axS2MF.set_xlabel(r'Residue Number', fontsize=15, weight='bold')
axS2MF.set_ylim(0,1.0)
axS2MF.grid(linestyle='--',)
axS2MF.set_yticks(np.arange(0,1.1,0.1), minor=True)
S2MFFig.savefig('{}Ubiquitin/Analysis/OrderParameterS2_10ns_CompareCoupling.png'.format(CorrFuncLoc2), dpi=600, bbox_inches='tight')

In [ ]:
ResCorrDF_UBQ_NVE['26'].plot(logx=True)
ResCorrDF_UBQ_NVT_CF02ps['26'].plot(logx=True)
ResCorrDF_UBQ_NVT_CF2ps['26'].plot(logx=True)
ResCorrDF_UBQ_NVT_CF20ps['26'].plot(logx=True)


## Fitting the Ubiquitin full correlation functions 

In [ ]:
tau_mem=11.0; thresh=0.25; fthresh=False; ShortSim=False; FixFit=True; Fix_Tf=True;
FitDF_UBQ_NVE, covarMat_lists = fitCorrF(ResCorrDF_UBQ_NVE,
                                     RESCorrDF_UBQ_NVE_STD, tau_mem, [4], 0, fixfit=FixFit)

In [ ]:
tau_mem=11.0; thresh=0.25; fthresh=False; ShortSim=False; FixFit=False; Fix_Tf=True;
OptFitDF_UBQ_NVE_Cons_t00, covarMat_lists = fitCorrF(OptResCorrDF_UBQ_NVE_Avg,
                                     OptResCorrDF_UBQ_NVE_Std, tau_mem, [2,4], 0, fixfit=FixFit)

In [ ]:
thresh=0.50; tau_mem=11.0;
OptFitDF_UBQ_NVE_FixTM12, covarMat_lists = fitCorrF(OptResCorrDF_UBQ_NVE_Avg,
                                                 OptResCorrDF_UBQ_NVE_Std, 12.0, [2,4],
                                                 tstart=1, tcorr=4.11, tscl=2.0,
                                                 logt=[],
                                                 fixfit=False, first_neg=False)
OptFitDF_UBQ_NVE_FixTM12['Resname']= OptFitDF_UBQ_NVE_FixTM12['Resname'].astype('int64')

In [ ]:
thresh=0.50; tau_mem=11.0;
OptFitDF_UBQ_NVE_FixTM12_LnT, covarMat_lists = fitCorrF(OptResCorrDF_UBQ_NVE_Avg,
                                                 OptResCorrDF_UBQ_NVE_Std, 12.0, [2,4],
                                                 tstart=1, tcorr=4.11, tscl=2.0,
                                                 logt=lnt_steps,
                                                 fixfit=False, first_neg=False)
OptFitDF_UBQ_NVE_FixTM12_LnT['Resname']= OptFitDF_UBQ_NVE_FixTM12_LnT['Resname'].astype('int64')

In [ ]:
thresh=0.50; tau_mem=11.0;
OptFitDF_UBQ_NVE_FixTM12_FixFit_LnT, covarMat_lists = fitCorrF(OptResCorrDF_UBQ_NVE_Avg,
                                                 OptResCorrDF_UBQ_NVE_Std, 12.0, [4],
                                                 tstart=1, tcorr=4.11, tscl=2.0,
                                                 logt=lnt_steps,
                                                 fixfit=True, first_neg=False)
OptFitDF_UBQ_NVE_FixTM12_FixFit_LnT['Resname']= OptFitDF_UBQ_NVE_FixTM12_FixFit_LnT['Resname'].astype('int64')

In [ ]:
#OptFitDF_UBQ_NVE_FixTM12['tau_b'].plot(logy=True)
OptFitDF_UBQ_NVE_FixTM12_LnT['tau_b'].plot(logy=True)
OptFitDF_UBQ_NVE_FixTM12_FixFit_LnT['tau_b'].plot(logy=True)

In [ ]:
thresh=0.50; tau_mem=12.0;
OptFitDF_UBQ_NVT_CF02ps_FixTM12, covarMat_lists = fitCorrF(ResCorrDF_UBQ_NVT_CF02ps,
                                                 RESCorrDF_UBQ_NVT_CF02ps_STD, 12.0, [2,4],
                                                 tstart=1, tcorr=4.11, tscl=3.0,
                                                 logt=[],
                                                 fixfit=False, first_neg=False)
OptFitDF_UBQ_NVT_CF02ps_FixTM12['Resname']= OptFitDF_UBQ_NVT_CF02ps_FixTM12['Resname'].astype('int64')

In [ ]:
thresh=0.50; tau_mem=13.0;
OptFitDF_UBQ_NVT_CF02ps_FixTM11_LnT, covarMat_lists = fitCorrF(ResCorrDF_UBQ_NVT_CF02ps,
                                                 RESCorrDF_UBQ_NVT_CF02ps_STD, 12.0, [2,4],
                                                 tstart=1, tcorr=4.11, tscl=3.0,
                                                 logt=lnt_steps,
                                                 fixfit=False, first_neg=False)
OptFitDF_UBQ_NVT_CF02ps_FixTM11_LnT['Resname']= OptFitDF_UBQ_NVT_CF02ps_FixTM12_LnT['Resname'].astype('int64')

In [ ]:
thresh=0.50; tau_mem=13.0;
OptFitDF_UBQ_NVT_CF02ps_FixTM12_FixFit_LnT, covarMat_lists = fitCorrF(ResCorrDF_UBQ_NVT_CF02ps,
                                                 RESCorrDF_UBQ_NVT_CF02ps_STD, 12.0, [4],
                                                 tstart=1, tcorr=4.11, tscl=2.5,
                                                 logt=lnt_steps,
                                                 fixfit=True, first_neg=False)
OptFitDF_UBQ_NVT_CF02ps_FixTM12_FixFit_LnT['Resname']= OptFitDF_UBQ_NVT_CF02ps_FixTM12_FixFit_LnT['Resname'].astype('int64')

In [ ]:
OptFitDF_UBQ_NVT_CF02ps_FixTM12_LnT[['tau_b','tau_b_err']].plot(y='tau_b', yerr='tau_b_err', logy=True)
OptFitDF_UBQ_NVT_CF02ps_FixTM13_FixFit_LnT[['tau_b','tau_b_err']].plot(y='tau_b', yerr='tau_b_err', logy=True, ax=plt.gca())

In [ ]:
OptFitDF_UBQ_NVT_CF02ps_FixTM13_FixFit_LnT['tau_b_err'].values

In [ ]:
FitDF_UBQ_NVT_02ps_Cons_t00, covarMat_lists = fitCorrF(ResCorrDF_UBQ_NVT_CF02ps,
                                     RESCorrDF_UBQ_NVT_CF02ps_STD, 12.5, [3], 0, fixfit=FixFit, first_neg=False)

In [ ]:
thresh=0.50; tau_mem=18.0;
OptFitDF_UBQ_NVT_CF2ps_FixTM15_LnT, covarMat_lists = fitCorrF(ResCorrDF_UBQ_NVT_CF2ps,
                                                 RESCorrDF_UBQ_NVT_CF2ps_STD, 15.0, [2,4],
                                                 tstart=1, tcorr=4.11, tscl=4.0,
                                                 logt=lnt_steps,
                                                 fixfit=False, first_neg=False)
OptFitDF_UBQ_NVT_CF02ps_FixTM15_LnT['Resname']= OptFitDF_UBQ_NVT_CF02ps_FixTM15_LnT['Resname'].astype('int64')

In [ ]:
thresh=0.50; tau_mem=15.0;
OptFitDF_UBQ_NVT_CF2ps_FixTM16, covarMat_lists = fitCorrF(ResCorrDF_UBQ_NVT_CF2ps,
                                                 RESCorrDF_UBQ_NVT_CF2ps_STD, 16.0, [2,4],
                                                 tstart=1, tcorr=4.11, tscl=4.0,
                                                 logt=[],
                                                 fixfit=False, first_neg=False)
OptFitDF_UBQ_NVT_CF2ps_FixTM16['Resname']= OptFitDF_UBQ_NVT_CF2ps_FixTM16['Resname'].astype('int64')

In [ ]:
FitDF_UBQ_NVT_2ps_FixTM16_FixFit_LnT, covarMat_lists = fitCorrF(ResCorrDF_UBQ_NVT_CF2ps,
                                                 RESCorrDF_UBQ_NVT_CF2ps_STD, 16.0, [4],
                                                 tstart=1, tcorr=4.11, tscl=4.0,
                                                 logt=lnt_steps,
                                                 fixfit=True, first_neg=False)
FitDF_UBQ_NVT_2ps_FixTM16_FixFit_LnT['Resname']= FitDF_UBQ_NVT_2ps_FixTM16_FixFit_LnT['Resname'].astype('int64')

In [ ]:
FitDF_UBQ_NVT_2ps_FixTM16_FixFit_LnT['tau_b'].plot(logy=True)

In [ ]:
fig_compcorr = plt.figure(3, figsize=(8,6))
_plot_CorrelationFunctionCompare('Analysis/CompCorrFunction_Fits', 'Ubiquitin', 'NVEvsNVT',
                                 fig_compcorr, [OptFitDF_UBQ_NVE_FixTM12_FixFit_LnT,
                                                OptFitDF_UBQ_NVT_CF02ps_FixTM13_FixFit_LnT,
                                                FitDF_UBQ_NVT_2ps_FixTM16_FixFit_LnT],
                                               [OptResCorrDF_UBQ_NVE_Avg,
                                                ResCorrDF_UBQ_NVT_CF02ps,
                                                ResCorrDF_UBQ_NVT_CF2ps],
                                                   fitdf_names=['NVE Fit TM 10.0ns',
                                                                'CF 0.2 ps^-1: Fit TM 11.0ns',
                                                                'CF 2 ps^-1: Fit TM 16.0ns'],
                                                   mdnames=['MD NVE', 'MD NVT: 0.2 ps^(-1)', 'MD NVT: 2 ps^(-1)'])

In [ ]:
FitDF_UBQ_NVT_20ps_FixTM50_FixFit_LnT, covarMat_lists = fitCorrF(ResCorrDF_UBQ_NVT_CF20ps,
                                                       RESCorrDF_UBQ_NVT_CF20ps_STD, 50.0, [4],
                                                       tstart=1, tcorr=4.11, tscl=15.0,
                                                       logt=lnt_steps,
                                                       fixfit=True, first_neg=False)
FitDF_UBQ_NVT_20ps_FixTM50_FixFit_LnT['Resname'] = FitDF_UBQ_NVT_20ps_FixTM50_FixFit_LnT['Resname'].astype('int64')

In [ ]:
#FitDF_UBQ_NVE.to_csv('{}{}/PROD_NVE/FitDF_Fix2Exp_tm11ns_Cons_t00.csv'.format(CorrFuncLoc2, 'Ubiquitin'))
OptFitDF_UBQ_NVE_FixTM12_FixFit_LnT.to_csv('{}{}/PROD_NVE/FitDF_OptSim_Fix2Exp_tm12ns_FixFit_LnT_tbm150ps.csv'.format(CorrFuncLoc2, 'Ubiquitin'))
OptFitDF_UBQ_NVT_CF02ps_FixTM12_FixFit_LnT.to_csv('{}{}/PROD_NVT/CF0-2ps/FitDF_Fix2Exp_Coupling02ps_tm12ns_FixFit_LnT_tbm150ps.csv'.format(CorrFuncLoc2,'Ubiquitin'))
FitDF_UBQ_NVT_2ps_FixTM16_FixFit_LnT.to_csv('{}{}/PROD_NVT/CF2ps/FitDF_Fix2Exp_Coupling2ps_tm16ns_FixFit_LnT_tbm150ps.csv'.format(CorrFuncLoc2,'Ubiquitin'))
FitDF_UBQ_NVT_20ps_FixTM50_FixFit_LnT.to_csv('{}{}/PROD_NVT/CF20ps/FitDF_Fit2Exp_Coupling20ps_tm50ns_FixFit_LnT_tbm150ps.csv'.format(CorrFuncLoc2,'Ubiquitin'))

In [ ]:
OptFitDF_UBQ_NVE_FixTM12_FixFit_LnT['Resname'] = OptFitDF_UBQ_NVE_FixTM12_FixFit_LnT['Resname'].astype('int64')
OptFitDF_UBQ_NVT_CF02ps_FixTM12_FixFit_LnT['Resname'] = OptFitDF_UBQ_NVT_CF02ps_FixTM12_FixFit_LnT['Resname'].astype('int64')
FitDF_UBQ_NVT_2ps_FixTM16_FixFit_LnT['Resname'] = FitDF_UBQ_NVT_2ps_FixTM16_FixFit_LnT['Resname'].astype('int64')
#FitDF_UBQ_NVT_20ps_Cons_t00['Resname'] = FitDF_UBQ_NVT_02ps_Cons_t00['Resname'].astype('int64')

figcompopt, axcompopt = plt.subplots(2, 2, sharex=True, figsize=(18, 14),num=49218)
figcompopt.subplots_adjust(hspace=0.1,wspace=0.1)

OptFitDF_UBQ_NVE_FixTM12_FixFit_LnT[['Resname','C_a','C_a_err']].plot(x='Resname', y='C_a', yerr='C_a_err', c='blue',
                                                                ax=axcompopt[0,0], label=r'$\mathbf{NVE}$',
                                                         linewidth=2, marker='o')

OptFitDF_UBQ_NVT_CF02ps_FixTM12_FixFit_LnT[['Resname','C_a','C_a_err']].plot(x='Resname', y='C_a', yerr='C_a_err', c='orange',
                                                               ax=axcompopt[0,0],
                                                               label=r'$\mathbf{ NVT : \gamma = 0.2 \ \ ps^{-1}}$',
                                                               linewidth=2, marker='d')

FitDF_UBQ_NVT_2ps_FixTM16_FixFit_LnT[['Resname','C_a','C_a_err']].plot(x='Resname', y='C_a', yerr='C_a_err', c='green',
                                                               ax=axcompopt[0,0],
                                                               label=r'$\mathbf{ NVT : \gamma = 2 \ \ ps^{-1}}$',
                                                               linewidth=2, marker='d')

axcompopt[0,0].set_ylabel('Amplitude', fontsize=15, weight='bold')


OptFitDF_UBQ_NVE_FixTM12_FixFit_LnT[['Resname','C_b','C_b_err']].plot(x='Resname', y='C_b', yerr='C_b_err', c='blue',
                                                                ax=axcompopt[0,1],  label=r'$\mathbf{NVE}$',
                                                        linewidth=2, marker='o')

OptFitDF_UBQ_NVT_CF02ps_FixTM12_FixFit_LnT[['Resname','C_b','C_b_err']].plot(x='Resname', y='C_b', yerr='C_b_err', c='orange',
                                                               ax=axcompopt[0,1],
                                                               label=r'$\mathbf{ NVT : \gamma = 0.2 \ \ ps^{-1}}$',
                                                               linewidth=2, marker='d')

FitDF_UBQ_NVT_2ps_FixTM16_FixFit_LnT[['Resname','C_b','C_b_err']].plot(x='Resname', y='C_b', yerr='C_b_err', c='green',
                                                               ax=axcompopt[0,1],
                                                               label=r'$\mathbf{ NVT : \gamma = 2 \ \ ps^{-1}}$',
                                                               linewidth=2, marker='d')

OptFitDF_UBQ_NVE_FixTM12_FixFit_LnT[['Resname','tau_a','tau_a_err']].plot(x='Resname', y='tau_a', yerr='tau_a_err', c='blue',
                                                                    ax=axcompopt[1,0], label=r'$\mathbf{NVE}$',
                                                             linewidth=2, marker='o')

OptFitDF_UBQ_NVT_CF02ps_FixTM12_FixFit_LnT[['Resname','tau_a','tau_a_err']].plot(x='Resname', y='tau_a', yerr='tau_a_err', c='orange',
                                                               ax=axcompopt[1,0],
                                                               label=r'$\mathbf{ NVT : \gamma = 0.2 \ \ ps^{-1}}$',
                                                               linewidth=2, marker='d')

FitDF_UBQ_NVT_2ps_FixTM16_FixFit_LnT[['Resname','tau_a','tau_a_err']].plot(x='Resname', y='tau_a', yerr='tau_a_err', c='green',
                                                                   ax=axcompopt[1,0],
                                                                   label=r'$\mathbf{ NVT : \gamma = 2 \ \ ps^{-1}}$', 
                                                                   linewidth=2, marker='d')
axcompopt[1,0].set_ylim(2, 12)
axcompopt[1,0].set_ylabel('Correlation Time (ns)', fontsize=15, weight='bold')

OptFitDF_UBQ_NVE_FixTM12_FixFit_LnT[['Resname','tau_b','tau_b_err']].plot(x='Resname', y='tau_b', yerr='tau_b_err', c='blue',
                                                                    ax=axcompopt[1,1], label=r'$\mathbf{NVE}$',
                                                             logy=True, linewidth=2, marker='o')

OptFitDF_UBQ_NVT_CF02ps_FixTM12_FixFit_LnT[['Resname','tau_b','tau_b_err']].plot(x='Resname', y='tau_b', yerr='tau_b_err', c='orange',
                                                               ax=axcompopt[1,1],
                                                               label=r'$\mathbf{ NVT : \gamma = 0.2 \ \ ps^{-1}}$',
                                                               linewidth=2, marker='d')

FitDF_UBQ_NVT_2ps_FixTM16_FixFit_LnT[['Resname','tau_b','tau_b_err']].plot(x='Resname', y='tau_b', yerr='tau_b_err', c='green',
                                                                   ax=axcompopt[1,1],
                                                                   label=r'$\mathbf{ NVT : \gamma = 2 \ \ ps^{-1}}$', 
                                                                   linewidth=2, marker='d')
axcompopt[1,1].set_ylim(1e-4, 1e0)


for axcpt,ttext in zip(axcompopt.flatten(), [r'$\mathbf{A_{1}}$', r'$\mathbf{A_{2}}$', r'$\mathbf{\tau_{1}}$', r'$\mathbf{\tau_{2}}$']):
    axcpt.tick_params(labelsize=16)
    axcpt.legend(frameon=False, prop={'size':15})
    axcpt.set_title(ttext, weight='bold', fontsize=18,  y=1.0)
    
for axcpt in axcompopt.flatten()[0:2]:
    axcpt.set_ylim(0, 1.0)

for axcpt in axcompopt.flatten()[2:]:
    axcpt.set_xticks(np.arange(0,76, 10))
    axcpt.set_xticks(np.arange(0,76, 5), minor=True)
    axcpt.set_xticklabels(np.arange(0,76,10))
    axcpt.set_xlabel('Residue Number', weight='bold', fontsize=16)
    
figcompopt.savefig('{}Ubiquitin/Analysis/Compare_NVE_NVTCf2ps_FixTM_FixFit_LnT_tbm150.png'.format(CorrFuncLoc2),
                   dpi=600, bbox_inches='tight')


In [ ]:
FitDF_UBQ_NVT_2ps.Resname

In [ ]:
(FitDF_UBQ_NVT_02ps[['tau_a', 'tau_b']]/FitDF_UBQ_NVE[['tau_a','tau_b']])['tau_a'].plot(logy=True, color='blue')
(FitDF_UBQ_NVT_2ps[['tau_a', 'tau_b']]/FitDF_UBQ_NVE[['tau_a','tau_b']])['tau_a'].plot(logy=True, color='green')
(FitDF_UBQ_NVT_20ps[['tau_a', 'tau_b']]/FitDF_UBQ_NVE[['tau_a','tau_b']])['tau_a'].plot(logy=True, color='red')

In [ ]:
(FitDF_GB3_NVT_02ps_Tstop0_BT0[['tau_a', 'tau_b']]/FitDF_GB3_NVE_Opt_TM11[['tau_a','tau_b']])['tau_b'].plot(logy=True, color='blue')
(FitDF_GB3_NVT_2ps_Tstop0_BT0[['tau_a', 'tau_b']]/FitDF_GB3_NVE_Opt_TM11[['tau_a','tau_b']])['tau_b'].plot(logy=True, color='green')
(FitDF_GB3_NVT_20ps[['tau_a', 'tau_b']]/FitDF_GB3_NVE_Opt_TM11[['tau_a','tau_b']])['tau_b'].plot(logy=True, color='red')

## Fitting the Ubiquitin internal correlation Functions

In [ ]:
UBQ_S2_NVE = read_OrderParameterFiles(UBQatom_df, 'Ubiquitin','NVE', gamma='CF2ps')
UBQ_S2_CF2ps = read_OrderParameterFiles(UBQatom_df, 'Ubiquitin','NVT', gamma='CF2ps')
UBQ_S2_CF02ps = read_OrderParameterFiles(UBQatom_df, 'Ubiquitin','NVT', gamma='CF0-2ps')
UBQ_S2_CF20ps = read_OrderParameterFiles(UBwwwQatom_df, 'Ubiquitin','NVT', gamma='CF20ps')

In [ ]:
tau_mem = 11.0
MFFits_UBQ_NVE, cvmatris_NVE = FitMFCorr(MFCorrDF_UBQ_NVE, MFCorrDF_UBQ_NVE_STD, UBQ_S2_NVE,
                                         tau_mem, True, parms=[2],  tstart=0)
MFFits_UBQ_NVT_2ps, cvmatris_NVT = FitMFCorr(MFCorrDF_UBQ_NVT_CF2ps, MFCorrDF_UBQ_NVT_CF2ps_STD, UBQ_S2_CF2ps,
                                             tau_mem, True, parms=[2],  tstart=0)
MFFits_UBQ_NVT_02ps, cvmatris_NVT = FitMFCorr(MFCorrDF_UBQ_NVT_CF02ps, MFCorrDF_UBQ_NVT_CF02ps_STD, UBQ_S2_CF02ps,
                                              tau_mem, True, parms=[2],  tstart=0)
MFFits_UBQ_NVT_20ps, cvmatris_NVT = FitMFCorr(MFCorrDF_UBQ_NVT_CF20ps, MFCorrDF_UBQ_NVT_CF20ps_STD, UBQ_S2_CF20ps,
                                              50.0, True, parms=[2],  tstart=0)


In [ ]:
## Save the GB3 fits
MFFits_UBQ_NVE.to_csv('{}{}/PROD_NVE/MFFitDF_Fix2Exp_tm11ns.csv'.format(CorrFuncLoc2, 'Ubiquitin'))
MFFits_UBQ_NVT_02ps.to_csv('{}{}/PROD_NVT/CF0-2ps/MFFitDF_Fix2Exp_Coupling02ps_tm11ns.csv'.format(CorrFuncLoc2,'Ubiquitin'))
MFFits_UBQ_NVT_2ps.to_csv('{}{}/PROD_NVT/CF2ps/MFFitDF_Fix2Exp_Coupling2ps_tm11ns.csv'.format(CorrFuncLoc2,'Ubiquitin'))
MFFits_UBQ_NVT_20ps.to_csv('{}{}/PROD_NVT/CF20ps/MFFitDF_Fix2Exp_Coupling20ps_tm50ns.csv'.format(CorrFuncLoc2,'Ubiquitin'))

In [ ]:
FitDF_UBQ_NVT_02ps.loc[10,['tau_a','tau_b','tau_g']]

In [ ]:
tc = np.log(tau_mem-10.0)
if np.isnan(tc) | (tc < -5.0):
    print(tc)

## Plotting the different parameters

In [ ]:
def _load_Experimental_Results(protein, prot_topology, min_thresh=0.4):
    

    CAInd = prot_topology.select('name CA and protein')
    CAatom_name = pd.Series(['{}'.format(prot_topology.atom(atmx)) for atmx in CAInd])
    atom_df = CAatom_name.str.split('-',expand=True).rename(columns={0:'RESNAME',1:'ATMNAME'})
    atom_df['RESNAME'] = atom_df['RESNAME'].apply(lambda rr: '{}'.format(rr[:3] + str(int(rr[3:])+1)))
    atom_df['RESID'] = atom_df['RESNAME'].apply(lambda rr: int(rr[3:]))
    
    exp_nmr_df = pd.DataFrame(index=np.arange(0,len(atom_df),1), columns=['RESNAME','RESID', 'R1', 'T1','R2','T2', 'NOE'])
    exp_nmr_df['RESNAME'] = atom_df['RESNAME']
    exp_nmr_df['RESID'] = atom_df['RESID']
    
    for nmr in ['R1','R2','NOE']:
        nmrdf = pd.read_csv('{}{}/{}_{}_Experiment_Capture_AllPoints.csv'.format(CorrFuncLoc2, protein, protein, nmr),
                            header=None, names=['BarNumber', nmr])
        exp_nmr_df[nmr] = nmrdf[nmr].values
    
    exp_nmr_df['T1'] = 1/exp_nmr_df['R1']
    exp_nmr_df['T2'] = 1/exp_nmr_df['R2']
    
    mask_noexp = nmrdf['NOE'] < min_thresh
    exp_nmr_df.loc[mask_noexp,['R1','T1','R2','T2','NOE']] = np.nan
    
    return exp_nmr_df

In [ ]:
GB3_NMRRelax_Exp

In [ ]:
GB3_NMRRelax_Exp = _load_Experimental_Results('GB3', GB3Top, min_thresh=0.4)
GB3_NMRRelax_Exp.to_csv('{}{}/GB3_NMRRelax_Experiment.csv'.format(CorrFuncLoc2,'GB3'))

In [ ]:
GB3_NMRRelax_Exp.dtypes

In [ ]:
UBQ_NMRRelax_EXPFromSI = pd.read_excel('{}{}/ExpNMRRelaxation_1995_TjandraSI.xlsx'.format(CorrFuncLoc2,'Ubiquitin'))
UBQ_NMRRelax_Exp = pd.DataFrame(index=UBQ_NMRRelax_EXPFromSI.index, columns=['R1','T1','R2','T2','NOE'])
UBQ_NMRRelax_Exp['RESNAME'] = UBQ_NMRRelax_EXPFromSI['RESNAME']
UBQ_NMRRelax_Exp['RESID'] = UBQ_NMRRelax_Exp['RESNAME'].apply(lambda x: int(x[1:]))
UBQ_NMRRelax_Exp['T1'] = UBQ_NMRRelax_EXPFromSI[['T1,1','T1,2']].mean(axis=1)
UBQ_NMRRelax_Exp['R1'] = 1/UBQ_NMRRelax_Exp['T1']
UBQ_NMRRelax_Exp['T2'] = UBQ_NMRRelax_EXPFromSI[['T2,1','T2,2']].mean(axis=1)
UBQ_NMRRelax_Exp['R2'] = 1/UBQ_NMRRelax_Exp['T2']
UBQ_NMRRelax_Exp['NOE'] = UBQ_NMRRelax_EXPFromSI[['NOE,1','NOE,2']].mean(axis=1)

In [ ]:
UBQ_NMRRelax_Exp.dtypes

In [ ]:
UBQ_NMRRelax_Exp.to_csv('{}{}/UBQ_NMRRelax_Experiment.csv'.format(CorrFuncLoc2,'Ubiquitin'))

In [ ]:
## Load the GB3 Full Correlation Function Fits:
FitDF_GB3_NVE = pd.read_csv('{}{}/PROD_NVE/FitDF_OptSim_Fix2Exp_tm10ns_FixTM_FixFit_LnT_tbm150ps.csv'.format(CorrFuncLoc2, 'GB3'), index_col=0)
FitDF_GB3_NVT_02ps = pd.read_csv('{}{}/PROD_NVT/CF0-2ps/FitDF_Fix2Exp_Coupling02ps_tm11ns_FixTM_FixFit_LnT_tbm150ps.csv'.format(CorrFuncLoc2,'GB3'), index_col=0)
FitDF_GB3_NVT_2ps = pd.read_csv('{}{}/PROD_NVT/CF2ps/FitDF_Fix2Exp_Coupling2ps_tm16ns_FixTM_FixFit_LnT_tbm150ps.csv'.format(CorrFuncLoc2,'GB3'), index_col=0)
FitDF_GB3_NVT_20ps = pd.read_csv('{}{}/PROD_NVT/CF20ps/FitDF_Fix2Exp_Coupling20ps_tm50ns_FixTM_FixFit_LnT_tbm150ps.csv'.format(CorrFuncLoc2,'GB3'), index_col=0)

In [ ]:
NMRFitDF_GB3_NVE_FixFit_TM10 = calc_NMRFit(FitDF_GB3_NVE, 600, GB3_NMRRelax_Exp)
NMRFitDF_GB3_NVT_CF02ps_FixFit_TM11_FixFit_LnT = calc_NMRFit(FitDF_GB3_NVT_02ps, 600, GB3_NMRRelax_Exp)
NMRFitDF_GB3_NVT_CF2ps_FixTM16_FixFit_lnt= calc_NMRFit(FitDF_GB3_NVT_2ps, 600, GB3_NMRRelax_Exp)
NMRFitDF_GB3_NVT_CF20ps_FixTM50_FixFit_lnt= calc_NMRFit(FitDF_GB3_NVT_20ps, 600, GB3_NMRRelax_Exp)

In [ ]:
OptResCorrDF_GB3_NVE_Avg.iloc[1,:].plot()
NMRFitDF_GB3_NVE_FixFit_TM10[['C_a','C_b']].sum(axis=1).plot()
plt.gca().axhline(np.power(1.02/1.041,6),0,54,color='k')
plt.gca().set_ylim(0,1.0)

In [ ]:
NMRFitDF_GB3_NVE_BestBiExp_TM12 = calc_NMRFit(FitDF_GB3_NVE_Opt_FixTM12_lnt, 600, GB3_NMRRelax_Exp)

In [ ]:
NMRFitDF_GB3_NVE_BestBiExp_TM12['NOE'].plot()

In [ ]:
NMRFitDF_GB3_NVE = calc_NMRFit(FitDF_GB3_NVE, 600, GB3_NMRRelax_Exp)
#NMRFitDF_GB3_NVT_02ps = calc_NMRFit(FitDF_GB3_NVT_02ps_FixTM_Cons_t00, 600, GB3_NMRRelax_Exp)
#NMRFitDF_GB3_NVT_2ps= calc_NMRFit(FitDF_GB3_NVT_2ps_FixTM_Cons_t00, 600, GB3_NMRRelax_Exp)
#NMRFitDF_GB3_NVT_20ps = calc_NMRFit(FitDF_GB3_NVT_20ps_FixTM_Cons_t00, 600, GB3_NMRRelax_Exp)

In [ ]:
NMRFitDF_GB3_NVE_FixFit_TM10.to_csv('{}{}/PROD_NVE/NMRRelax_FitDF_OptSim_Fix2Exp_tm10ns_FixTM_FixFit_LnT_tbm150ps.csv'.format(CorrFuncLoc2, 'GB3'))
NMRFitDF_GB3_NVT_CF02ps_FixFit_TM11_FixFit_LnT.to_csv('{}{}/PROD_NVT/CF0-2ps/NMRRelax_FitDF_Fix2Exp_Coupling02ps_tm11ns_FixTM_FixFit_LnT_tbm150ps.csv'.format(CorrFuncLoc2, 'GB3'))
NMRFitDF_GB3_NVT_CF2ps_FixTM16_FixFit_lnt.to_csv('{}{}/PROD_NVT/CF2ps/NMRRelax_FitDF_Fix2Exp_Coupling2ps_tm16ns_FixTM_FixFit_LnT_tbm150ps.csv'.format(CorrFuncLoc2, 'GB3'))
NMRFitDF_GB3_NVT_CF20ps_FixTM50_FixFit_lnt.to_csv('{}{}/PROD_NVT/CF20ps/NMRRelax_FitDF_Fix2Exp_Coupling20ps_tm50ns_FixTM_FixFit_LnT_tbm150ps.csv'.format(CorrFuncLoc2, 'GB3'))

In [ ]:
NMRFitDF_UBQ_NVE = calc_NMRFit(OptFitDF_UBQ_NVE_FixTM12_FixFit_LnT, 600, UBQ_NMRRelax_Exp)
NMRFitDF_UBQ_NVT_02ps = calc_NMRFit(OptFitDF_UBQ_NVT_CF02ps_FixTM12_FixFit_LnT, 600, UBQ_NMRRelax_Exp)
NMRFitDF_UBQ_NVT_2ps= calc_NMRFit(FitDF_UBQ_NVT_2ps_FixTM16_FixFit_LnT, 600, UBQ_NMRRelax_Exp)
NMRFitDF_UBQ_NVT_20ps = calc_NMRFit(FitDF_UBQ_NVT_20ps_FixTM50_FixFit_LnT, 600, UBQ_NMRRelax_Exp)

In [ ]:
NMRFitDF_UBQ_NVE

In [ ]:
NMRFitDF_UBQ_NVE_BestBiExP_FixTM12 = calc_NMRFit(OptFitDF_UBQ_NVE_FixTM12_LnT, 600, UBQ_NMRRelax_Exp)
NMRFitDF_UBQ_NVE_FixFit_FixTM12 = calc_NMRFit(OptFitDF_UBQ_NVE_FixTM12_FixFit_LnT, 600, UBQ_NMRRelax_Exp)

In [ ]:
NMRFitDF_UBQ_NVE.to_csv('{}{}/PROD_NVE/NMRRelax_FitDF_OptSim_Fix2Exp_FixTM12_FixFit_LnT_tbm150ps.csv'.format(CorrFuncLoc2, 'Ubiquitin'))
NMRFitDF_UBQ_NVT_02ps.to_csv('{}{}/PROD_NVT/CF0-2ps/NMRRelax_Fix2Exp_Coupling02ps_FixTM12_FixFit_LnT_tbm150ps.csv'.format(CorrFuncLoc2,'Ubiquitin'))
NMRFitDF_UBQ_NVT_2ps.to_csv('{}{}/PROD_NVT/CF2ps/NMRRelax_Fix2Exp_Coupling2ps_FixTM16_FixFit_LnT_tbm150ps.csv'.format(CorrFuncLoc2,'Ubiquitin'))
NMRFitDF_UBQ_NVT_20ps.to_csv('{}{}/PROD_NVT/CF20ps/NMRRelax_Fix2Exp_Coupling20ps_FixTM50_FixFit_LnT_tbm150ps.csv'.format(CorrFuncLoc2,'Ubiquitin'))

In [ ]:
np.sqrt(NMRFitDF_GB3_NVE_BestBiExp_TM10['R1_SE'].mean())

In [ ]:
#NMRFitDF_GB3_NVE_BestBiExp_TM10[['Resname','R1']].plot(x='Resname', y='R1', ax=plt.gca(), color='g', label='Sum=1')
NMRFitDF_GB3_NVE_FixFit_TM10[['Resname','R1']].plot(x='Resname', y='R1', color='b', label='A_1=S2', ax=plt.gca())
#NMRFitDF_GB3_NVE_ws2_logt[['Resname','R1']].plot(x='Resname', y='R1', color='r', label='A_1=S2 wLogt', ax=plt.gca())
GB3_NMRRelax_Exp[['RESID','R1']].plot(x='RESID', y='R1', ax=plt.gca(), color='k', label='Exp')


In [ ]:
#NMRFitDF_GB3_NVE_BestBiExp_TM10[['Resname','R2']].plot(x='Resname', y='R2', ax=plt.gca(), color='g', label='Sum=1')
NMRFitDF_GB3_NVE_FixFit_TM10[['Resname','R2']].plot(x='Resname', y='R2', color='b', label='A_1=S2', ax=plt.gca())
#NMRFitDF_GB3_NVE_ws2_logt[['Resname','R2']].plot(x='Resname', y='R2', color='r', label='A_1=S2 wLogt', ax=plt.gca())
GB3_NMRRelax_Exp[['RESID','R2']].plot(x='RESID', y='R2', ax=plt.gca(), color='k', label='Exp')


In [ ]:
#NMRFitDF_GB3_NVE_BestBiExp_TM10[['Resname','NOE']].plot(x='Resname', y='NOE', ax=plt.gca(), color='g', label='Sum=1')
NMRFitDF_GB3_NVE_FixFit_TM10[['Resname','NOE']].plot(x='Resname', y='NOE', color='b', label='A_1=S2', ax=plt.gca())
#NMRFitDF_GB3_NVE_ws2_logt[['Resname','NOE']].plot(x='Resname', y='NOE', color='r', label='A_1=S2 wLogt', ax=plt.gca())
GB3_NMRRelax_Exp[['RESID','NOE']].plot(x='RESID', y='NOE', ax=plt.gca(), color='k', label='Exp')


In [ ]:
UBQ_NMRRelax_Exp['RESID'] = UBQ_NMRRelax_Exp['RESID'].astype(int)
UBQ_NMRRelax_Exp[['RESID','R1']].plot(x='RESID', y='R1')
NMRFitDF_UBQ_NVE['Resname'] = NMRFitDF_UBQ_NVE['Resname'].astype(int)
NMRFitDF_UBQ_NVE[['Resname','R1']].plot(x='Resname', y='R1',ax=plt.gca())

In [ ]:
## Load the GB3 Internal Correlation Function Fits:
MFFits_GB3_NVE = pd.read_csv('{}{}/PROD_NVE/MFFitDF_Fix2Exp_tm11ns.csv'.format(CorrFuncLoc2, 'GB3'), index_col=0)
MFFits_GB3_NVT_02ps = pd.read_csv('{}{}/PROD_NVT/CF0-2ps/MFFitDF_Fix2Exp_Coupling02ps_tm11ns.csv'.format(CorrFuncLoc2,'GB3'))
MFFits_GB3_NVT_2ps = pd.read_csv('{}{}/PROD_NVT/CF2ps/MFFitDF_Fix2Exp_Coupling2ps_tm11ns.csv'.format(CorrFuncLoc2,'GB3'))
MFFits_GB3_NVT_20ps = pd.read_csv('{}{}/PROD_NVT/CF20ps/MFFitDF_Fix2Exp_Coupling20ps_tm50ns.csv'.format(CorrFuncLoc2,'GB3'))

In [ ]:
tau_eff = MFFits_GB3_NVE['tau_a']*FitDF_GB3_NVE['tau_a']/(MFFits_GB3_NVE['tau_a'] + FitDF_GB3_NVE['tau_a'])
MFFits_GB3_NVE['tau_a']

In [ ]:
## Load the Ubiquitin Full Correlation Function Fits
FitDF_UBQ_NVE = pd.read_csv('{}{}/PROD_NVE/FitDF_Fix2Exp_tm11ns.csv'.format(CorrFuncLoc2, 'Ubiquitin'),
                             index_col=0)
FitDF_UBQ_NVT_02ps = pd.read_csv('{}{}/PROD_NVT/CF0-2ps/FitDF_Fix2Exp_Coupling02ps_tm11ns.csv'.format(CorrFuncLoc2,'Ubiquitin'), 
                                 index_col=0)
FitDF_UBQ_NVT_2ps = pd.read_csv('{}{}/PROD_NVT/CF2ps/FitDF_Fix2Exp_Coupling2ps_tm11ns.csv'.format(CorrFuncLoc2,'Ubiquitin'),
                                 index_col=0)
FitDF_UBQ_NVT_20ps = pd.read_csv('{}{}/PROD_NVT/CF20ps/FitDF_Fit2Exp_Coupling20ps_tm50ns.csv'.format(CorrFuncLoc2,'Ubiquitin'),
                                 index_col=0)

In [ ]:
## Load the Ubiquiting Internal Correlation Function Fits:
MFFits_UBQ_NVE = pd.read_csv('{}{}/PROD_NVE/MFFitDF_Fix2Exp_tm11ns.csv'.format(CorrFuncLoc2, 'Ubiquitin'))
MFFits_UBQ_NVT_02ps = pd.read_csv('{}{}/PROD_NVT/CF0-2ps/MFFitDF_Fix2Exp_Coupling02ps_tm11ns.csv'.format(CorrFuncLoc2,'Ubiquitin'))
MFFits_UBQ_NVT_2ps = pd.read_csv('{}{}/PROD_NVT/CF2ps/MFFitDF_Fix2Exp_Coupling2ps_tm11ns.csv'.format(CorrFuncLoc2,'Ubiquitin'))
MFFits_UBQ_NVT_20ps = pd.read_csv('{}{}/PROD_NVT/CF20ps/MFFitDF_Fix2Exp_Coupling20ps_tm50ns.csv'.format(CorrFuncLoc2,'Ubiquitin'))

In [ ]:
figUBQ2ps = plt.figure(39181, figsize=(10,6))
axubq2ps = figUBQ2ps.add_subplot(111)
FitDF_UBQ_NVE['tau_a'].plot(style=['-d'], color='b', label=r'$\mathbf{NVE}$', ax=axubq2ps)
FitDF_UBQ_NVT_2ps['tau_a'].plot(style=['-o'],color='g', label=r'$\mathbf{NVT:\gamma = 2 \ \ ps^{-1}}$', ax=axubq2ps)

axubq2ps.set_xlabel('Residue Number', weight='bold', fontsize=16)
axubq2ps.set_ylabel(r'$\mathbf{Correlation time: \tau_{a} \ \ (ns) }$', fontsize=16)
axubq2ps.tick_params(labelsize=15)
axubq2ps.legend(loc='best', frameon=False, prop={'size':15})
#figUBQ2ps.savefig('{}Ubiquitin/Analysis/CompareNVE_NVTCp2ps_LongCorrelationTimes_2ExpFit.png'.format(CorrFuncLoc2), dpi=600, bbox_inches='tight')

In [ ]:
figUBQ2pstb = plt.figure(39182, figsize=(10,6))
axubq2pstb = figUBQ2pstb.add_subplot(111)
FitDF_UBQ_NVE['tau_b'].plot(style=['-d'], color='b', logy=True, label=r'$\mathbf{NVE}$', ax=axubq2pstb)
FitDF_UBQ_NVT_2ps['tau_b'].plot(style=['-o'],color='g', logy=True, label=r'$\mathbf{NVT:\gamma = 2 \ \ ps^{-1}}$', ax=axubq2pstb)

axubq2pstb.set_xlabel('Residue Number', weight='bold', fontsize=16)
axubq2pstb.set_ylabel(r'$\mathbf{Correlation time: \tau_{b} \ \ (ns) }$', fontsize=16)
axubq2pstb.tick_params(labelsize=15)
axubq2pstb.legend(loc='best', frameon=False, prop={'size':15})
figUBQ2pstb.savefig('{}Ubiquitin/Analysis/CompareNVE_NVTCp2ps_ShortCorrelationTimes_2ExpFit.png'.format(CorrFuncLoc2), dpi=600, bbox_inches='tight')

In [ ]:
figGB32ps = plt.figure(30193, figsize=(10,6))
axgb32ps = figGB32ps.add_subplot(111)

FitDF_GB3_NVE[['Resname','tau_a']].plot(x = 'Resname', y = 'tau_a', style=['-d'], color='b',
                                         ax=axgb32ps, label=r'$\mathbf{NVE}$')
FitDF_GB3_NVT_2ps[['Resname','tau_a']].plot(x = 'Resname', y = 'tau_a', style=['-o'], color='g',
                                             ax=axgb32ps, label=r'$\mathbf{NVT:\gamma = 2 \ \ ps^{-1}}$')
axgb32ps.set_xlabel('Residue Number', weight='bold', fontsize=15)
axgb32ps.set_ylabel(r'$\mathbf{Correlation time: \tau_{a} \ \ (ns) }$', fontsize=15)
axgb32ps.tick_params(labelsize=14)
axgb32ps.legend(loc='best', frameon=False, prop={'size':15} )
figGB32ps.savefig('{}GB3/Analysis/CompareNVE_NVTCp2ps_LongCorrelationTimes_2ExpFit.png'.format(CorrFuncLoc2), dpi=600, bbox_inches='tight')

In [ ]:
figGB32pstb = plt.figure(30193, figsize=(10,6))
axgb32pstb = figGB32pstb.add_subplot(111)

FitDF_GB3_NVE[['Resname','tau_b']].plot(x = 'Resname', y = 'tau_b', style=['-d'], color='b',
                                        logy=True, ax=axgb32pstb, label=r'$\mathbf{NVE}$')
FitDF_GB3_NVT_2ps[['Resname','tau_b']].plot(x = 'Resname', y = 'tau_b', style=['-o'], color='g',
                                            logy=True, ax=axgb32pstb, label=r'$\mathbf{NVT:\gamma = 2 \ \ ps^{-1}}$')

axgb32pstb.set_xlabel('Residue Number', weight='bold', fontsize=15)
axgb32pstb.set_ylabel(r'$\mathbf{Correlation time: \tau_{b} \ \ (ns) }$', fontsize=15)
axgb32pstb.tick_params(labelsize=14)
axgb32pstb.legend(loc='best', frameon=False, prop={'size':15} )
figGB32pstb.savefig('{}GB3/Analysis/CompareNVE_NVTCp2ps_ShortCorrelationTimes_2ExpFit.png'.format(CorrFuncLoc2), dpi=600, bbox_inches='tight')

In [ ]:
## Plotting Models and Correlation Functions
time_series = MFCorrDF_GB3_NVE.index.values
parameters = MFFits_GB3_NVE[['C_a','tau_a','C_b']].values
model_res2 = func_CorrInt2(time_series, *parameters[0])
plt.semilogx(time_series, model_res2)
plt.semilogx(time_series, MFCorrDF_GB3_NVE['2'].values)

In [ ]:

parameters = FitDF_GB3_NVE[['C_a','tau_a','C_b','tau_b']].values
model_res2 = func_exp_decay4(time_series, *parameters[0])
#model_res2_wUltra =  

plt.semilogx(time_series, model_res2)
plt.semilogx(time_series, ResCorrDF_GB3_NVE['2'].values)

In [ ]:
func

In [ ]:
import statsmodels.api as sm

In [ ]:
def NVT2NVE_Regression(nvedata, nvtdata):
    OLS_NVT2NVE = sm.OLS(nvtdata, nvedata)
    ols_fit = OLS_NVT2NVE.fit()
    return ols_fit.params[0], ols_fit.rsquared

In [ ]:
GB3_NVT_Fits = {'0.2':FitDF_GB3_NVT_02ps, '2':FitDF_GB3_NVT_2ps,'20':FitDF_GB3_NVT_20ps}
GB3_NVT_Regression = {}
for fit_key in GB3_NVT_Fits.keys():
    
    ta_fit = NVT2NVE_Regression(FitDF_GB3_NVE['tau_a'].values, GB3_NVT_Fits[fit_key]['tau_a'])
    tb_fit = NVT2NVE_Regression(FitDF_GB3_NVE['tau_b'].values, GB3_NVT_Fits[fit_key]['tau_b'])
    GB3_NVT_Regression[fit_key] = {'tau_a':ta_fit, 'tau_b': tb_fit}
    
    

In [ ]:
UBQ_NVT_Fits = {'0.2':FitDF_UBQ_NVT_02ps, '2':FitDF_UBQ_NVT_2ps,'20':FitDF_UBQ_NVT_20ps}
UBQ_NVT_Regression = {}
for fit_key in UBQ_NVT_Fits.keys():
    
    ta_fit = NVT2NVE_Regression(FitDF_UBQ_NVE['tau_a'].values, UBQ_NVT_Fits[fit_key]['tau_a'])
    tb_fit = NVT2NVE_Regression(FitDF_UBQ_NVE['tau_b'].values, UBQ_NVT_Fits[fit_key]['tau_b'])
    UBQ_NVT_Regression[fit_key] = {'tau_a':ta_fit, 'tau_b': tb_fit}
    

In [ ]:
GB3_NVT_Regression

In [ ]:
UBQ_NVT_Regression

In [ ]:

plt.scatter(x=FitDF_GB3_NVE['tau_a'].values, y=FitDF_GB3_NVT_02ps['tau_a'].values,
            c='orange', marker='s', label=r'$\mathbf{NVT:\gamma = 0.2 \ \ ps^{-1}}$', edgecolor='k')
plt.scatter(x=FitDF_GB3_NVE['tau_a'].values, y=FitDF_GB3_NVT_2ps['tau_a'].values,
            c='green', marker='o' ,  label = r'$\mathbf{NVT:\gamma = 2 \ \ ps^{-1}}$',edgecolor='k')
plt.scatter(x=FitDF_GB3_NVE['tau_a'].values, y=FitDF_GB3_NVT_20ps['tau_a'].values,
            c='red', marker='d', label = r'$\mathbf{NVT:\gamma = 20 \ \ ps^{-1}}$', edgecolor='k')
plt.plot(np.arange(1.0, 6.1,0.1), np.arange(1.0, 6.1,0.1), color='k', linewidth=2)
plt.legend(frameon=False)
plt.yscale('log')
#plt.xscale('log')
plt.gca().set_ylabel('NVT Correlation times (ns)', weight='bold', fontsize=15)
plt.gca().set_xlabel('NVE Correlation times (ns)', weight='bold', fontsize=15)
plt.gca().tick_params(labelsize=15)

plt.savefig('{}GB3/Analysis/CompareLongCorrelationTime_2ExpFit_ScatterPlot.png'.format(CorrFuncLoc2), bbox_inches='tight', dpi=600)

In [ ]:

plt.scatter(x=FitDF_UBQ_NVE['tau_a'].values, y=FitDF_UBQ_NVT_02ps['tau_a'].values,
            c='orange', marker='s', label=r'$\mathbf{NVT:\gamma = 0.2 \ \ ps^{-1}}$', edgecolor='k')
plt.scatter(x=FitDF_UBQ_NVE['tau_a'].values, y=FitDF_UBQ_NVT_2ps['tau_a'].values,
            c='green', marker='o' ,  label = r'$\mathbf{NVT:\gamma = 2 \ \ ps^{-1}}$', edgecolor='k')
plt.scatter(x=FitDF_UBQ_NVE['tau_a'].values, y=FitDF_UBQ_NVT_20ps['tau_a'].values,
            c='red', marker='d', label = r'$\mathbf{NVT:\gamma = 20 \ \ ps^{-1}}$', edgecolor='k')
plt.plot(np.arange(1.0, 6.1,0.1), np.arange(1.0, 6.1,0.1), color='k')
plt.legend(frameon=False)
plt.yscale('log')
plt.gca().set_ylabel('NVT Correlation times (ns)', weight='bold', fontsize=15)
plt.gca().set_xlabel('NVE Correlation times (ns)', weight='bold', fontsize=15)
plt.gca().tick_params(labelsize=15)

plt.savefig('{}Ubiquitin/Analysis/CompareLongCorrelationTime_2ExpFit_ScatterPlot.png'.format(CorrFuncLoc2), bbox_inches='tight', dpi=600)

In [ ]:
FitDF_GB3_NVE[['tau_b','tau_g']].values.flatten()

In [ ]:
CorrTauFig, axCorrT = plt.subplots(1,1, figsize=(8,6))
axCorrT.scatter(x=FitDF_GB3_NVE[['tau_b','tau_g']].values.flatten(), y=FitDF_GB3_NVT_02ps[['tau_b','tau_g']].values.flatten(),
            c='orange', marker='s', label=r'$\mathbf{NVT:\gamma = 0.2 \ \ ps^{-1}}$', edgecolor='k')
axCorrT.scatter(x=FitDF_GB3_NVE[['tau_b','tau_g']].values.flatten(), y=FitDF_GB3_NVT_2ps[['tau_b','tau_g']].values.flatten(),
            c='green', marker='o' ,  label = r'$\mathbf{NVT:\gamma = 2 \ \ ps^{-1}}$', edgecolor='k')
axCorrT.scatter(x=FitDF_GB3_NVE[['tau_b','tau_g']].values.flatten(), y=FitDF_GB3_NVT_20ps[['tau_b','tau_g']].values.flatten(),
            c='red', marker='P', label = r'$\mathbf{NVT:\gamma = 20 \ \ ps^{-1}}$', edgecolor='k')
plt.plot(np.arange(0.0001, 3.01,0.01), np.arange(0.0001, 3.01,0.01), color='k')
axCorrT.set_xlim(0.0001,10.0)
axCorrT.set_ylim(0.0001,10.0)
axCorrT.legend(frameon=False)
plt.yscale('log')
plt.xscale('log')
axCorrT.set_ylabel('NVT Correlation times (ns)', weight='bold', fontsize=15)
axCorrT.set_xlabel('NVE Correlation times (ns)', weight='bold', fontsize=15)
axCorrT.tick_params(labelsize=15)

CorrTauFig.savefig('{}GB3/Analysis/CompareShortCorrelationTime_2ExpFit_ScatterPlot.png'.format(CorrFuncLoc2), bbox_inches='tight', dpi=600)

In [ ]:
CorrTauFig, axCorrT = plt.subplots(1,1, figsize=(8,6))
axCorrT.scatter(x=FitDF_UBQ_NVE[['tau_b','tau_g']].values.flatten(), y=FitDF_UBQ_NVT_02ps[['tau_b','tau_g']].values.flatten(),
            c='orange', marker='s', label=r'$\mathbf{NVT:\gamma = 0.2 \ \ ps^{-1}}$', edgecolor='k')
axCorrT.scatter(x=FitDF_UBQ_NVE[['tau_b','tau_g']].values.flatten(), y=FitDF_UBQ_NVT_2ps[['tau_b','tau_g']].values.flatten(),
            c='green', marker='o' ,  label = r'$\mathbf{NVT:\gamma = 2 \ \ ps^{-1}}$', edgecolor='k')
axCorrT.scatter(x=FitDF_UBQ_NVE[['tau_b','tau_g']].values.flatten(), y=FitDF_UBQ_NVT_20ps[['tau_b','tau_g']].values.flatten(),
            c='red', marker='P', label = r'$\mathbf{NVT:\gamma = 20 \ \ ps^{-1}}$', edgecolor='k')
plt.plot(np.arange(0.0001, 3.01,0.01), np.arange(0.0001, 3.01,0.01), color='k')
axCorrT.set_xlim(0.0001,10.0)
axCorrT.set_ylim(0.0001,10.0)
axCorrT.legend(frameon=False)
plt.yscale('log')
plt.xscale('log')
axCorrT.set_ylabel('NVT Correlation times (ns)', weight='bold', fontsize=15)
axCorrT.set_xlabel('NVE Correlation times (ns)', weight='bold', fontsize=15)
axCorrT.tick_params(labelsize=15)

CorrTauFig.savefig('{}Ubiquitin/Analysis/CompareShortCorrelationTime_2ExpFit_ScatterPlot.png'.format(CorrFuncLoc2), bbox_inches='tight', dpi=600)

In [ ]:
## Loading the Order Parameter calculation directly from the MS 
S2DF = pd.read_csv('{}/OrderParameter_Direct.csv'.format(LocDF_NVT.loc['Run1',('GB3', 'CF0-2ps')]), names=['RESN',r'$S^{2}$'])

In [ ]:
figS2GB3 = plt.figure(3280, figsize=(10,6))
axS2gb3 = figS2GB3.add_subplot(111)

GB3_S2_NVE.mean(axis=1).plot(style='-d', color='b', markeredgecolor='k', ax=axS2gb3,
                            label = r'$\mathbf{NVE}$',
                             linewidth=2.5, markersize=6.5, alpha=0.75)
GB3_S2_CF02ps.mean(axis=1).plot(style='-s', color='orange', markeredgecolor='k', ax=axS2gb3,
                               label = r'$\mathbf{NVT: \gamma = 0.2 \ \ ps^{-1}}$',
                                linewidth=2.5, markersize=6.5, alpha=0.75)
GB3_S2_CF2ps.mean(axis=1).plot(style='-o',  color='green', markeredgecolor='k', ax=axS2gb3,
                              label = r'$\mathbf{NVT: \gamma = 2 \ \ ps^{-1}}$',
                               linewidth=2.5, markersize=6.5, alpha=0.75)
GB3_S2_CF20ps.mean(axis=1).plot(style='-P', color='red', markeredgecolor='k', ax=axS2gb3,
                               label = r'$\mathbf{NVT: \gamma = 20 \ \ ps^{-1}}$',
                                linewidth=2.5, markersize=6.5, alpha=0.75)

axS2gb3.set_ylabel(r'$\mathbf{S^2}$', fontsize=18)
axS2gb3.set_xlabel(r'Residue Number', fontsize=18, weight='bold')
axS2gb3.tick_params(labelsize=17)
axS2gb3.set_xticks(np.arange(0,60,5), minor=True)
axS2gb3.set_yticks(np.arange(0,1.1,0.1), minor=True)
axS2gb3.legend(frameon=False, loc=8, prop={'size':16})


In [ ]:
UBQ_S2_CF2ps_restack = UBQ_S2_CF2ps.unstack().reset_index().rename(columns={'level_0':'Runs',0:r'$S^2$'})
UBQ_S2_CF2ps_restack['Gamma'] = r'$NVT: \gamma = \ \ 2 \ \ ps^{-1}$'
UBQ_S2_CF02ps_restack = UBQ_S2_CF02ps.unstack().reset_index().rename(columns={'level_0':'Runs',0:r'$S^2$'})
UBQ_S2_CF02ps_restack['Gamma'] = r'$NVT: \gamma = \ \ 0.2 \ \ ps^{-1}$'
UBQ_S2_CF20ps_restack = UBQ_S2_CF20ps.unstack().reset_index().rename(columns={'level_0':'Runs',0:r'$S^2$'})
UBQ_S2_CF20ps_restack['Gamma'] = r'$NVT: \gamma = \ \ 20 \ \ ps^{-1}$'

In [ ]:
UBQ_Order_parameter = pd.concat([UBQ_S2_CF02ps_restack, UBQ_S2_CF2ps_restack, UBQ_S2_CF20ps_restack])

In [ ]:
figS2UBQ = plt.figure(3281, figsize=(10,6))
axS2ubq = figS2UBQ.add_subplot(111)

UBQ_S2_NVE.mean(axis=1).plot(style='-x', color='blue', markeredgecolor='k',
                                label = r'$\mathbf{NVE}$',
                                ax = axS2ubq, linewidth=2.5, markersize=6.5, alpha=0.75)

UBQ_S2_CF02ps.mean(axis=1).plot(style='-s', color='orange', markeredgecolor='k',
                                label = r'$\mathbf{NVT: \gamma = 0.2 \ \ ps^{-1}}$',
                                ax = axS2ubq, linewidth=2.5, markersize=6.5, alpha=0.75)

UBQ_S2_CF2ps.mean(axis=1).plot(style='-o',  color='green', markeredgecolor='k',
                               label = r'$\mathbf{NVT: \gamma = 2 \ \ ps^{-1}}$',
                               ax = axS2ubq, linewidth=2.5, markersize=6.5, alpha=0.75)

UBQ_S2_CF20ps.mean(axis=1).plot(style='-d', color='red', markeredgecolor='k',
                                label = r'$\mathbf{NVT: \gamma = 20 \ \ ps^{-1}}$', 
                                ax = axS2ubq, linewidth=2.5, markersize=6.5, alpha=0.75)


axS2ubq.set_ylabel(r'$\mathbf{S^2}$', fontsize=18)
axS2ubq.set_xlabel(r'Residue Number', fontsize=18, weight='bold')
axS2ubq.tick_params(labelsize=17)
axS2ubq.set_xticks(np.arange(0,80,5), minor=True)
axS2ubq.set_yticks(np.arange(0,1.1,0.1), minor=True)
axS2ubq.legend(frameon=False, loc=8, prop={'size':16})

In [ ]:
tm_loc = np.where(x<11)[0][-1] + 1
x = RESCorrDF_NVE[5].index[:tm_loc]

y = RESCorrDF_NVE[5].values[:tm_loc]
dy = RESCorrDF_NVE_STD[5].values[:tm_loc]
chi, params, covarMat, ymodel = do_Expstyle_fit2(2, x, y, dy=dy)

In [ ]:
corr_time_index = np.arange(0,50001,1)*5/1e3
RESCorrDF_NVE = pd.DataFrame(index= corr_time_index, columns=GB3atom_df.iloc[1:]['RESID'])
RESCorrDF_NVE_STD = RESCorrDF_NVE.copy()
for rcol in RESCorrDF_NVE.columns:
    rescfdf = pd.DataFrame(index=np.arange(0,50001,1)*5/1e3, columns=RUNS)
    for R in RUNS:
        NVECorr28 = pd.read_csv('{}/NHCorrFunc/NH{}CorrelationFunctions.dat'.format(LocDF.loc[R,('GB3','NVE')], rcol), delim_whitespace=True)
        rescfdf.loc[:,R] = NVECorr28['<P2>'].values
    RESCorrDF_NVE[rcol] = rescfdf.mean(axis=1)
    RESCorrDF_NVE_STD[rcol] = rescfdf.std(axis=1)

In [ ]:
corr_time_index = np.arange(0,50001,1)*10/1e3
RESCorrDF_NVT = pd.DataFrame(index= corr_time_index, columns=GB3atom_df.iloc[1:]['RESID'])
RESCorrDF_NVT_STD = RESCorrDF_NVT.copy()
for rcol in RESCorrDF_NVT.columns:
    rescfdf_NVT = pd.DataFrame(index= corr_time_index, columns=RUNS)
    
    for R in RUNS:
        NVTCorr28 = pd.read_csv('{}/NHCorrFunc/NH{}CorrelationFunctions.dat'.format(LocDF.loc[R,('GB3','NVT')], rcol), delim_whitespace=True)
        rescfdf_NVT.loc[:,R] = NVTCorr28['<P2>'].values
    RESCorrDF_NVT[rcol] = rescfdf_NVT.mean(axis=1)
    RESCorrDF_NVT_STD[rcol] = rescfdf_NVT.std(axis=1)

In [ ]:
figcf, axescf = plt.subplots(5,11, sharex=True, sharey=True, figsize=(20,18), num=39581)
for axcf, resn in (zip(axescf.flatten(), RESCorrDF_NVT.columns)):
    RESCorrDF_NVE[resn].plot(ax=axcf, logx=True, label='NVE', legend=True)
    RESCorrDF_NVT[resn].plot(ax=axcf, logx=True, label='NVT', legend=True)
    axcf.text(0.05,0.05,GB3atom_df.iloc[resn-1]['RESNAME'])

## Comparing the Different types of fits

In [ ]:
## Load the GB3 Full Correlation Function Fits:
FitDF_GB3_NVE_Lin_FN = pd.read_csv('{}{}/PROD_NVE/FitDF_Fix2Exp_tm11ns_FirstNegStop_Linear.csv'.format(CorrFuncLoc2, 'GB3'))
#FitDF_GB3_NVT_02ps_Lin_FN = pd.read_csv('{}{}/PROD_NVT/CF0-2ps/FitDF_Fix2Exp_Coupling02ps_tm11ns_FirstNegStop_Linear.csv'.format(CorrFuncLoc2,'GB3'))
FitDF_GB3_NVT_2ps_Lin_FN = pd.read_csv('{}{}/PROD_NVT/CF2ps/FitDF_Fix2Exp_Coupling2ps_tm11ns_FirstNegStop_Linear.csv'.format(CorrFuncLoc2,'GB3'))
#FitDF_GB3_NVT_20ps_Lin_FN = pd.read_csv('{}{}/PROD_NVT/CF20ps/FitDF_Fix2Exp_Coupling20ps_tm50ns_FirstNegStop_Linear.csv'.format(CorrFuncLoc2,'GB3'))

In [ ]:
## Load the GB3 Full Correlation Function Fits Soft Loss:
FitDF_GB3_NVE_SL_TM = pd.read_csv('{}{}/PROD_NVE/FitDF_Fix2Exp_tm11ns.csv'.format(CorrFuncLoc2, 'GB3'))
#FitDF_GB3_NVT_02ps_SL_TM = pd.read_csv('{}{}/PROD_NVT/CF0-2ps/FitDF_Fix2Exp_Coupling02ps_tm11ns.csv'.format(CorrFuncLoc2,'GB3'))
FitDF_GB3_NVT_2ps_SL_TM = pd.read_csv('{}{}/PROD_NVT/CF2ps/FitDF_Fix2Exp_Coupling2ps_tm11ns.csv'.format(CorrFuncLoc2,'GB3'))
#FitDF_GB3_NVT_20ps_SL_TM = pd.read_csv('{}{}/PROD_NVT/CF20ps/FitDF_Fix2Exp_Coupling20ps_tm50ns.csv'.format(CorrFuncLoc2,'GB3'))

In [ ]:
## Load the GB3 Full Correlation Function Fits:
FitDF_GB3_NVE_SL_FN = pd.read_csv('{}{}/PROD_NVE/FitDF_Fix2Exp_tm11ns_FirstNegStop_SoftL.csv'.format(CorrFuncLoc2, 'GB3'))
#FitDF_GB3_NVT_02ps_Lin_FN = pd.read_csv('{}{}/PROD_NVT/CF0-2ps/FitDF_Fix2Exp_Coupling02ps_tm11ns_FirstNegStop_Linear.csv'.format(CorrFuncLoc2,'GB3'))
FitDF_GB3_NVT_2ps_SL_FN = pd.read_csv('{}{}/PROD_NVT/CF2ps/FitDF_Fix2Exp_Coupling2ps_tm11ns_FirstNegStop_SoftL.csv'.format(CorrFuncLoc2,'GB3'))
#FitDF_GB3_NVT_20ps_Lin_FN = pd.read_csv('{}{}/PROD_NVT/CF20ps/FitDF_Fix2Exp_Coupling20ps_tm50ns_FirstNegStop_Linear.csv'.format(CorrFuncLoc2,'GB3'))

In [ ]:
## Load the GB3 Full Correlation Function Fits:
FitDF_GB3_NVE_Lin_TM = pd.read_csv('{}{}/PROD_NVE/FitDF_Fix2Exp_tm11ns_FixTM_Linear.csv'.format(CorrFuncLoc2, 'GB3'))
#FitDF_GB3_NVT_02ps_Lin_FN = pd.read_csv('{}{}/PROD_NVT/CF0-2ps/FitDF_Fix2Exp_Coupling02ps_tm11ns_FirstNegStop_Linear.csv'.format(CorrFuncLoc2,'GB3'))
FitDF_GB3_NVT_2ps_Lin_TM = pd.read_csv('{}{}/PROD_NVT/CF2ps/FitDF_Fix2Exp_Coupling2ps_tm11ns_FixTM_Linear.csv'.format(CorrFuncLoc2,'GB3'))
#FitDF_GB3_NVT_20ps_Lin_FN = pd.read_csv('{}{}/PROD_NVT/CF20ps/FitDF_Fix2Exp_Coupling20ps_tm50ns_FirstNegStop_Linear.csv'.format(CorrFuncLoc2,'GB3'))

In [ ]:
plt.rcParams['axes.titlepad'] = 0.0
figGB3CF_NVE, axgb3cf_nve = plt.subplots(2,2, sharex=True, figsize=(12,10), num=32131)
figGB3CF_NVE.subplots_adjust(hspace=0.045, wspace=0.1)

FitDF_GB3_NVE_SL_TM.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='-', color='b', marker='o',
                             ax=axgb3cf_nve[0][0], label='Fixed_SoftLoss')
FitDF_GB3_NVE_Lin_TM.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='-', color='r', marker='x',
                             ax=axgb3cf_nve[0][0], label='Fixed_Linear')

FitDF_GB3_NVE_SL_FN.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='--', color='b', marker='o',
                             ax=axgb3cf_nve[0][0], label='FirstNeg_SoftLoss')
FitDF_GB3_NVE_Lin_FN.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='--', color='r', marker='x',
                             ax=axgb3cf_nve[0][0], label='FirstNeg_Linear')



axgb3cf_nve[0][0].set_ylim(0,1.0)
axgb3cf_nve[0][0].set_title('C_a',weight='bold',fontsize=12, pad=-14)

FitDF_GB3_NVE_SL_TM.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='-', color='b', marker='o',
                             ax=axgb3cf_nve[0][1], label='Fixed_SoftLoss')
FitDF_GB3_NVE_Lin_TM.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='-', color='r', marker='x',
                             ax=axgb3cf_nve[0][1], label='Fixed_Linear')

FitDF_GB3_NVE_SL_FN.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='--', color='b', marker='o',
                             ax=axgb3cf_nve[0][1], label='FirstNeg_SoftLoss')
FitDF_GB3_NVE_Lin_FN.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='--', color='r', marker='x',
                             ax=axgb3cf_nve[0][1], label='FirstNeg_Linear')

axgb3cf_nve[0][1].set_ylim(0,1.0)
axgb3cf_nve[0][1].set_title('C_b', weight='bold', fontsize=12, pad=-14)

FitDF_GB3_NVE_SL_TM.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='-', color='b', marker='o',
                             ax=axgb3cf_nve[1][0], label='Fixed_SoftLoss')
FitDF_GB3_NVE_Lin_TM.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='-', color='r', marker='x',
                             ax=axgb3cf_nve[1][0], label='Fixed_Linear')

FitDF_GB3_NVE_SL_FN.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='--', color='b', marker='o',
                             ax=axgb3cf_nve[1][0], label='FirstNeg_SoftLoss')
FitDF_GB3_NVE_Lin_FN.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='--', color='r', marker='x',
                             ax=axgb3cf_nve[1][0], label='FirstNeg_Linear')

axgb3cf_nve[1][0].set_ylim(2.0, 4.5)
axgb3cf_nve[1][0].set_title('tau_a',weight='bold',fontsize=12, y=0.95, pad=-0.0)

FitDF_GB3_NVE_SL_TM.plot(x='Resname', y='tau_b', yerr='tau_b_err', linestyle='-', color='b', marker='o',
                             ax=axgb3cf_nve[1][1], label='Fixed_SoftLoss')
FitDF_GB3_NVE_Lin_TM.plot(x='Resname', y='tau_b', yerr='tau_b_err', linestyle='-', color='r', marker='x',
                             ax=axgb3cf_nve[1][1], label='Fixed_Linear')

FitDF_GB3_NVE_SL_FN.plot(x='Resname', y='tau_b', yerr='tau_b_err', logy=True, linestyle='--', color='b', marker='o',
                             ax=axgb3cf_nve[1][1], label='FirstNeg_SoftLoss')
FitDF_GB3_NVE_Lin_FN.plot(x='Resname', y='tau_b', yerr='tau_b_err',logy=True, linestyle='--', color='r', marker='x',
                             ax=axgb3cf_nve[1][1], label='FirstNeg_Linear')

axgb3cf_nve[1][1].set_title('tau_b', weight='bold', fontsize=12,  y=0.95, pad=0.0)

figGB3CF_NVE.savefig('D:/Research/LangevinDynamics_RotationalDiffusion/GB3/Analysis/CompareLossFunction_NVE_Amplitudes_Relaxationtimes_DiffFixedCutOffs.png',
               dpi=600,bbox_inches='tight')

In [ ]:
plt.rcParams['axes.titlepad'] = 0.0
figGB3CF_NVT, axgb3cf_nvt = plt.subplots(2,2, sharex=True, figsize=(12,10), num=32131)
figGB3CF_NVT.subplots_adjust(hspace=0.045, wspace=0.1)

FitDF_GB3_NVT_2ps_SL_TM.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='-', color='b', marker='o',
                             ax=axgb3cf_nvt[0][0], label='Fixed_SoftLoss')
FitDF_GB3_NVT_2ps_Lin_TM.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='-', color='r', marker='x',
                             ax=axgb3cf_nvt[0][0], label='Fixed_Linear')

FitDF_GB3_NVT_2ps_SL_FN.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='--', color='b', marker='o',
                             ax=axgb3cf_nvt[0][0], label='FirstNeg_SoftLoss')
FitDF_GB3_NVT_2ps_Lin_FN.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='--', color='r', marker='x',
                             ax=axgb3cf_nvt[0][0], label='FirstNeg_Linear')



axgb3cf_nvt[0][0].set_ylim(0,1.0)
axgb3cf_nvt[0][0].set_title('C_a',weight='bold',fontsize=12, pad=-14)

FitDF_GB3_NVT_2ps_SL_TM.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='-', color='b', marker='o',
                             ax=axgb3cf_nvt[0][1], label='Fixed_SoftLoss')
FitDF_GB3_NVT_2ps_Lin_TM.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='-', color='r', marker='x',
                             ax=axgb3cf_nvt[0][1], label='Fixed_Linear')

FitDF_GB3_NVT_2ps_SL_FN.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='--', color='b', marker='o',
                             ax=axgb3cf_nvt[0][1], label='FirstNeg_SoftLoss')
FitDF_GB3_NVT_2ps_Lin_FN.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='--', color='r', marker='x',
                             ax=axgb3cf_nvt[0][1], label='FirstNeg_Linear')

axgb3cf_nvt[0][1].set_ylim(0,1.0)
axgb3cf_nvt[0][1].set_title('C_b', weight='bold', fontsize=12, pad=-14)

FitDF_GB3_NVT_2ps_SL_TM.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='-', color='b', marker='o',
                             ax=axgb3cf_nvt[1][0], label='Fixed_SoftLoss')
FitDF_GB3_NVT_2ps_Lin_TM.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='-', color='r', marker='x',
                             ax=axgb3cf_nvt[1][0], label='Fixed_Linear')

FitDF_GB3_NVT_2ps_SL_FN.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='--', color='b', marker='o',
                             ax=axgb3cf_nvt[1][0], label='FirstNeg_SoftLoss')
FitDF_GB3_NVT_2ps_Lin_FN.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='--', color='r', marker='x',
                             ax=axgb3cf_nvt[1][0], label='FirstNeg_Linear')

axgb3cf_nvt[1][0].set_ylim(4.5, 8.5)
axgb3cf_nvt[1][0].set_title('tau_a',weight='bold',fontsize=12, y=0.95, pad=-0.0)

FitDF_GB3_NVT_2ps_SL_TM.plot(x='Resname', y='tau_b', yerr='tau_b_err', linestyle='-', color='b', marker='o',
                             ax=axgb3cf_nvt[1][1], label='Fixed_SoftLoss')
FitDF_GB3_NVT_2ps_Lin_TM.plot(x='Resname', y='tau_b', yerr='tau_b_err', linestyle='-', color='r', marker='x',
                             ax=axgb3cf_nvt[1][1], label='Fixed_Linear')

FitDF_GB3_NVT_2ps_SL_FN.plot(x='Resname', y='tau_b', yerr='tau_b_err', logy=True, linestyle='--', color='b', marker='o',
                             ax=axgb3cf_nvt[1][1], label='FirstNeg_SoftLoss')
FitDF_GB3_NVT_2ps_Lin_FN.plot(x='Resname', y='tau_b', yerr='tau_b_err',logy=True, linestyle='--', color='r', marker='x',
                             ax=axgb3cf_nvt[1][1], label='FirstNeg_Linear')

axgb3cf_nvt[1][1].set_title('tau_b', weight='bold', fontsize=12,  y=0.95, pad=0.0)

figGB3CF_NVT.savefig('D:/Research/LangevinDynamics_RotationalDiffusion/GB3/Analysis/CompareLossFunction_NVT_Amplitudes_Relaxationtimes_DiffFixedCutOffs.png',
               dpi=600,bbox_inches='tight')

In [ ]:
FitDF_NVE[['tau_a','tau_b','tau_g','tau_d']]

In [ ]:
tm_loc = np.where(x<11)[0][-1] + 1
x = RESCorrDF_NVE[5].index[:tm_loc]

y = RESCorrDF_NVE[5].values[:tm_loc]
dy = RESCorrDF_NVE_STD[5].values[:tm_loc]
chi, params, covarMat, ymodel = do_Expstyle_fit2(2, x, y, dy=dy)

In [ ]:
params

In [ ]:
pd.Series(index=x,data=ymodel).plot(logx=True)
RESCorrDF_NVE[5].plot(logx=True, label='NVE', legend=True)

In [ ]:
RESCorrDF_NVE[14]

In [ ]:
figcf1, axcf1 = plt.subplots(1,1,figsize=(8,6))
ResCorrDF_NVE[14].plot(ax=axcf1, logx=True, label='NVE', legend=True)
ResCorrDF_NVT[14].plot(ax=axcf1, logx=True, label='NVT', legend=True)

In [ ]:
rescfdf = pd.DataFrame(index=np.arange(0,50001,1)*5/1e3, columns=RUNS)
for R in RUNS:
    NVECorr28 = pd.read_csv('{}/MFCorrFunc/MF46CorrelationFunctions.dat'.format(LocDF.loc[R,('Ubiquitin','NVE')]), delim_whitespace=True)
    rescfdf.loc[:,R] = NVECorr28['<P2>'].values

In [ ]:
rescfdf.mean(axis=1).iloc[1:].plot(logx=True)

In [ ]:
rescfdf_NVT = pd.DataFrame(index=np.arange(0,50001,1)*10/1e3, columns=RUNS)
for R in RUNS:
    NVTCorr28 = pd.read_csv('{}/NHCorrFunc/NH46CorrelationFunctions.dat'.format(LocDF.loc[R,('Ubiquitin','NVT')]), delim_whitespace=True)
    rescfdf_NVT.loc[:,R] = NVTCorr28['<P2>'].values

In [ ]:
ax = rescfdf.mean(axis=1).iloc[1:].plot(logx=True)
rescfdf_NVT.mean(axis=1).iloc[1:].plot(logx=True, ax=ax)

In [ ]:
rescfdf_NVT['Run1'].plot(logx=True)

In [ ]:
rescfdf

In [ ]:
rescfdf.plot(x='#Time', y='<P2>', logx=True)
NVTCorr28.plot(x='#Time', y='<P2>', logx=True, ax=plt.gca())

In [ ]:
rescfdf_NVT

In [ ]:
NVTCorr28.shape

In [ ]:
ax = RESCorrDF_NVE[45].plot(logx=True, c='k')
ax = RESCorrDF_NVT[45].plot(logx=True, ax=ax,c='r')

In [ ]:
AvgResCorrFunctions_TrpCage_NVE = pd.read_excel('D:/Research/LangevinDynamics_RotationalDiffusion/TRP-Cage/AverageFullCorrelationFunctions_NVE.xlsx', index_col=0)
STDResCorrFunctions_TrpCage_NVE = pd.read_excel('D:/Research/LangevinDynamics_RotationalDiffusion/TRP-Cage/STDFullCorrelationFunctions_NVE.xlsx', index_col=0)

In [ ]:
AvgResCorrFunctions_TrpCage_NVE

In [ ]:
AvgResCorrFunctions_TrpCage_NVE.loc[4.2055]

In [ ]:
AvgResCorrFunctions_TrpCage_NVT_Cf2ps = pd.read_excel('D:/Research/LangevinDynamics_RotationalDiffusion/TRP-Cage/AverageFullCorrelationFunctions_NVT_CF2ps.xlsx', index_col=0)
STDResCorrFunctions_TrpCage_NVT_Cf2ps = pd.read_excel('D:/Research/LangevinDynamics_RotationalDiffusion/TRP-Cage/STDFullCorrelationFunctions_NVT_CF2ps.xlsx', index_col=0)

In [ ]:
ResCorrFig = plt.figure(1, figsize=(8,6))
for rcol in AvgResCorrFunctions_TrpCage_NVE.columns:
    axCF = ResCorrFig.add_subplot(111)
    
    tstop_arr_NVE = np.where(AvgResCorrFunctions_TrpCage_NVE[rcol].values < 0)[0]
    AvgResCorrFunctions_TrpCage_NVE[rcol].plot(logx=True, ax = axCF, label=r'$\mathbf{ NVE }$', color='b')
    axCF.axvline(AvgResCorrFunctions_TrpCage_NVE.index.values[tstop_arr_NVE[0]-1], 0, 1, color='b',
                 linestyle='--', alpha=0.5)
    axCF.fill_between(AvgResCorrFunctions_TrpCage_NVE.index.values,
                      AvgResCorrFunctions_TrpCage_NVE[rcol].values-STDResCorrFunctions_TrpCage_NVE[rcol].values,
                      AvgResCorrFunctions_TrpCage_NVE[rcol].values+STDResCorrFunctions_TrpCage_NVE[rcol].values,
                      color='b',alpha=0.5)
    
    #ResCorrDF_GB3_NVT_CF02ps[rcol].plot(logx=True, ax = axCF, label=r'$\mathbf{ NVT : \gamma = 0.2 \ \ ps^{-1}}$')
    tstop_arr_NVT = np.where(AvgResCorrFunctions_TrpCage_NVT_Cf2ps[rcol].values < 0)[0]
    
    AvgResCorrFunctions_TrpCage_NVT_Cf2ps[rcol].plot(logx=True, ax = axCF,
                                                     label=r'$\mathbf{ NVT : \gamma = 2 \ \ ps^{-1}}$',
                                                     color='g')
    #ResCorrDF_GB3_NVT_CF20ps[rcol].plot(logx=True, ax = axCF, label=r'$\mathbf{ NVT : \gamma = 20 \ \ ps^{-1}}$')
    axCF.set_title('Residue: {}'.format(rcol), fontsize=15, weight='bold')
    axCF.axvline(AvgResCorrFunctions_TrpCage_NVT_Cf2ps.index.values[tstop_arr_NVT[0]-1], 0, 1, color='g',
                 linestyle='--', alpha=0.5)
    axCF.fill_between(AvgResCorrFunctions_TrpCage_NVT_Cf2ps.index.values,
                      AvgResCorrFunctions_TrpCage_NVT_Cf2ps[rcol].values-STDResCorrFunctions_TrpCage_NVT_Cf2ps[rcol].values,
                      AvgResCorrFunctions_TrpCage_NVT_Cf2ps[rcol].values+STDResCorrFunctions_TrpCage_NVT_Cf2ps[rcol].values,
                      color='g',alpha=0.5)
    
    axCF.legend(loc=3)
    axCF.set_ylabel(r'$\mathbf{C(\tau)}$', fontsize=13)
    axCF.set_xlabel(r'$\mathbf{\tau \ \ (ns)}$',fontsize=13)
    axCF.tick_params(labelsize=15)
    ResCorrFig.savefig('{}TRP-Cage/Analysis/CompareCorrelationFunctions_WithCoupling_Res{}_wErr.png'.format(CorrFuncLoc2, rcol),dpi=600, bbox_inches='tight')
    ResCorrFig.clear()

In [ ]:
def _get_FirstNegativeTime(corrdf):
    
    time_df = pd.DataFrame(index=corrdf.columns, columns=['time','neg_exists'])
    for col in corrdf.columns:
        tstop_arr = np.where(corrdf[col].values < 0)[0]
        if tstop_arr.size > 0:
            tstop = tstop_arr[0]
            neg_exists=1
        else:
            print('No Negative values found in the correlation')
            tstop = np.where(corrdf.index.values==tau_mem*2)[0][0]
            neg_exists=0
        print('Final Value in the Fit is at {} : {} ns'.format(tstop-1, corrdf.index.values[tstop-1]))
        
        time_df.loc[col,'time'] = corrdf.index.values[tstop-1]
        time_df.loc[col,'neg_exists'] = neg_exists
        
    return time_df

In [ ]:
TrpCage_FirstNegTime_NVE = _get_FirstNegativeTime(AvgResCorrFunctions_TrpCage_NVE)
TrpCage_FirstNegTime_NVT_CF2ps = _get_FirstNegativeTime(AvgResCorrFunctions_TrpCage_NVT_Cf2ps)

In [ ]:
TrpCage_FirstNegTime_NVE['time'].mean()

In [ ]:
TrpCage_FirstNegTime_NVT_CF2ps['time'].mean()

In [ ]:
tau_mem=4.75; thresh=0.25; fthresh=False; ShortSim=False; FixFit=True; Fix_Tf=True;
FitDF_TrpCage_NVE_Lin_TM, covarMat_lists = fitCorrF(AvgResCorrFunctions_TrpCage_NVE,
                                     STDResCorrFunctions_TrpCage_NVE, tau_mem, [4], 1, fixfit=FixFit, first_neg=False)
tau_mem=8.5;
FitDF_TrpCage_NVT_CF2ps_Lin_TM, covarMat_lists = fitCorrF(AvgResCorrFunctions_TrpCage_NVT_Cf2ps,
                                     STDResCorrFunctions_TrpCage_NVT_Cf2ps, tau_mem, [4], 1, fixfit=FixFit, first_neg=False)

In [ ]:
tau_mem=4.75; thresh=0.25; fthresh=False; ShortSim=False; FixFit=True; Fix_Tf=True;
FitDF_TrpCage_NVE_Lin_TM_noSTD, covarMat_lists = fitCorrF_noSTD(AvgResCorrFunctions_TrpCage_NVE,
                                                          tau_mem, [4], 1, fixfit=FixFit, first_neg=False)
tau_mem=8.5;
FitDF_TrpCage_NVT_CF2ps_Lin_TM_noSTD, covarMat_lists = fitCorrF_noSTD(AvgResCorrFunctions_TrpCage_NVT_Cf2ps,
                                                                tau_mem, [4], 1, fixfit=FixFit, first_neg=False)

In [ ]:
tau_mem=4.75; thresh=0.25; fthresh=False; ShortSim=False; FixFit=True; Fix_Tf=True;
FitDF_TrpCage_NVE_Lin_FN_noSTD, covarMat_lists = fitCorrF_noSTD(AvgResCorrFunctions_TrpCage_NVE,
                                                          tau_mem, [4], 1, fixfit=FixFit, first_neg=True)
tau_mem=8.5;
FitDF_TrpCage_NVT_CF2ps_Lin_FN_noSTD, covarMat_lists = fitCorrF_noSTD(AvgResCorrFunctions_TrpCage_NVT_Cf2ps,
                                                                tau_mem, [4], 1, fixfit=FixFit, first_neg=True)

In [ ]:
FitDF_TrpCage_NVE_Lin_FN, covarMat_lists = fitCorrF(AvgResCorrFunctions_TrpCage_NVE,
                                     STDResCorrFunctions_TrpCage_NVE, tau_mem, [4], 1, fixfit=FixFit, first_neg=True)
FitDF_TrpCage_NVT_CF2ps_Lin_FN, covarMat_lists = fitCorrF(AvgResCorrFunctions_TrpCage_NVT_Cf2ps,
                                     STDResCorrFunctions_TrpCage_NVT_Cf2ps, tau_mem, [4], 1, fixfit=FixFit, first_neg=True)

In [ ]:
def adjust_compfit(fdf, lossfunc, cutoff):
    plist = ['Resname', 'C_a', 'C_a_err', 'C_b', 'C_b_err', 'tau_a', 'tau_a_err', 'tau_b', 'tau_b_err']
    pfdf = fdf.loc[:,plist].copy()
    pfdf['Loss'] = lossfunc
    pfdf['CutOff'] = cutoff
    return pfdf

In [ ]:
FitDF_NVE_SL_TM_TrpCage_Params = FitDF_TrpCage_NVE_SL_TM[['Resname', 'C_a', 'C_a_err', 'C_b', 'C_b_err', 'tau_a', 'tau_a_err', 'tau_b', 'tau_b_err']]
FitDF_NVE_SL_TM_TrpCage_Params['Loss'] = 'SoftL'
FitDF_NVE_SL_TM_TrpCage_Params['CutOff'] = 'Fixed'

In [ ]:
FitDF_TrpCage_NVE_SL_TMParams = adjust_compfit(FitDF_TrpCage_NVE_SL_TM, 'SoftL','Fixed')
FitDF_TrpCage_NVE_SL_NPParams = adjust_compfit(FitDF_TrpCage_NVE_SL_FN, 'SoftL','FirstNegative')

In [ ]:
dy=[]
len(dy)

In [ ]:
import seaborn as sns

In [ ]:
sns.relplot(x='Resname',y='Value
            ')

In [ ]:
FitDF_TrpCage_NVT_CF2ps_SL_FN[['Resname', 'C_a', 'C_a_err', 'C_b', 'C_b_err', 'tau_a', 'tau_a_err', 'tau_b', 'tau_b_err']]

In [ ]:
FitDF_TrpCage_NVE_SL_TM.loc[2:6,['Resname','C_a','tau_a','C_b','tau_b']]

In [ ]:
FitDF_TrpCage_NVE_SL_FN.loc[2:6, ['Resname','C_a','tau_a','C_b','tau_b']]

In [ ]:
FitDF_TrpCage_NVE_SL_FN

In [ ]:
plt.rcParams['axes.titlepad'] = 0.0
figTrp, axamptrp = plt.subplots(2,2, sharex=True, figsize=(12,10))
figTrp.subplots_adjust(hspace=0.045, wspace=0.1)

FitDF_TrpCage_NVE_SL_FN.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='-', color='b', marker='o',
                             ax=axamptrp[0][0], label='NVE_FirstNeg')
FitDF_TrpCage_NVE_SL_TM.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='-', color='y', marker='d',
                             ax=axamptrp[0][0], label='NVE_Fixed')

FitDF_TrpCage_NVT_CF2ps_SL_FN.plot(x='Resname', y='C_a', yerr='C_a_err',linestyle='--', color='b', marker='o',
                                   ax=axamptrp[0][0], label='NVT_FirstNeg' )
FitDF_TrpCage_NVT_CF2ps_SL_FN.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='--', color='y', marker='d',
                                   ax=axamptrp[0][0], label='NVT_Fixed')

axamptrp[0][0].set_ylim(0,1.0)
axamptrp[0][0].set_title('C_a',weight='bold',fontsize=12, pad=-14)

FitDF_TrpCage_NVE_SL_FN.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='-', color='b', marker='o',
                             ax=axamptrp[0][1], label='NVE_FirstNeg')
FitDF_TrpCage_NVE_SL_TM.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='-', color='y', marker='d',
                             ax=axamptrp[0][1], label='NVE_Fixed')

FitDF_TrpCage_NVT_CF2ps_SL_FN.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='--', color='b', marker='o',
                                   ax=axamptrp[0][1], label='NVT_FirstNeg')
FitDF_TrpCage_NVT_CF2ps_SL_FN.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='--', color='y', marker='d',
                                   ax=axamptrp[0][1], label='NVT_Fixed')

axamptrp[0][1].set_ylim(0,1.0)
axamptrp[0][1].set_title('C_b', weight='bold', fontsize=12, pad=-14)

FitDF_TrpCage_NVE_SL_FN.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='-', color='b', marker='o',
                             ax=axamptrp[1][0], label='NVE_FirstNeg')
FitDF_TrpCage_NVE_SL_TM.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='-', color='y', marker='d',
                             ax=axamptrp[1][0], label='NVE_Fixed')

FitDF_TrpCage_NVT_CF2ps_SL_FN.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='--', color='b', marker='o',
                                   ax=axamptrp[1][0], label='NVT_FirstNeg')
FitDF_TrpCage_NVT_CF2ps_SL_FN.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='--', color='y', marker='d',
                                   ax=axamptrp[1][0], label='NVT_Fixed')

axamptrp[1][0].set_ylim(0.5, 2.5)
axamptrp[1][0].set_title('tau_a',weight='bold',fontsize=12, y=0.95, pad=-0.0)

FitDF_TrpCage_NVE_SL_FN.plot(x='Resname', y='tau_b', logy=True, yerr='tau_b_err', linestyle='-', color='b', marker='o',
                             ax=axamptrp[1][1], label='NVE_FirstNeg')
FitDF_TrpCage_NVE_SL_TM.plot(x='Resname', y='tau_b', logy=True, yerr='tau_a_err', linestyle='-', color='y', marker='d',
                             ax=axamptrp[1][1], label='NVE_Fixed')

FitDF_TrpCage_NVT_CF2ps_SL_FN.plot(x='Resname', y='tau_b', logy=True, yerr='tau_b_err', linestyle='--', color='b', marker='o',
                                   ax=axamptrp[1][1], label='NVT_FirstNeg')
FitDF_TrpCage_NVT_CF2ps_SL_FN.plot(x='Resname', y='tau_b', logy=True, yerr='tau_b_err', linestyle='--', color='y', marker='d',
                                   ax=axamptrp[1][1], label='NVT_Fixed')

axamptrp[1][1].set_title('tau_b', weight='bold', fontsize=12,  y=0.95, pad=0.0)

figTrp.savefig('D:/Research/LangevinDynamics_RotationalDiffusion/TRP-Cage/CompareCutOff_Amplitudes_Relaxationtimes_DiffFixedCutOffs_Linear.png',
               dpi=600,bbox_inches='tight')

In [ ]:
plt.rcParams['axes.titlepad'] = 0.0
figTrpCF_NVE, axtrpcf_nve = plt.subplots(2,2, sharex=True, figsize=(12,10))
figTrpCF_NVE.subplots_adjust(hspace=0.045, wspace=0.1)

FitDF_TrpCage_NVE_SL_TM.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='-', color='b', marker='o',
                             ax=axtrpcf_nve[0][0], label='Fixed_SoftLoss')
FitDF_TrpCage_NVE_Lin_TM.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='-', color='r', marker='x',
                             ax=axtrpcf_nve[0][0], label='Fixed_Linear')

FitDF_TrpCage_NVE_SL_FN.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='--', color='b', marker='o',
                             ax=axtrpcf_nve[0][0], label='FirstNeg_SoftLoss')
FitDF_TrpCage_NVE_Lin_FN.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='--', color='r', marker='x',
                             ax=axtrpcf_nve[0][0], label='FirstNeg_Linear')



axtrpcf_nve[0][0].set_ylim(0,1.0)
axtrpcf_nve[0][0].set_title('C_a',weight='bold',fontsize=12, pad=-14)

FitDF_TrpCage_NVE_SL_TM.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='-', color='b', marker='o',
                             ax=axtrpcf_nve[0][1], label='Fixed_SoftLoss')
FitDF_TrpCage_NVE_Lin_TM.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='-', color='r', marker='x',
                             ax=axtrpcf_nve[0][1], label='Fixed_Linear')

FitDF_TrpCage_NVE_SL_FN.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='--', color='b', marker='o',
                             ax=axtrpcf_nve[0][1], label='FirstNeg_SoftLoss')
FitDF_TrpCage_NVE_Lin_FN.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='--', color='r', marker='x',
                             ax=axtrpcf_nve[0][1], label='FirstNeg_Linear')

axtrpcf_nve[0][1].set_ylim(0,1.0)
axtrpcf_nve[0][1].set_title('C_b', weight='bold', fontsize=12, pad=-14)

FitDF_TrpCage_NVE_SL_TM.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='-', color='b', marker='o',
                             ax=axtrpcf_nve[1][0], label='Fixed_SoftLoss')
FitDF_TrpCage_NVE_Lin_TM.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='-', color='r', marker='x',
                             ax=axtrpcf_nve[1][0], label='Fixed_Linear')

FitDF_TrpCage_NVE_SL_FN.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='--', color='b', marker='o',
                             ax=axtrpcf_nve[1][0], label='FirstNeg_SoftLoss')
FitDF_TrpCage_NVE_Lin_FN.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='--', color='r', marker='x',
                             ax=axtrpcf_nve[1][0], label='FirstNeg_Linear')

axtrpcf_nve[1][0].set_ylim(0.8, 1.25)
axtrpcf_nve[1][0].set_title('tau_a',weight='bold',fontsize=12, y=0.95, pad=-0.0)

FitDF_TrpCage_NVE_SL_TM.plot(x='Resname', y='tau_b', yerr='tau_b_err', linestyle='-', color='b', marker='o',
                             ax=axtrpcf_nve[1][1], label='Fixed_SoftLoss')
FitDF_TrpCage_NVE_Lin_TM.plot(x='Resname', y='tau_b', yerr='tau_b_err', linestyle='-', color='r', marker='x',
                             ax=axtrpcf_nve[1][1], label='Fixed_Linear')

FitDF_TrpCage_NVE_SL_FN.plot(x='Resname', y='tau_b', yerr='tau_b_err', logy=True, linestyle='--', color='b', marker='o',
                             ax=axtrpcf_nve[1][1], label='FirstNeg_SoftLoss')
FitDF_TrpCage_NVE_Lin_FN.plot(x='Resname', y='tau_b', yerr='tau_b_err',logy=True, linestyle='--', color='r', marker='x',
                             ax=axtrpcf_nve[1][1], label='FirstNeg_Linear')

axtrpcf_nve[1][1].set_title('tau_b', weight='bold', fontsize=12,  y=0.95, pad=0.0)

figTrpCF_NVE.savefig('D:/Research/LangevinDynamics_RotationalDiffusion/TRP-Cage/CompareLossFunction_NVE_Amplitudes_Relaxationtimes_DiffFixedCutOffs.png',
               dpi=600,bbox_inches='tight')

In [ ]:
plt.rcParams['axes.titlepad'] = 0.0
figTrpCF_NVT, axtrpcf_nvt = plt.subplots(2,2, sharex=True, figsize=(12,10))
figTrpCF_NVT.subplots_adjust(hspace=0.045, wspace=0.1)

FitDF_TrpCage_NVT_CF2ps_SL_TM.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='-', color='b', marker='o',
                             ax=axtrpcf_nvt[0][0], label='Fixed_SoftLoss')
FitDF_TrpCage_NVT_CF2ps_Lin_TM.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='-', color='r', marker='x',
                             ax=axtrpcf_nvt[0][0], label='Fixed_Linear')

FitDF_TrpCage_NVT_CF2ps_SL_FN.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='--', color='b', marker='o',
                             ax=axtrpcf_nvt[0][0], label='FirstNeg_SoftLoss')
FitDF_TrpCage_NVT_CF2ps_Lin_FN.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='--', color='r', marker='x',
                             ax=axtrpcf_nvt[0][0], label='FirstNeg_Linear')



axtrpcf_nvt[0][0].set_ylim(0,1.0)
axtrpcf_nvt[0][0].set_title('C_a',weight='bold',fontsize=12, pad=-14)

FitDF_TrpCage_NVT_CF2ps_SL_TM.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='-', color='b', marker='o',
                             ax=axtrpcf_nvt[0][1], label='Fixed_SoftLoss')
FitDF_TrpCage_NVT_CF2ps_Lin_TM.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='-', color='r', marker='x',
                             ax=axtrpcf_nvt[0][1], label='Fixed_Linear')

FitDF_TrpCage_NVT_CF2ps_SL_FN.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='--', color='b', marker='o',
                             ax=axtrpcf_nvt[0][1], label='FirstNeg_SoftLoss')
FitDF_TrpCage_NVT_CF2ps_Lin_FN.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='--', color='r', marker='x',
                             ax=axtrpcf_nvt[0][1], label='FirstNeg_Linear')

axtrpcf_nvt[0][1].set_ylim(0,1.0)
axtrpcf_nvt[0][1].set_title('C_b', weight='bold', fontsize=12, pad=-14)

FitDF_TrpCage_NVT_CF2ps_SL_TM.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='-', color='b', marker='o',
                             ax=axtrpcf_nvt[1][0], label='Fixed_SoftLoss')
FitDF_TrpCage_NVT_CF2ps_Lin_TM.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='-', color='r', marker='x',
                             ax=axtrpcf_nvt[1][0], label='Fixed_Linear')

FitDF_TrpCage_NVT_CF2ps_SL_FN.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='--', color='b', marker='o',
                             ax=axtrpcf_nvt[1][0], label='FirstNeg_SoftLoss')
FitDF_TrpCage_NVT_CF2ps_Lin_FN.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='--', color='r', marker='x',
                             ax=axtrpcf_nvt[1][0], label='FirstNeg_Linear')

axtrpcf_nvt[1][0].set_ylim(0.8, 2.0)
axtrpcf_nvt[1][0].set_title('tau_a',weight='bold',fontsize=12, y=0.95, pad=-0.0)

FitDF_TrpCage_NVT_CF2ps_SL_TM.plot(x='Resname', y='tau_b', yerr='tau_b_err', linestyle='-', color='b', marker='o',
                             ax=axtrpcf_nvt[1][1], label='Fixed_SoftLoss')
FitDF_TrpCage_NVT_CF2ps_Lin_TM.plot(x='Resname', y='tau_b', yerr='tau_b_err', linestyle='-', color='r', marker='x',
                             ax=axtrpcf_nvt[1][1], label='Fixed_Linear')

FitDF_TrpCage_NVT_CF2ps_SL_FN.plot(x='Resname', y='tau_b', yerr='tau_b_err', logy=True, linestyle='--', color='b', marker='o',
                             ax=axtrpcf_nvt[1][1], label='FirstNeg_SoftLoss')
FitDF_TrpCage_NVT_CF2ps_Lin_FN.plot(x='Resname', y='tau_b', yerr='tau_b_err',logy=True, linestyle='--', color='r', marker='x',
                             ax=axtrpcf_nvt[1][1], label='FirstNeg_Linear')

axtrpcf_nvt[1][1].set_title('tau_b', weight='bold', fontsize=12,  y=0.95, pad=0.0)

figTrpCF_NVT.savefig('D:/Research/LangevinDynamics_RotationalDiffusion/TRP-Cage/CompareLossFunction_NVT_Amplitudes_Relaxationtimes_DiffFixedCutOffs.png',
               dpi=600,bbox_inches='tight')

In [ ]:
plt.rcParams['axes.titlepad'] = 0.0
figTrpSTD, axamptrpstd = plt.subplots(2,2, sharex=True, figsize=(12,10), num=4545)
figTrpSTD.subplots_adjust(hspace=0.045, wspace=0.1)

FitDF_TrpCage_NVE_Lin_FN.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='-', color='b', marker='o',
                             ax=axamptrpstd[0][0], label='NVE_wSTD')
FitDF_TrpCage_NVE_Lin_FN_noSTD.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='-', color='y', marker='d',
                             ax=axamptrpstd[0][0], label='NVE_noSTD')

FitDF_TrpCage_NVT_CF2ps_Lin_FN.plot(x='Resname', y='C_a', yerr='C_a_err',linestyle='--', color='b', marker='o',
                                   ax=axamptrpstd[0][0], label='NVT_wSTD' )
FitDF_TrpCage_NVT_CF2ps_Lin_FN_noSTD.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='--', color='y', marker='d',
                                   ax=axamptrpstd[0][0], label='NVT_noSTD')

axamptrpstd[0][0].set_ylim(0,1.0)
axamptrpstd[0][0].set_title('C_a',weight='bold',fontsize=12, pad=-14)

FitDF_TrpCage_NVE_Lin_FN.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='-', color='b', marker='o',
                             ax=axamptrpstd[0][1], label='NVE_wSTD')
FitDF_TrpCage_NVE_Lin_FN_noSTD.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='-', color='y', marker='d',
                             ax=axamptrpstd[0][1], label='NVE_noSTD')

FitDF_TrpCage_NVT_CF2ps_Lin_FN.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='--', color='b', marker='o',
                                   ax=axamptrpstd[0][1], label='NVT_wSTD')
FitDF_TrpCage_NVT_CF2ps_Lin_FN_noSTD.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='--', color='y', marker='d',
                                   ax=axamptrpstd[0][1], label='NVT_noSTD')

axamptrpstd[0][1].set_ylim(0,1.0)
axamptrpstd[0][1].set_title('C_b', weight='bold', fontsize=12, pad=-14)

FitDF_TrpCage_NVE_Lin_FN.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='-', color='b', marker='o',
                             ax=axamptrpstd[1][0], label='NVE_wSTD')
FitDF_TrpCage_NVE_Lin_FN_noSTD.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='-', color='y', marker='d',
                             ax=axamptrpstd[1][0], label='NVE_noSTD')

FitDF_TrpCage_NVT_CF2ps_Lin_FN.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='--', color='b', marker='o',
                                   ax=axamptrpstd[1][0], label='NVT_wSTD')
FitDF_TrpCage_NVT_CF2ps_Lin_FN_noSTD.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='--', color='y', marker='d',
                                   ax=axamptrpstd[1][0], label='NVT_noSTD')

axamptrpstd[1][0].set_ylim(0.5, 2.5)
axamptrpstd[1][0].set_title('tau_a',weight='bold',fontsize=12, y=0.95, pad=-0.0)

FitDF_TrpCage_NVE_Lin_FN.plot(x='Resname', y='tau_b', logy=True, yerr='tau_b_err', linestyle='-', color='b', marker='o',
                             ax=axamptrpstd[1][1], label='NVE_wSTD')
FitDF_TrpCage_NVE_Lin_FN_noSTD.plot(x='Resname', y='tau_b', logy=True, yerr='tau_a_err', linestyle='-', color='y', marker='d',
                             ax=axamptrpstd[1][1], label='NVE_noSTD')

FitDF_TrpCage_NVT_CF2ps_Lin_FN.plot(x='Resname', y='tau_b', logy=True, yerr='tau_b_err', linestyle='--', color='b', marker='o',
                                   ax=axamptrpstd[1][1], label='NVT_wSTD')
FitDF_TrpCage_NVT_CF2ps_Lin_FN_noSTD.plot(x='Resname', y='tau_b', logy=True, yerr='tau_b_err', linestyle='--', color='y', marker='d',
                                   ax=axamptrpstd[1][1], label='NVT_noSTD')

axamptrpstd[1][1].set_ylim(0.00005, 1.0)
axamptrpstd[1][1].set_title('tau_b', weight='bold', fontsize=12,  y=0.95, pad=0.0)

figTrpSTD.savefig('D:/Research/LangevinDynamics_RotationalDiffusion/TRP-Cage/CompareSTD_Amplitudes_Relaxationtimes_FirstNegCutOFF_Linear.png',
               dpi=600,bbox_inches='tight')

In [ ]:
plt.rcParams['axes.titlepad'] = 0.0
figTrpSTD, axamptrpstd = plt.subplots(2, 2, sharex=True, figsize=(12,10), num=4771)
figTrpSTD.subplots_adjust(hspace=0.045, wspace=0.1)

FitDF_TrpCage_NVE_Lin_TM.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='-', color='b', marker='o',
                             ax=axamptrpstd[0][0], label='NVE_wSTD')
FitDF_TrpCage_NVE_Lin_TM_noSTD.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='-', color='y', marker='d',
                             ax=axamptrpstd[0][0], label='NVE_noSTD')

FitDF_TrpCage_NVT_CF2ps_Lin_TM.plot(x='Resname', y='C_a', yerr='C_a_err',linestyle='--', color='b', marker='o',
                                   ax=axamptrpstd[0][0], label='NVT_wSTD' )
FitDF_TrpCage_NVT_CF2ps_Lin_TM_noSTD.plot(x='Resname', y='C_a', yerr='C_a_err', linestyle='--', color='y', marker='d',
                                   ax=axamptrpstd[0][0], label='NVT_noSTD')

axamptrpstd[0][0].set_ylim(0,1.0)
axamptrpstd[0][0].set_title('C_a',weight='bold',fontsize=12, pad=-14)

FitDF_TrpCage_NVE_Lin_TM.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='-', color='b', marker='o',
                             ax=axamptrpstd[0][1], label='NVE_wSTD')
FitDF_TrpCage_NVE_Lin_TM_noSTD.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='-', color='y', marker='d',
                             ax=axamptrpstd[0][1], label='NVE_noSTD')

FitDF_TrpCage_NVT_CF2ps_Lin_TM.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='--', color='b', marker='o',
                                   ax=axamptrpstd[0][1], label='NVT_wSTD')
FitDF_TrpCage_NVT_CF2ps_Lin_TM_noSTD.plot(x='Resname', y='C_b', yerr='C_b_err', linestyle='--', color='y', marker='d',
                                   ax=axamptrpstd[0][1], label='NVT_noSTD')

axamptrpstd[0][1].set_ylim(0,1.0)
axamptrpstd[0][1].set_title('C_b', weight='bold', fontsize=12, pad=-14)

FitDF_TrpCage_NVE_Lin_TM.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='-', color='b', marker='o',
                             ax=axamptrpstd[1][0], label='NVE_wSTD')
FitDF_TrpCage_NVE_Lin_TM_noSTD.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='-', color='y', marker='d',
                             ax=axamptrpstd[1][0], label='NVE_noSTD')

FitDF_TrpCage_NVT_CF2ps_Lin_TM.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='--', color='b', marker='o',
                                   ax=axamptrpstd[1][0], label='NVT_wSTD')
FitDF_TrpCage_NVT_CF2ps_Lin_TM_noSTD.plot(x='Resname', y='tau_a', yerr='tau_a_err', linestyle='--', color='y', marker='d',
                                   ax=axamptrpstd[1][0], label='NVT_noSTD')

axamptrpstd[1][0].set_ylim(0.5, 2.5)
axamptrpstd[1][0].set_title('tau_a',weight='bold',fontsize=12, y=0.95, pad=-0.0)

FitDF_TrpCage_NVE_Lin_TM.plot(x='Resname', y='tau_b', logy=True, yerr='tau_b_err', linestyle='-', color='b', marker='o',
                             ax=axamptrpstd[1][1], label='NVE_wSTD')
FitDF_TrpCage_NVE_Lin_TM_noSTD.plot(x='Resname', y='tau_b', logy=True, yerr='tau_a_err', linestyle='-', color='y', marker='d',
                             ax=axamptrpstd[1][1], label='NVE_noSTD')

FitDF_TrpCage_NVT_CF2ps_Lin_TM.plot(x='Resname', y='tau_b', logy=True, yerr='tau_b_err', linestyle='--', color='b', marker='o',
                                   ax=axamptrpstd[1][1], label='NVT_wSTD')
FitDF_TrpCage_NVT_CF2ps_Lin_TM_noSTD.plot(x='Resname', y='tau_b', logy=True, yerr='tau_b_err', linestyle='--', color='y', marker='d',
                                   ax=axamptrpstd[1][1], label='NVT_noSTD')

axamptrpstd[1][1].set_ylim(0.00005, 1.0)
axamptrpstd[1][1].set_title('tau_b', weight='bold', fontsize=12,  y=0.95, pad=0.0)

figTrpSTD.savefig('D:/Research/LangevinDynamics_RotationalDiffusion/TRP-Cage/CompareSTD_Amplitudes_Relaxationtimes_FixedCutOFF_Linear.png',
               dpi=600,bbox_inches='tight')

In [ ]:
FitDF_TrpCage_NVE_Lin_FN.to_csv('D:/Research/LangevinDynamics_RotationalDiffusion/TRP-Cage/Analysis/FitDF_NVE_Linear_FirstNegative_wSTD.csv')
FitDF_TrpCage_NVE_Lin_FN_noSTD.to_csv('D:/Research/LangevinDynamics_RotationalDiffusion/TRP-Cage/Analysis/FitDF_NVE_Linear_FirstNegative_noSTD.csv')
FitDF_TrpCage_NVE_Lin_TM.to_csv('D:/Research/LangevinDynamics_RotationalDiffusion/TRP-Cage/Analysis/FitDF_NVE_Linear_FixedCutOff_wSTD.csv')
FitDF_TrpCage_NVE_Lin_TM_noSTD.to_csv('D:/Research/LangevinDynamics_RotationalDiffusion/TRP-Cage/Analysis/FitDF_NVE_Linear_FixedCutOff_noSTD.csv')

In [ ]:
FitDF_TrpCage_NVE_SL_FN.to_csv('D:/Research/LangevinDynamics_RotationalDiffusion/TRP-Cage/Analysis/FitDF_NVE_SoftL_FirstNegative_wSTD.csv')
#FitDF_TrpCage_NVE_SL_FN_noSTD.to_csv('D:/Research/LangevinDynamics_RotationalDiffusion/TRP-Cage/Analysis/FitDF_NVE_SoftL_FirstNegative_noSTD.csv')
FitDF_TrpCage_NVE_SL_TM.to_csv('D:/Research/LangevinDynamics_RotationalDiffusion/TRP-Cage/Analysis/FitDF_NVE_SoftL_FixedCutOff_wSTD.csv')
#FitDF_TrpCage_NVE_SL_TM_noSTD.to_csv('D:/Research/LangevinDynamics_RotationalDiffusion/TRP-Cage/Analysis/FitDF_NVE_SoftL_FixedCutOff_noSTD.csv')

In [ ]:
FitDF_TrpCage_NVE.loc[2:6,['Resname','C_a','tau_a','C_b','tau_b']]

In [ ]:
FitDF_TrpCage_NVT_CF2ps_SL_TM.loc[2:6,['Resname','C_a','tau_a','C_b','tau_b']]

In [ ]:
FitDF_TrpCage_NVE_Lin_FN.loc[2:6,['Resname','C_a','tau_a','C_b','tau_b']]

In [ ]:
FitDF_TrpCage_NVE['tau_b']

In [ ]:
FitDF_TrpCage_NVT_CF2ps['tau_b']

In [ ]:
FitDF_TrpCage_NVT_CF2ps.to_csv('D:/Research/LangevinDynamics_RotationalDiffusion/TRP-Cage/FitDF_Fix2Exp_NVT_Cf2ps_FirstNegTstop_Linear.csv')

In [ ]:
FitDF_TrpCage_NVE.to_csv('D:/Research/LangevinDynamics_RotationalDiffusion/TRP-Cage/FitDF_Fix2Exp_NVE_FirstNegTstop_Linear.csv')